# DATA-DRIVEN 1D MECHANICAL EARTH MODEL — POSEIDON 2

## Increment 8.0.1 — Final Corrective Release of the Pore-Pressure Data-Gap, Hydrostatic-Reference, and Effective-Stress Screening Framework

**Classification:** Tier C — Screening-Level / Uncalibrated Educational

This notebook is intentionally conservative. It inventories the absence of approved pressure/stress calibration data, builds configured hydrostatic gauge-pressure references, and pairs them with Increment 7 vertical-stress scenarios to show effective-stress sensitivity. It does **not** select or fit an NCT, run Eaton/Bowers/equivalent-depth/drilling-exponent transforms, infer overpressure, extrapolate, or modify an upstream curve.


### Step 1 — Locate or extract the final project safely and idempotently

In Colab, this cell reuses a tree only after its Increment 8.0.1 ledger verifies. Otherwise it extracts `Poseidon_1D_MEM_Increment_08_v8.0.1.zip` into a new versioned folder. An incomplete older folder is never deleted or overwritten. Set `PROJECT_ROOT_OVERRIDE` before running the cell only if you deliberately want to use another complete Increment 8.0.1 or locked Increment 7.0.4 tree.


In [ ]:
from pathlib import Path, PurePosixPath
from zipfile import ZipFile
from datetime import datetime, timezone
import hashlib, os, shutil, sys, tempfile

RELEASE_ARCHIVE_NAME = "Poseidon_1D_MEM_Increment_08_v8.0.1.zip"
RELEASE_LEDGER_NAME = "INCREMENT_08_0_1_SHA256SUMS.txt"
RELEASE_FOLDER_NAME = "Poseidon_1D_MEM_Increment_08_v8.0.1"


def verify_release_tree(root):
    root = Path(root)
    ledger_path = root / RELEASE_LEDGER_NAME
    if not ledger_path.is_file():
        return False, (RELEASE_LEDGER_NAME,)
    mismatches, seen = [], set()
    try:
        for line in ledger_path.read_text(encoding="utf-8").splitlines():
            if not line or line.startswith("#"):
                continue
            expected, relative = line.split("  ", 1)
            rel = PurePosixPath(relative)
            if (len(expected) != 64 or any(c not in "0123456789abcdef" for c in expected)
                    or "\\" in relative or rel.as_posix() != relative
                    or rel.is_absolute() or ".." in rel.parts or relative in seen):
                return False, ("malformed release ledger",)
            seen.add(relative)
            path = root / relative
            actual = (hashlib.sha256(path.read_bytes()).hexdigest()
                      if path.is_file() and not path.is_symlink() else "MISSING_OR_SYMLINK")
            if actual != expected:
                mismatches.append(relative)
        actual_files = {
            path.relative_to(root).as_posix()
            for path in root.rglob("*")
            if path.is_file() and not path.is_symlink()
        }
        expected_files = seen | {RELEASE_LEDGER_NAME}
        if actual_files != expected_files:
            missing = sorted(expected_files - actual_files)
            unlisted = sorted(actual_files - expected_files)
            mismatches.extend(f"missing:{name}" for name in missing)
            mismatches.extend(f"unlisted:{name}" for name in unlisted)
        symlinks = sorted(
            path.relative_to(root).as_posix()
            for path in root.rglob("*")
            if path.is_symlink()
        )
        mismatches.extend(f"symlink:{name}" for name in symlinks)
    except (OSError, ValueError):
        return False, ("unreadable release ledger",)
    return bool(seen) and not mismatches, tuple(mismatches)


def safe_extract(archive, target):
    archive, target = Path(archive), Path(target)
    temporary = Path(tempfile.mkdtemp(prefix=".p2mem-inc8-extract-", dir=str(target.parent)))
    try:
        with ZipFile(archive) as zf:
            seen, total_size = set(), 0
            for info in zf.infolist():
                name = info.filename
                member = PurePosixPath(name)
                canonical = member.as_posix()
                mode = (info.external_attr >> 16) & 0o170000
                total_size += info.file_size
                if (not name or "\x00" in name or "\\" in name
                        or member.is_absolute() or ".." in member.parts
                        or canonical != name or canonical in seen or mode == 0o120000
                        or info.file_size > 100 * 1024 * 1024
                        or total_size > 250 * 1024 * 1024):
                    raise RuntimeError(f"Unsafe ZIP member: {name!r}")
                seen.add(canonical)
            zf.extractall(temporary)
        valid, mismatches = verify_release_tree(temporary)
        if not valid:
            raise RuntimeError(
                "Extracted release failed its own checksum ledger: " + repr(mismatches[:5]))
        os.replace(temporary, target)
    finally:
        if temporary.exists():
            shutil.rmtree(temporary)

IN_COLAB = False
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    drive = None

if IN_COLAB:
    drive.mount("/content/drive", force_remount=False)
    drive_root = Path("/content/drive/MyDrive")
    override = globals().get("PROJECT_ROOT_OVERRIDE")
    project_root = Path(override).expanduser() if override else None
    if project_root is not None:
        release_ok, _ = verify_release_tree(project_root)
        baseline_ok = (project_root / "INCREMENT_07_0_4_SHA256SUMS.txt").is_file()
        if not (project_root / "pyproject.toml").is_file() or not (release_ok or baseline_ok):
            raise RuntimeError(
                "PROJECT_ROOT_OVERRIDE is neither a verified Increment 8.0.1 tree "
                "nor a locked Increment 7.0.4 project root.")
    else:
        root_candidates = [
            drive_root / RELEASE_FOLDER_NAME,
            drive_root / "Poseidon_1D_MEM",
        ]
        project_root = next(
            (root for root in root_candidates if verify_release_tree(root)[0]), None)
        if project_root is None:
            archive_candidates = [
                drive_root / RELEASE_ARCHIVE_NAME,
                *sorted(drive_root.glob(f"*/{RELEASE_ARCHIVE_NAME}")),
            ]
            archive = next((path for path in archive_candidates if path.is_file()), None)
            if archive is None:
                raise FileNotFoundError(
                    f"Upload {RELEASE_ARCHIVE_NAME} to MyDrive. Older Increment 8 ZIPs "
                    "are intentionally not accepted by this final notebook.")
            target = drive_root / RELEASE_FOLDER_NAME
            if target.exists():
                stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
                target = drive_root / f"{RELEASE_FOLDER_NAME}_clean_{stamp}"
            safe_extract(archive, target)
            project_root = target
else:
    project_root = Path.cwd().resolve()
    if not (project_root / "pyproject.toml").is_file():
        raise RuntimeError("Run from the project root (the folder containing pyproject.toml).")

os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Project root:", project_root)
print("Project ready:", (project_root / "pyproject.toml").is_file())

### Step 2 — Verify the locked Increment 7.0.4 foundation before writing

In [ ]:
import hashlib

BASELINE_ZIP_SHA256 = "f5c9a9b50dd0edff6e5b31a42d5bba4ab23b993f5ef11c6217122b13f29d89ae"
ledger = project_root / "INCREMENT_07_0_4_SHA256SUMS.txt"
if not ledger.is_file():
    raise FileNotFoundError("Locked Increment 7.0.4 ledger is missing.")

ledger_entries = []
ledger_mismatches = []
for line in ledger.read_text(encoding="utf-8").splitlines():
    if not line or line.startswith("#"):
        continue
    expected, relative = line.split("  ", 1)
    path = project_root / relative
    actual = hashlib.sha256(path.read_bytes()).hexdigest() if path.is_file() else "MISSING"
    ledger_entries.append(relative)
    if actual != expected:
        ledger_mismatches.append(relative)

permitted_preexisting_changes = {"README.md", "pyproject.toml", "p2mem/__init__.py"}
assert len(ledger_entries) == 235
assert set(ledger_mismatches).issubset(permitted_preexisting_changes)
print("Locked ledger entries checked:", len(ledger_entries))
print("Expected version-file mismatches if Increment 8 is already extracted:", ledger_mismatches)

### Step 3 — Write the complete Increment 8 implementation and tests

In [ ]:
%%writefile pyproject.toml
[build-system]
requires = ["setuptools>=68.0"]
build-backend = "setuptools.build_meta"

[project]
name = "p2mem"
version = "0.8.1"
description = "Screening-level 1D Mechanical Earth Model workflow for Poseidon 2 (Tier C, uncalibrated / educational)."
readme = "README.md"
requires-python = ">=3.9"
license = { text = "All Rights Reserved. Copyright (c) 2026 Mikael Elgo. This is a personal portfolio project; no license is granted for reuse, redistribution, or commercial use without the author's explicit written permission." }
authors = [
    { name = "Mikael Elgo" }
]
keywords = ["geomechanics", "mechanical-earth-model", "pore-pressure", "wellbore-stability", "portfolio-project"]
classifiers = [
    "Development Status :: 3 - Alpha",
    "Programming Language :: Python :: 3",
    "Intended Audience :: Science/Research",
    "Topic :: Scientific/Engineering",
    "License :: Other/Proprietary License",
]

# Runtime dependencies are deliberately minimal. No unit-handling libraries
# (e.g. Pint) are used: unit conversions are implemented explicitly in
# p2mem.units so that every conversion factor is visible, documented, and
# testable rather than delegated to a third-party unit registry. PyYAML is
# added in Increment 2 for exactly one purpose: parsing the human-authored,
# human-reviewable per-file LAS curve contracts in
# config/las_curve_contracts.yml - a plain-text, diffable format was judged
# preferable to a hand-rolled config parser or a hard-coded Python dict.
dependencies = [
    "numpy>=1.24",
    "pyyaml>=6.0",
    # Increment 8 renders QC figures by default; this is therefore a runtime,
    # not merely a notebook-development, dependency.
    "matplotlib>=3.7",
]

[project.optional-dependencies]
dev = [
    "pytest>=7.4",
]

[tool.setuptools.packages.find]
include = ["p2mem*"]

[tool.pytest.ini_options]
testpaths = ["tests"]
python_files = ["test_*.py"]


In [ ]:
%%writefile p2mem/__init__.py
"""
p2mem - Poseidon 2 1D Mechanical Earth Model workflow package.

Project classification: Tier C - Screening-Level / Uncalibrated Educational
1D Mechanical Earth Model (see project design review, Rev 1). Nothing in
this package should be presented as a calibrated, operational, or
field-validated result unless an explicit independent calibration record
is attached to that specific output.

This package is under incremental, gated construction.

* Increment 1 / 1.1 delivered the project skeleton and the unit-control
  system (``p2mem.units``).
* Increment 2 added an auditable LAS-ingestion layer with explicit
  per-file curve contracts (``p2mem.io.las``, ``p2mem.io.inventory``,
  ``p2mem.models``) for the four approved wells (Poseidon 2, Boreas 1,
  Poseidon North 1, Proteus 1ST2). It performs LAS parsing, curve-identity
  resolution, NULL-sentinel handling, and factual inventory generation
  ONLY - no deviation-survey processing, MD-to-TVD/TVDSS transformation,
  checkshot processing, formation-top correction, petrophysical
  interpretation, or any later-phase geomechanical calculation.
* Increment 2.1 / 2.1.1 are corrective patches to Increment 2, applied
  after independent technical audits, WITHOUT changing scope or the
  underlying LAS-parsing/curve-resolution architecture (which both audits
  found sound). 2.1 corrected: canonical array naming (every array is now
  explicitly unit-suffixed, e.g. ``VP_m_s`` rather than ``DTCO``, so a
  name can never be mistaken for the wrong physical quantity or unit);
  the measured-depth curve is now located via an explicit contract role
  rather than by matching a canonical name spelled "DEPT"; several
  file-identity checks (filename, SHA-256, WELL, VERS, WRAP, NULL) that
  were not previously blocking now are; curve-coverage statistics now
  report raw AND canonical values with explicit units; and per-well
  batch failures are now typed (``p2mem.models.IngestionFailure``)
  instead of bare caught exceptions. 2.1.1 corrected a packaging-only gap
  (three notebook ``%%writefile`` cells that had drifted from their
  packaged source files). See ``INCREMENT_02_v2.1_MANIFEST.md`` and
  ``INCREMENT_02_v2.1.1_MANIFEST.md`` for the full audits and
  corrected-file checksums.
* Increment 3 adds Petrel deviation-survey ingestion with explicit
  per-file contracts (``p2mem.io.deviation``), a standard minimum-
  curvature trajectory engine with a numerically stable ratio-factor
  limit (``p2mem.trajectory``), an explicit MD-referenced/TVD-referenced/
  TVDSS depth-reference framework and MD-to-TVD/TVDSS interpolation with
  no silent extrapolation (``p2mem.depth_mapping``), and typed dataclasses
  for all of the above (``p2mem.deviation_models``) - for the same four
  approved wells. It independently reproduces the Petrel-supplied
  trajectory to millimetre scale for three of the four wells and
  discloses (rather than resolves) a real, larger trajectory-
  reconstruction discrepancy found in Proteus 1ST2's deeper section - see
  ``INCREMENT_03_MANIFEST.md``. It performs deviation-survey ingestion,
  trajectory validation, and depth mapping ONLY - no checkshot
  processing, formation-top correction, petrophysical interpretation, or
  any later-phase geomechanical calculation.
* Increment 3.1 is a corrective patch to Increment 3, applied after an
  independent technical audit, WITHOUT changing scope, equations, real
  well data, or the locked LAS/Increment-2.1.1 foundation. It corrected
  four defects: (1) the four deviation-survey source filenames are the
  exact, literal names as they exist in Google Drive, which contain
  spaces (e.g. ``"Poseidon 2_dev.txt"``) - Increment 3 had incorrectly
  substituted underscores in the contract keys, notebook mapping, and
  tests, which would have failed to resolve against the real files;
  internal well keys (e.g. ``Poseidon_2``) remain underscored and are
  unaffected; (2) every exported CSV/JSON/manifest field is now
  guaranteed to carry a basename only, never a full environment-dependent
  build path (runtime-only diagnostic objects may still retain one);
  (3) the previously undisclosed inference that the supplied ``DLS``
  column is normalized as degrees per 30 metres is now explicitly flagged
  with a new, independently per-file-verified
  ``DLS_NORMALIZATION_INFERRED_AS_DEG_PER_30M`` WARNING (mirroring the
  pre-existing MD-unit-inference warning); (4) the dogleg angle between
  successive stations is now computed with a numerically stable
  ``arctan2(||cross||, dot)`` vector formulation (``p2mem.trajectory``)
  instead of ``arccos``, which was ill-conditioned near a zero dogleg and
  previously reported a spurious ~1e-6-degree value for two stations with
  identical inclination/azimuth. The real four-well data, station counts,
  tolerances, and the unresolved Proteus 1ST2 trajectory discrepancy are
  all unchanged by this patch. See ``INCREMENT_03_1_MANIFEST.md`` for the
  full audit and re-verification record.

* Increment 4 adds checkshot (velocity survey) ingestion with explicit
  per-file contracts (``p2mem.io.checkshot``, ``config/checkshot_
  contracts.yml``), typed checkshot dataclasses (``p2mem.checkshot_
  models``), deterministic checkshot inventory/QC-table builders
  (``p2mem.io.checkshot_inventory``), and a numerical time-depth layer
  (``p2mem.time_depth``): duplicate-tie detection/conditioning, average/
  interval velocity diagnostics, checkshot-vs-locked-survey depth-
  reference comparison, forward/inverse piecewise-linear time-depth
  interpolation with explicit coverage masking (no extrapolation), LAS
  MD-to-checkshot-time mapping within validated checkshot coverage only,
  and a Poseidon-2-only sonic-checkshot drift diagnostic (trapezoidal
  integration of sonic slowness vs. the checkshot-interpolated OWT
  increment over the same MD/Depth interval). Three checkshot files are
  admitted: ``Poseidon2-Checkshot.txt`` (Poseidon 2 - the ONLY checkshot
  approved to define a primary time-depth relationship),
  ``Boreas1-Checkshot.txt`` and ``Proteus1-Checkshot.txt`` (Boreas 1 and
  Proteus 1ST2 - supporting QC data only, newly admitted in this
  increment, never transferred into Poseidon 2 as a substitute time-depth
  model). Proteus 1ST2's association with ``Proteus1-Checkshot.txt`` is
  explicitly disclosed as inferred/unverified (no embedded well
  identifier). Poseidon North 1 has no approved checkshot file
  (``checkshot_availability: NOT_AVAILABLE`` - a factual data gap, not an
  ingestion failure). This increment reuses the LOCKED Increment 1
  ``owt_to_twt``/``twt_to_owt`` unit functions and the LOCKED Increment
  3/3.1.1 ``petrel_source_trace`` survey trajectory unchanged; it performs
  checkshot QC and time-depth framework work ONLY - no formation-top
  correction, lithology interpretation, density modelling, pore-pressure
  prediction, elastic properties, rock strength, stress modelling, or
  wellbore-stability analysis. See ``INCREMENT_04_MANIFEST.md`` for the
  full technical detail and independently recomputed statistics. NOTE:
  ``INCREMENT_04_MANIFEST.md`` contained one identified defect, corrected
  by Increment 4.1 below - do not rely on its original, uncorrected
  statement that TVDSS is strictly increasing after Depth-tie conditioning
  for all three wells.

* Increment 4.1 is a narrowly scoped corrective patch to Increment 4,
  applied after an independent technical/numerical-method audit, WITHOUT
  beginning Increment 5 or any formation-top/petrophysics/pore-pressure/
  mechanical-properties/stress/wellbore-stability work. It corrected two
  defects and hardened two numerical contracts: (1) ``tvdss_to_owt``/
  ``owt_to_tvdss`` (and their TWT equivalents) previously resolved a
  repeated (tied) value on the axis being inverted by silently keeping
  whichever tied row appeared first in the Depth-conditioned table and
  discarding the other (``p2mem.time_depth._build_strictly_increasing_
  table``, REMOVED) - an ORDER-DEPENDENT tie-break with no audit trail
  beyond a bare count. This is replaced by ``build_axis_conditioned_
  lookup_table``/``build_axis_conditioned_tables_for_well``: an explicit,
  ORDER-INVARIANT policy that groups every tied value by exact equality
  regardless of parse order, registers every tied row in a new typed
  audit register (``AxisTimeDepthTieRegisterEntry`` /
  ``checkshot_time_axis_tie_register.csv`` - separate from, and never
  confused with, the pre-existing Depth-axis ``DuplicateTieRegisterEntry``
  register), and uses the tied group's dependent-value MEDIAN as the
  conditioned representative (order-invariant; disclosed as reducing to
  the arithmetic mean for the size-2 groups observed in this project's
  real data). A genuine reversal (not a tie) in the axis being inverted
  raises ``TimeDepthError`` rather than being sorted, discarded, or forced
  monotonic. (2) ``INCREMENT_04_MANIFEST.md``'s statement that TVDSS is
  strictly increasing after Depth-tie conditioning for all three wells was
  INCORRECT - independently reproduced counts (Poseidon 2: two TVDSS-axis
  and two OWT-axis ties; Boreas 1: one TVDSS-axis tie, zero OWT-axis ties;
  Proteus 1ST2: none of either) are now documented in
  ``INCREMENT_04_1_MANIFEST.md`` and reflected in this module's own
  docstrings. (3) ``trapezoidal_integrate`` and (4) ``compute_sonic_
  checkshot_drift`` are hardened to validate their numerical
  preconditions (finite, one-dimensional, equal-length, strictly
  increasing MD/x where required) rather than silently integrating
  invalid input - most notably, a decreasing or duplicate MD run can no
  longer silently produce a physically invalid NEGATIVE transit time; it
  now raises ``TimeDepthError``. None of this hardening changes the
  already-verified real Poseidon 2 sonic-drift result, which is
  bit-for-bit unchanged. See ``INCREMENT_04_1_MANIFEST.md`` for the full
  audit, corrected statistics, and re-verification record.

* Increment 4.1.1 is a narrowly scoped numerical-validation corrective
  patch to Increment 4.1, applied after an independent numerical-method/
  software-QA audit, WITHOUT beginning Increment 5 or any formation-top/
  petrophysics/pore-pressure/mechanical-properties/stress/wellbore-
  stability work. It corrected four blocking defects and one input-safety
  gap, none of which altered any previously verified REAL Poseidon
  2/Boreas 1/Proteus 1ST2 result: (1) ``build_axis_conditioned_lookup_
  table`` grouped ALL occurrences of an identical axis value together
  GLOBALLY before checking for a reversal, so a reversal that returned to
  an already-seen value (e.g. ``[100.0, 200.0, 100.0]``) was silently
  hidden rather than raising ``TimeDepthError`` - it now evaluates the
  ORIGINAL, ungrouped sequence's successive differences for negativity
  BEFORE any grouping is attempted, which is provably equivalent to the
  4.1 behavior for every legitimate adjacent tie and strictly stronger
  against a non-adjacent reversal. (2) ``compute_sonic_checkshot_drift``/
  ``find_longest_finite_positive_run`` validated MD monotonicity only
  within the selected finite-positive-VP run, so a decreasing or
  duplicate MD value outside that run (e.g. at a NaN-VP station) could
  pass silently - the COMPLETE canonical ``md_m`` array is now required
  finite and strictly increasing before run-selection (the real Poseidon
  2 MD array, 31,897 samples, was independently re-verified to already
  satisfy this). (3) ``compare_checkshot_to_survey`` reached an untyped
  NumPy ``ValueError`` ("zero-size array to reduction operation") if
  every checkshot Depth row fell outside the locked survey's own MD
  coverage - it now raises a typed ``TimeDepthError`` naming the well,
  the checkshot Depth range, and the survey MD coverage. (4)
  ``p2mem.io.checkshot.load_checkshot_surveys`` did not catch
  ``TimeDepthError`` raised during numerical conditioning, so a defect in
  one well's data could stop the entire batch - it is now caught per well
  (never via a blanket ``except Exception``) and recorded as a typed
  ``CheckshotIngestionFailure(error_type="numerical_conditioning_
  failure")``, isolated exactly like every other expected per-well
  failure. (5) ``seconds_to_milliseconds``/``milliseconds_to_seconds``
  coerced their input directly, unlike ``p2mem.units``'s Increment-1
  input-safety policy, so a boolean, numeric-looking string, or complex
  value would be silently reinterpreted rather than rejected - both now
  reject such input with ``TypeError`` via a local, documented copy of
  ``p2mem.units``'s identical private dtype check (``p2mem/units.py``
  itself remains LOCKED and unmodified). See
  ``INCREMENT_04_1_1_MANIFEST.md`` for the full audit, the regression-test
  list, and the re-verification record.

* Increment 5 adds contract-driven formation-top ingestion, HRS-versus-
  selected-readable source RECONCILIATION, and survey-corrected
  stratigraphic depth mapping (``p2mem.top_models``, ``p2mem.io.tops``,
  ``p2mem.io.tops_inventory``) for the two approved wells with formation-
  top data (Poseidon 2, Boreas 1). Each well has TWO independently
  supplied top files - an "HRS" file (``Top_Name``/``MDRT_m`` only, no
  well name in the file body) and a "selected readable" file
  (``TOP_NAME``/``MDRT_M``/``TVDSS_M``/``NOTE``, with ``#``-comment
  headers and a dashed separator line, deliberately parsed rather than
  treated as data) - and neither is silently preferred: markers are
  matched by exact/normalized name or an explicit human-authored alias
  contract (never fuzzy matching), and MDRT is cross-checked between the
  two sources within a documented 0.005 m tolerance; a marker on which
  the two sources disagree beyond that tolerance is excluded from depth
  mapping (``mapping_status == "not_mapped_mdrt_unresolved"``) rather
  than resolved by picking one file. Reconciled MDRT is mapped through
  the LOCKED Increment 3.1.1 ``petrel_source_trace`` survey trajectory
  using the existing, unmodified ``p2mem.depth_mapping.
  map_las_md_to_tvd_tvdss`` (per marker, so one out-of-coverage marker
  never blocks the rest of the well; extrapolation is never performed),
  producing ``TVD_survey_m``/``TVDSS_survey_corrected_m`` alongside an
  explicit, unambiguous ``TVDSS_residual_source_minus_survey_m =
  TVDSS_source_m - TVDSS_survey_corrected_m`` residual field. Both file
  representations for both wells are classified
  ``well_identity_evidence_status = "inferred_unverified"`` (filename-only
  or in-file-comment association, never independently content-verified -
  a more conservative classification than Increment 4's checkshot files).
  Poseidon North 1 and Proteus 1ST2 have no approved formation-top file
  and are recorded as ``FormationTopAvailabilityRecord(..., "NOT_
  AVAILABLE")`` - never substituted or depth-correlated from another
  well. Independently recomputing (never hardcoding) this increment's two
  required regression findings against the real approved files confirmed:
  Poseidon 2's "selected readable" file's supplied TVDSS equals
  ``MDRT - 21.8 m`` EXACTLY for every one of its 9 markers (21.8 m is the
  well's own rotary-table elevation) - a literal vertical-well-assumption
  depth-reference defect, corrected in the derived
  ``TVDSS_survey_corrected_m`` representation while the raw
  ``TVDSS_source_m`` column is preserved unmodified; survey-corrected
  residuals reproduce Sea Bed at ~0 m, Plover Fm (Top Reservoir) at
  +1.68 m, and TD at +2.49 m, exactly as the approved Rev 1 design
  anticipated. Boreas 1's supplied TVDSS does NOT follow that pattern and
  its maximum absolute source-versus-survey residual is 0.0416 m, below
  the approved 0.05 m tolerance - confirming it was generated from the
  well's real surveyed trajectory, not a vertical-well shortcut, and it is
  therefore NOT "corrected" the way Poseidon 2 is. See
  ``INCREMENT_05_MANIFEST.md`` for the full audit, the per-file contracts,
  and the complete reconciliation/mapping results.

* Increment 5.1 is a narrowly scoped corrective patch to Increment 5,
  applied after an independent technical/software-QA audit, WITHOUT
  beginning Increment 6 or any petrophysics/pore-pressure/mechanical-
  properties/stress/wellbore-stability work, and WITHOUT changing any
  previously verified real Poseidon 2/Boreas 1 formation-top result. It
  corrected three defects and one documentation-accuracy gap: (1)
  ``reconcile_formation_top_sources`` documented "zero canonical markers
  in common between the two sources" as a fatal ``NO_COMMON_MARKERS``
  ERROR, but actually tested the emptiness of the UNION of both sources'
  canonical names - so two entirely DISJOINT, non-empty marker sets (e.g.
  HRS names sharing nothing with the readable file's names) were silently
  accepted as one-sided ``NOT_COMPARABLE`` entries instead of being
  rejected; the check now explicitly evaluates the INTERSECTION of the two
  sources' canonical names and raises the documented ERROR (and
  ``load_formation_top_well`` consequently raises
  ``TopSourceReconciliationError``) whenever both sources are non-empty but
  share nothing, while a legitimate one-sided marker (with at least one
  marker genuinely shared) remains the pre-existing, non-fatal
  ``NOT_COMPARABLE`` case. (2) ``load_formation_top_surveys`` recorded only
  the HRS file's path in every ``TopIngestionFailure.source_path``,
  regardless of which file/stage actually failed - so a failure originating
  from the "selected readable" file (missing file, malformed content, or a
  contract mismatch) could leak that file's own absolute path, unsanitized,
  into exported issues/availability/manifest rows (the sanitizer only ever
  stripped the recorded, and in that case WRONG, HRS path).
  ``TopIngestionFailure`` now carries an explicit ``failure_origin``
  ("hrs"/"readable"/"reconciliation"/"mapping"/"unknown") and both
  ``hrs_path``/``readable_path`` fields, ``load_formation_top_well`` tags
  each raised exception with the stage that actually failed, and every
  exporting function in ``p2mem.io.tops_inventory`` now sanitizes BOTH
  candidate paths (literal substring replacement only, never a regex) and
  reports the correctly identified failing file's basename as context. (3)
  ``reconcile_formation_top_sources`` - the public in-memory API, as
  distinct from the file parsers, which already enforced this - did not
  validate its own documented contract before any numerical comparison:
  non-finite (NaN/Inf) or negative MDRT/TVDSS values, a shorter
  ``NOTE_source`` tuple (previously reaching an untyped ``IndexError``), a
  non-1-dimensional array, and an unvalidated ``mdrt_agreement_tolerance_m``
  keyword (previously accepting ``NaN``, a negative value, or a boolean,
  reaching an incidental ``TypeError`` only for a string) were all silently
  accepted or reached an undocumented incidental error. All of these are
  now rejected before any comparison, with deliberate, documented
  exceptions (``TopParsingError`` for a value/structural defect,
  ``TypeError`` for a type-class defect - matching this module's existing
  split), while valid Python ``int``/``float`` and NumPy integer/floating
  scalars (including 0-d/size-1 arrays) continue to work. (4)
  ``INCREMENT_05_MANIFEST.md`` stated that no ``/home/``, ``/root/``,
  ``/content/``, or absolute path exists ANYWHERE in the package - this was
  inaccurate, since existing synthetic tests and historical documentation
  intentionally contain fake absolute-path strings as test inputs; the
  precise, narrowly scoped claim ("no environment-dependent build path
  appears in exported CSV/JSON outputs") is now stated explicitly in
  ``INCREMENT_05_1_MANIFEST.md``, which also acknowledges the prior
  wording was overbroad. ``INCREMENT_05_MANIFEST.md`` itself is a locked
  historical record and is NOT rewritten. See ``INCREMENT_05_1_MANIFEST.md``
  for the full audit, the regression-test list, and the real-data
  non-regression verification record.

* Increment 5.1.1 is a further narrowly scoped corrective patch to
  Increment 5.1, applied after an independent technical/software-QA audit,
  WITHOUT beginning Increment 6 and WITHOUT changing any scientific result,
  tolerance, depth-mapping method, or formation-top contract. It corrected
  one remaining reconciliation edge case and three documentation-accuracy
  gaps. (1) Increment 5.1 correctly rejected two non-empty, disjoint
  marker sets as a fatal ``NO_COMMON_MARKERS`` ERROR, but its condition
  (``both sources non-empty AND intersection empty``, plus a separate
  both-empty case) still silently accepted the remaining zero-common-
  markers configuration: exactly ONE source entirely empty and the other
  non-empty - the intersection of an empty set with anything is itself
  empty, so this was still a zero-common-markers condition, but the
  ``hrs_by_canon and readable_by_canon`` non-empty guard skipped it.
  ``reconcile_formation_top_sources`` now uses the single, strictly
  correct check ``if not common_markers`` (the intersection of the two
  sources' canonical names), which is empty in every zero-common-markers
  configuration - both empty, either side alone empty, or both non-empty
  and disjoint - and is never empty whenever at least one canonical marker
  is genuinely shared, so a legitimate one-sided marker alongside at least
  one shared marker remains the pre-existing, non-fatal ``NOT_COMPARABLE``
  case. (2) ``INCREMENT_05_1_MANIFEST.md`` stated that the real Increment
  5 baseline ZIP's SHA-256 "matches" a governing-prompt-supplied hash that
  was, in fact, a two-character truncation of the real 64-character value
  - a mathematical impossibility that is now stated transparently in
  ``INCREMENT_05_1_1_MANIFEST.md`` as a documentation/input typo, never as
  baseline corruption or uncertainty. (3) ``INCREMENT_05_1_MANIFEST.md``
  Section 7 stated that ``outputs/`` is excluded from the delivered ZIP;
  this was false - the delivered Increment 5.1 ZIP packages the
  ``outputs/`` tree (including the byte-identical Increment 5 formation-
  top outputs/figures) exactly as every prior increment's ZIP has; only
  the dev-only regenerated-comparison directories and private raw source
  files are excluded, and this is now stated accurately. (4) the same
  manifest's clean-room section stated that no ``*.las`` file exists in
  the package; this was false because small, intentionally packaged
  synthetic LAS/checkshot/deviation/top fixtures exist under
  ``tests/fixtures/`` for portable testing - the corrected wording
  distinguishes these fictional fixtures (never leaked private data) from
  the genuinely excluded real/private project LAS, deviation, checkshot,
  and formation-top source files. See ``INCREMENT_05_1_1_MANIFEST.md`` for
  the full audit, the regression-test list, and the real-data
  non-regression verification record.

Increment 6 (this release, v0.6.0) adds the gamma-ray QC, shale-proxy
sensitivity, well-frame assembly, and method-eligibility framework, on top
of the LOCKED Increment 1-5.1.2 foundation (no locked module, config,
test, fixture, notebook, output, or figure is modified by it). It
contributes:

* ``p2mem.wellframe_models`` / ``p2mem.wellframe`` - a typed, auditable
  per-well assembly of the locked canonical LAS curve arrays alongside the
  locked MD->TVD/TVDSS mapping, with per-sample validity masks, per-curve
  provenance, QC flags, and evidence classification. Sample count and
  order are preserved exactly; invalidity is expressed only through masks;
  no depth is ever extrapolated (a LAS sample outside the survey's own MD
  coverage is recorded as depth-unmapped, never clamped or held).
* ``config/petrophysics_eligibility.yml`` - the human-authored, reviewable
  per-well gamma-ray-family disposition, endpoint-sensitivity policy, and
  eligibility rules. Boreas 1 is formally excluded from every
  lithology-dependent, GR-normalized, shale-proxy and NCT-candidate
  calculation under the machine-readable reason
  ``BOREAS_ECGR_SCALE_UNRESOLVED``; it is EXCLUDED, never corrected or
  rescaled, because no calibration evidence exists to support any
  correction.
* ``p2mem.petrophysics_models`` / ``p2mem.petrophysics`` - factual
  GR-family QC statistics for every well (including excluded ones),
  per-well low/base/high endpoint scenarios whose measured endpoint values
  are recorded, and the dimensionless GR index with clipped and unclipped
  results retained side by side. The only permitted shale-proxy transform
  is the linear identity of the clipped index, exported under the
  self-labelling name ``VSH_GR_linear_proxy_frac``; every nonlinear Vsh
  transform (Larionov, Clavier, Stieber, ...) is deliberately DEFERRED
  pending a retrieved, verified primary-source method record.
* ``p2mem.method_eligibility`` - three input-admissibility masks
  (``eligible_density_for_sv``, ``eligible_dynamic_elastic``,
  ``eligible_sonic_nct_candidate``) plus contiguous-interval registers on
  MD/TVD/TVDSS with explicit, configured, tested gap tolerances that
  distinguish sample-count continuity from physical-depth continuity.
  ELIGIBILITY IS NOT VALIDITY: none of the gated methods is implemented,
  fitted, or validated here.
* ``p2mem.io.petrophysics_inventory`` - deterministic summary/interval/
  manifest builders that never export a per-sample real-data array and
  never emit an absolute path.

Increment 6 assigns NO named lithology anywhere. Gamma-ray response is not
uniquely diagnostic of rock type and no independent lithological evidence
exists in this project, so every classification it produces describes DATA
AND PROXY CONFIDENCE ONLY, guarded by an enforced prohibited-vocabulary
check.

Increment 6.1 (v0.6.1) is a narrowly scoped corrective patch
to Increment 6, applied after an independent geomechanics/rock-physics and
software audit. It changes no architecture and starts no new phase. Four
findings were corrected:

* **Vp/Vs terminology and boundaries.** Increment 6 labelled every excluded
  Vp/Vs ratio "non-physical" and used a strictly exclusive sqrt(2) bound.
  Both were wrong. With r = Vp/Vs, nu = (r^2 - 2) / (2 (r^2 - 1)) and
  K = rho (Vp^2 - (4/3) Vs^2), so: r = sqrt(2) gives nu = 0 EXACTLY and must
  be ACCEPTED by a non-negative-nu policy (the bound is now INCLUSIVE);
  sqrt(4/3) < r < sqrt(2) gives POSITIVE bulk modulus with negative Poisson
  ratio - unusual and outside this project's conservative policy, but not
  physically impossible; only r <= sqrt(4/3) implies a non-positive bulk
  modulus and is genuinely outside the isotropic elastic model; and r > 4 is
  a CONFIGURED PLAUSIBILITY LIMIT, not a Poisson-domain boundary (nu ~= 0.467
  there). The condition is renamed a CONFIGURED NON-NEGATIVE-POISSON-RATIO
  APPLICABILITY SCREEN, the three config bounds are separately named, and
  exclusions are diagnosed by regime (`n_ratio_nonpositive_bulk_modulus`,
  `n_ratio_positive_bulk_but_negative_poisson`,
  `n_ratio_above_configured_plausibility_max`, `n_vp_not_greater_than_vs`) -
  never aggregated under a single "non-physical" count.
* **Poseidon North 1 depth-tied status.** Its machine-readable `use_status`
  was `screening_proxy_allowed` while it has no approved formation tops,
  contradicting both the status vocabulary and the prose limitation. It is
  now `screening_proxy_allowed_depth_tied`, and a config invariant makes any
  contradictory has-tops/use-status pairing fail config loading loudly.
* **Unsupported lithology claims and a circular gate.** Project-specific
  "clastic-dominated" assertions and the unsupported "regionally persistent"
  cross-well claim are removed (two of the three GR-eligible wells have no
  approved tops, so cross-well stratigraphic persistence cannot be
  established). `clastic` and related rock-class terms join the prohibited
  vocabulary, and `named_lithology_assigned` is now DERIVED from an actual
  validation pass over every persisted label, per-well note, mask name and
  manifest statement - injecting a prohibited term makes the validation, the
  manifest flag and the completion gate fail together. Validation is
  three-tier: LABELS admit no prohibited term at all, per-well PROSE may name
  the method category ("shale proxy") but not assert a rock, and EXPLANATORY
  text may use rock names as generic examples of gamma-ray non-uniqueness
  but never in a sentence naming one of this project's wells.
* **Gross versus net/strict interval thickness.** The reported "qualifying
  thickness" summed block endpoint spans that may contain explicitly bridged
  ineligible samples. Thickness is now exported only under qualified names -
  `gross_thickness_*_m` (endpoint span, bridging-inclusive) and
  `net_thickness_*_m` (bridged gaps removed) - every mask is additionally
  decomposed under a `strict_no_gap` contiguity policy for comparison, every
  sensitivity case reports its bridged-sample and interrupted-block counts,
  and Figure 4 now draws qualifying and rejected sub-threshold blocks
  distinctly with the counted population stated.

Increment 6.1.1 (v0.6.2) is a second, narrower corrective
patch, applied after an independent audit of Increment 6.1. It starts no
new phase, changes no architecture, and alters no configured tolerance,
threshold, or qualifying-block policy. Five findings were corrected:

* **Named-lithology validator bypass.** The method-term allowlist that lets
  per-well prose name a METHOD ("shale proxy") contained "shale gas", which
  is not a method-category phrase but can be a direct geological/hydrocarbon
  assertion; sentences such as "This interval contains shale gas." therefore
  passed with zero violations. The allowlist is reduced to the two phrases
  that are genuinely method/quantity names required to describe this
  project's boundaries - ``shale proxy`` and ``shale volume`` - and the
  allowance is now evaluated PER SENTENCE and suppressed entirely in any
  sentence carrying a geological-assertion cue (``contains``, ``comprises``,
  ``bearing``, ``facies``, ...) or where the phrase is immediately preceded
  by a quantity cue (``has a high shale volume``). Regression tests prove
  the allowlist can no longer hide a geological assertion and that injecting
  either bypass case into manifest content makes ``named_lithology_assigned``
  true, ``lithology_validation.n_violations`` non-zero, and the completion
  gate fail.
* **Stale Vp/Vs scientific description.** The active top-level docstring of
  ``p2mem.method_eligibility`` still described the screen as keeping samples
  within the Poisson domain under a strictly EXCLUSIVE sqrt(2) ratio bound,
  contradicting the corrected implementation. It now states the configured
  non-negative-Poisson-ratio applicability screen with an INCLUSIVE
  ``Vp/Vs >= sqrt(2)`` bound and says explicitly that this is neither a
  physical-possibility test nor a boundary of the mathematical Poisson
  domain. A regression test asserts the corrected wording against the live
  module documentation.
* **Ambiguous interruption counts.** ``n_interruptions`` was a boolean-like
  flag reported as a count (always 1 for every bridged block), and
  ``n_interrupted_subruns`` did not describe what it counted. Interval
  records now carry three separately named, independently meaningful
  quantities: ``n_bridged_samples`` (ineligible samples absorbed inside the
  gross block), ``n_bridged_gaps`` (distinct bridged runs) and
  ``n_eligible_subruns`` (strictly contiguous eligible sub-runs), with the
  identity ``n_bridged_gaps == n_eligible_subruns - 1`` enforced at
  construction. The ambiguous aliases are gone from all active exports.
* **Notebook output-count error.** The notebook creates and checks eight
  deterministic CSV/JSON outputs but its gate text and printed label said
  seven. Both now say eight, and the gate asserts both the declared count
  and the existence of all eight files. Four figures remain separate,
  giving twelve outputs/figures in total.
* **Incorrect Increment 6.1 delta arithmetic.** The Increment 6.1 completion
  record stated "Changed (12)"; a clean recursive comparison gives 18
  changed, 3 added and 0 removed (21 path differences). That statement is
  explicitly superseded - not silently rewritten - in the Increment 6.1.1
  manifest and completion record, which report both measured deltas file by
  file.

Increment 6.1.2 (v0.6.3) is a third corrective patch, applied
after an independent audit of Increment 6.1.1. It starts no new phase, changes
no architecture, alters no scientific threshold, tolerance, endpoint scenario,
contiguity policy, GR disposition, depth-mapping rule, or real-data
interpretation, and adds no runtime dependency (NumPy and PyYAML remain the
only two). Two residual findings were corrected:

* **The named-lithology validator was bidirectionally incorrect.** Increment
  6.1.1 decided the method-phrase allowance from a SENTENCE-WIDE assertion-cue
  list plus a fixed LOOK-BEHIND window. Both were structurally wrong. Because
  nothing to the RIGHT of an allowed phrase was ever inspected, and because
  the normalizer destroyed possessives, direct assertions such as "Poseidon
  2's shale volume is high.", "Poseidon 2's shale volume is 70 percent." and
  "The interval's shale volume exceeds 60 percent." passed with ZERO
  violations. Conversely, because a sentence-wide cue fires without asking
  what the cue word is predicated OF, legitimate method and boundary
  statements such as "This method contains a shale proxy calculation." and
  "The analysis shows no shale volume was computed." were reported as
  lithological assertions. The allowance is no longer a property of the
  sentence: it is decided for EACH OCCURRENCE of an allowed phrase from that
  occurrence's own local grammar - whether it is possessed by a geological
  entity (``the interval's shale volume``), modified by a magnitude (``a high
  shale volume``), predicated with an amount or a dominance (``is 70
  percent``, ``exceeds 60 percent``, ``dominates the interval``), or sits in a
  sentence whose composition verb takes a GEOLOGICAL subject (``the unit
  comprises ...``, but not ``this method contains ...``). Apostrophes and
  possessives are parsed rather than erased, sentence boundaries include line
  breaks, and genuinely ambiguous project-specific sentences fail closed.
  ``SCOPE_LABEL`` remains absolute: a label gets no latitude at all.

* **Interval-record invariants were only partially enforced.** Increment 6.1.1
  claimed the interruption-count identities were enforced at construction, but
  the only check was ``n_bridged_gaps == n_eligible_subruns - 1``, skipped
  whenever either value was ``None`` or ``n_eligible_subruns`` was 0. Records
  with missing counts, negative counts, zero sub-runs, boolean counts, or
  bridged samples without a bridged gap were all constructible. All three
  counts are now validated together as one coherent record: each is required,
  must be a true integer count (booleans, strings, complex values and
  fractional floats are rejected, never coerced), ``n_bridged_samples >= 0``,
  ``n_bridged_gaps >= 0``, ``n_eligible_subruns >= 1``, ``n_bridged_gaps ==
  n_eligible_subruns - 1``, ``n_bridged_samples == 0`` if and only if
  ``n_bridged_gaps == 0``, and ``n_bridged_samples >= n_bridged_gaps`` because
  every distinct gap holds at least one sample. Each violation raises
  ``PetrophysicsInputError`` naming the offending field or relationship, and
  the removed ``n_interruptions`` / ``n_interrupted_subruns`` field names are
  rejected outright rather than silently ignored.

Every scientific result is unchanged by this patch: the same eight
deterministic outputs and four figures are produced, byte for byte.

Increment 6.1.3 (v0.6.4) is the fourth and architectural
corrective patch, applied after an independent audit of Increment 6.1.2. It
starts no new phase, changes no scientific threshold, tolerance, endpoint
scenario, contiguity policy, GR disposition, depth-mapping rule, or real-data
interpretation, and adds no runtime dependency (NumPy and PyYAML remain the
only two).

**The validator no longer tries to understand English.** Increments 6.1.1 and
6.1.2 each attempted to decide, from grammar, whether a sentence containing a
rock name was NAMING A METHOD or ASSERTING GEOLOGY - 6.1.1 with a
sentence-wide cue list and a fixed look-behind window, 6.1.2 with
per-occurrence possessive/modifier/predicate/subject analysis. Both were
audited and both failed in BOTH directions. 6.1.2 still missed "The interval
has a shale volume.", "The shale volume in Poseidon 2 exceeds 60 percent." and
"Poseidon 2 shale volume was determined to be high.", while wrongly rejecting
"The well contains no shale volume estimate." Every one of those is a list gap
or a window edge. The lesson is not that the lists were too small: free-text
geological-assertion detection is unbounded, and no finite grammar closes it.

Increment 6.1.3 replaces judgement with membership. Every scope is now decided
by set membership or exact equality:

* ``SCOPE_LABEL`` - verdicts. Zero allowance.
* ``SCOPE_INTERPRETIVE`` - project-specific prose. **ZERO ALLOWANCE.** There is
  no exemption path at all, so there is nothing to bypass. The
  ``allow_method_phrases`` parameter is gone.
* ``SCOPE_METHOD`` (new) - method and limitation statements. The text must
  EQUAL a member of the closed, provenance-tagged ``METHOD_STATEMENTS``
  registry. Not matched, not scored - equal. The registry has five members,
  measured rather than guessed by scanning every string literal in active
  packaged source and every persisted field under a zero-allowance rule.
* ``SCOPE_EXPLANATORY`` - generic scientific text. A rock name is permitted
  only when the sentence refers to no project well AND no project rock body.
  6.1.2 checked well names alone, which is why an assertion about "the
  interval" behaved inconsistently between scopes.

The deletion is the correction: ``ALLOWED_METHOD_TERM_PHRASES``,
``GEOLOGICAL_ENTITY_TOKENS``, ``COMPOSITION_PREDICATES``,
``MAGNITUDE_PREDICATES``, ``MAGNITUDE_WORDS``, ``COPULAR_VERBS``,
``ATTRIBUTIVE_ROCK_TOKENS``, ``NEGATION_TOKENS``, the look-behind and
look-ahead windows, subject resolution and occurrence classification are all
GONE, and a regression test asserts none of them is importable. Correctness is
proved by CLOSURE - every prohibited term, in every position, inside carrier
prose built from the exact constructions that defeated both previous
implementations - not by a list of example sentences.

This patch also closes a gap neither audit reported: three strings this project
WRITES INTO PACKAGED EXPORTS had never been inside the validated scope in any
increment, including a ``calibration_status`` value literally containing
``not_a_shale_volume`` persisted to every row of
``gr_proxy_sensitivity_summary.csv``. All three are now scanned in
``SCOPE_METHOD``, which raises ``n_fields_checked`` from 138 to 143.

**Interval-record type gate.** Increment 6.1.2 enforced the relational rules
but let three type defects through: ``0.0 / 0.0 / 1.0`` was accepted although
the contract requires integers; ``NaN`` and ``Inf`` escaped as bare
``ValueError`` / ``OverflowError`` from ``int()``; and an unknown keyword such
as a misspelled count name was silently ignored, leaving the real count unset.
Keyword acceptance is now a WHITELIST over the declared slots, floats are
rejected outright including whole-valued ones, and non-finite values raise a
typed ``PetrophysicsInputError`` before any conversion is attempted.

All eleven scientific outputs and figures are byte-identical to Increment
6.1.2; only the manifest changes, by the single scalar named above.

Increment 6.1.4 (v0.6.5) completes the architecture Increment
6.1.3 began. It is deliberately narrow: one scope rule, no other change.

Increment 6.1.3 closed LABEL and INTERPRETIVE by removing every exemption and
closed METHOD by exact registry membership - but left EXPLANATORY decided by a
FINITE TOKEN LIST of project references. That is the same shape of rule that
failed in 6.1.1 and 6.1.2, and it had the same defect. Ordinary stratigraphic
and exploration nouns were absent from the list, so in explanatory scope

    "The member is shale."               passed
    "The group is limestone."            passed
    "The package is a clean sandstone."  passed
    "The play is shale-dominated."       passed
    "The prospect is carbonate."         passed
    "The target is sandstone."           passed

while "The upper member is a clean sandstone reservoir." failed only because
``reservoir`` happened to be listed - an accident, which is what a list-shaped
rule produces.

The rule is INVERTED and made identical in kind to SCOPE_METHOD: explanatory
text carrying a prohibited term is a violation UNLESS the text is a member of
the closed ``GENERIC_EXPLANATORY_STATEMENTS`` registry. It FAILS CLOSED, so no
vocabulary gap can admit anything. ``PROJECT_WELL_NAME_TOKENS``,
``PROJECT_ROCK_BODY_TOKENS`` and ``PROJECT_REFERENCE_TOKENS`` are deleted, and
a regression test asserts none of them is importable. Nothing in the validator
now enumerates what a project reference looks like.

The generic registry is **empty**, as a measured fact rather than an omission:
this project persists exactly one explanatory field,
``manifest.named_lithology_statement``, and it carries no prohibited term at
all. The mechanism is nevertheless live and tested, because later increments
explaining gamma-ray non-uniqueness may genuinely need to write "a clean
sandstone and a clean limestone read alike on GR" - and when they do, that is
a registered, reviewable statement rather than a sentence admitted by shape.

All four validation scopes are now closed by membership or exact equality, and
the closure proof covers all of them. All twelve outputs and figures are
byte-identical to Increment 6.1.3.

Increment 6.1.5 (v0.6.6) replaces blacklist-based acceptance
with POSITIVE AUTHORIZATION, and is the final Increment 6 corrective patch.

Increments 6.1 through 6.1.4 all asked the same question in different ways:
"does this text contain a geological assertion?" All four shared one
structural assumption - that content is ACCEPTABLE BY DEFAULT and becomes
unacceptable only when a recognizer fires - and all four were defeated,
finally by a word no recognizer had been given. In the 6.1.4 package these
all passed label, interpretive and explanatory scope:

    "The interval is chalk."         "The interval is chert."
    "The interval is halite."        "The interval is tuff."
    "The interval is gypsum."        "The interval is basalt."
    "The interval is conglomerate."  "The interval is dolostone."
    "The interval is lignite."       "The interval is calcareous."

and so did "The interval is qxzite.", a word that does not exist. They already
failed in METHOD scope, which was the only scope then requiring registration -
and that is the clue this patch acts on. A longer blacklist would have caught
the first ten and still missed the eleventh, so lengthening it is neither a
completion criterion nor the mechanism this package's assurance rests on.

The model is inverted. Nothing is acceptable by default:

* ``SCOPE_LABEL`` - the value must be a member of ``APPROVED_LABELS``, a
  registry of typed, enumerated label values each declaring its field kind,
  purpose and provenance. Arbitrary caller-supplied label text is rejected
  even when it contains no recognizable rock name at all.
* ``SCOPE_INTERPRETIVE`` - the text must resolve to a registered
  ``statement_id``, or to a reviewed ``template_id`` whose substitutions are
  strictly typed. Unregistered free text is rejected unconditionally.
* ``SCOPE_METHOD`` - the text must resolve to a registered ``statement_id``,
  and the id and the exact rendered text are validated TOGETHER, so neither a
  renamed id nor an edited sentence passes on the strength of the other.
* ``SCOPE_EXPLANATORY`` - identical, for every non-empty statement, whether or
  not any prohibited term is detected. The early-pass behaviour equivalent to
  "if no prohibited term is found: accept" is gone from every scope.

The registries were MEASURED from the actual persisted Increment 6 export, not
designed: 17 approved label values, 17 registered statements (11 interpretive,
5 method, 1 explanatory) and 1 controlled template covering the three per-well
confidence rationales, whose only variable parts are three decimal literals.
Every registered entry carries a stable id, its scope, its exact text or
controlled template, a scientific purpose, a provenance justification, and its
permitted typed substitutions. Duplicate ids fail at import.

``PROHIBITED_LITHOLOGY_TERMS`` survives ONLY as a supplementary diagnostic
linter. It authorizes nothing, and it is never cited as evidence that all
named lithologies have been detected - it cannot be: its 28 terms include
neither ``chalk`` nor ``chert`` nor ``qxzite``, and every rejection listed
above happens with that linter returning empty.

THE DEFENSIBLE ASSURANCE STATEMENT, which supersedes the wording of every
earlier Increment 6 manifest: every persisted project-specific classification
and interpretive statement is generated from an approved typed value, a
controlled template, or a registered statement, and arbitrary free text cannot
enter these controlled fields. This is NOT a claim that the software
understands or exhaustively recognizes natural-language lithology; it does
not, and no earlier version did. Free-form notebook narrative and
documentation lie outside these controlled fields and remain subject to
ordinary manual scientific review.

All twelve outputs and figures are byte-identical to Increment 6.1.4.

Increment 6.1.6 (previous corrective patch, v0.6.7) enforces authorization at the EMISSION
BOUNDARY. Increment 6.1.5's positive-authorization model was sound but was
applied to a scope the manifest builder RECONSTRUCTED from dispositions,
confidences and masks - a parallel object, not the records written to disk. A
`GrEndpointScenario` carrying `description="The interval is chalk."` was
persisted verbatim by `build_gr_endpoint_scenario_rows()` while never entering
that 143-field scope, so it could not move `named_lithology_assigned`. The unit
validator was closed; the export path was not.

Five findings, all reproduced first:

* **Exported endpoint description bypass** - closed. The row builder now
  authorizes the value it is about to place in the record and raises
  ``PetrophysicsInputError`` if it cannot.
* **Incomplete output coverage** - closed. ``p2mem.io.output_policy`` declares a
  category for EVERY string column and JSON path of all eight artifacts (105
  entries). Nothing is unclassified; unknown artifacts, columns, JSON paths and
  categories fail closed. Categories: structural enum, identifier, filename,
  typed label, registered statement, controlled template, structured diagnostic,
  sanitized diagnostic. There is no general category admitting arbitrary prose.
* **Field-kind label mismatch** - closed. ``APPROVED_LABELS`` is keyed by
  ``(field_kind, value)``; ``use_status="GR"`` and ``mask_name="measured"`` now
  fail, the field kind is supplied by the caller and never parsed from a context
  string, and duplicate pairs fail at import.
* **Stale exported derivation** - closed. The superseded prohibited-term wording
  is replaced by a registered statement describing the model actually
  implemented, so the assurance prose is itself authorized.
* **Duplicated manifest statement** - closed. The manifest takes
  ``REGISTERED_STATEMENTS["named_lithology_statement"].text``; no second literal
  exists to diverge from it.

Export is two-stage: build the exact pre-serialization records, authorize every
string occurrence, write, then RE-READ the written bytes and authorize again.
A value mutated after authorization is caught before delivery, and nothing is
written at all if any field fails.

The assurance metrics now say what they count. ``n_fields_checked=143`` is gone;
the manifest reports ``scope_object_fields_checked`` alongside an
``emitted_field_coverage`` block giving total emitted string-field occurrences,
controlled occurrences, structural occurrences, occurrences outside the
guarantee, unclassified fields, unauthorized fields and field-kind mismatches.
Machine diagnostics and the operator-facing issue text are counted separately
and are explicitly OUTSIDE the controlled-interpretation guarantee.

All seven CSV artifacts and all four figures are byte-identical to
Increment 6.1.5. ``petrophysics_eligibility_manifest.json`` changes, and only
inside ``lithology_validation``: the derivation is corrected and the coverage
metrics become truthful. No numerical, disposition, mask, threshold or depth
value moves.

Increment 6.1.7 (this release, v0.6.8) corrects the remaining export-boundary
assurance defect. Increment 6.1.6 discovered fields from non-empty string
values, so missing controlled columns, empty or numeric-looking controlled
strings, non-string substitutions and unknown fields with non-prose values
could escape collection. It also wrote into the official directory before the
post-write check and compared aggregate coverage counts, allowing an
authorized value to be changed into a different authorized value without
detection and leaving partial artifacts after a rejected write.

Increment 6.1.7 defines exact schemas for all eight artifacts and validates
artifact inventory, ordered CSV columns, JSON keys, requiredness, strict types
and finite numerics independently of prose authorization. Every schema-declared
string occurrence is then authorized, including empty and numeric-looking
strings. Candidate files are written only to an isolated sibling directory,
re-read and re-validated, and every typed field and row is compared against the
authorized pre-serialization record before failure-atomic publication. Unknown
well identifiers, stale output artifacts, serializer failures, field additions
or deletions, row reordering, type changes and authorized-to-authorized value
changes all fail closed while leaving the official destination unchanged.
This is failure-atomic for handled process errors; it is not a claim of
multi-file atomicity across power loss or operating-system failure.

Increment 6.1.7 is now LOCKED. Every module, config, test, fixture, notebook,
manifest, ledger, output and figure it delivered is byte-identical in this
release; the only pre-existing files that change are this module's version and
narrative, ``pyproject.toml``'s version, and ``README.md``.

Increment 7 / 7.0.1 / 7.0.2 / 7.0.3 / 7.0.4 (this release, v0.7.4) adds DENSITY QC,
DENSITY-COVERAGE QUALIFICATION, and a VERTICAL OVERBURDEN-STRESS FRAMEWORK.

Increment 7.0.1 is a narrowly scoped corrective patch. It makes profile
truncation disposition-driven so every unresolved internal gap stops the
integral, including a short gap left unresolved because bridging is disabled
or a bridge precondition fails. It also rejects coercible non-numeric scalar
inputs and malformed closed configuration vocabularies. The 10 m bridge
threshold is documented as a sensitivity-tested heuristic rather than a
mathematical error bound, and the shallow-column endpoints are labelled as
conditional/illustrative scenarios rather than physical bounds.

Increment 7.0.2 is an assurance-only corrective patch. It fixes the completion
gate's scenario-basis lookup so the gate reads the emitted
``assumed_density_basis`` field declared by the real CSV schema, and it makes
the public frozen ``OverburdenConfig`` constructor enforce the same strict
types, finite numerics, closed vocabularies and internal invariants as the YAML
loader. This closes bypasses through direct construction and
``dataclasses.replace`` without changing any scientific input, calculation,
threshold, eligibility result or output artifact.

Increment 7.0.3 closes one configuration-to-computation provenance defect.
The high shallow-column scenario is implemented from the explicitly stored
same-well P05 density statistic, so ``scenario_high_percentile`` is now
required to equal exactly 5.0 at both YAML loading and public constructor
entry points. Earlier releases accepted other percentile labels even though
the calculation continued to use P05. The packaged configuration already
uses 5.0, so no scientific input, calculation, threshold, eligibility result,
CSV/JSON artifact or figure changes in this patch.

Increment 7.0.4 is a final corrective patch following an independent audit of
the full data and publication paths. The high shallow-column endpoint now uses
the P05 of the eligible integration population, not all finite density values;
per-call gap thresholds and all threshold-bearing records reject boolean,
textual, complex, negative and non-finite values; scenario totals disclose
assumed, conditioned and measured fractions that sum to one; the locked seabed
reader fails closed on missing, malformed, duplicate or non-finite matching
rows; and the JSON exporter writes explicit UTF-8 LF bytes on every platform.
The four approved wells have identical finite and eligible RHOB populations,
zero bridged contribution in the two published scenario wells, and a valid
unique seabed table, so the measured scientific values remain unchanged. The
assurance JSON and scenario/QC schemas change to state these contracts
truthfully. Increment 7.0.4 is now LOCKED.

What it does
------------
* ``config/overburden_stress.yml`` - the human-authored screening policy: the
  bulk-density plausibility band, the gap-conditioning threshold, standard
  gravity, the assumed seawater density, and the shallow-column scenarios. No
  well name appears anywhere in it.
* ``p2mem.overburden_models`` - typed records and the closed vocabularies for
  gap classes, gap dispositions, overburden statuses, limiting reasons, seabed
  bases and scenario names, with constructor-level invariants that refuse an
  incoherent record (a bridged shallow gap, a total reported while a component
  is unresolved, a gap count that disagrees with its sample count, an absolute
  status carrying a limiting reason).
* ``p2mem.density_qc`` - RHOB availability, unit and conversion confirmation,
  TWELVE explicit per-sample masks, factual QC statistics, gap classification
  into the five structurally distinct cases, and gap conditioning into a
  SEPARATELY NAMED array.
* ``p2mem.overburden`` - trapezoidal integration of ``rho*g*dz`` in TRUE
  VERTICAL DEPTH, run-time verification of the project's depth-sign
  convention against each frame's own arrays, evidence-derived method
  eligibility, transparent shallow-column scenarios, and the mandatory
  gap-threshold sensitivity.
* ``p2mem.io.overburden_policy`` / ``overburden_registry`` /
  ``overburden_inventory`` / ``overburden_workflow`` - the Increment 6.1.7
  output-policy architecture GENERALIZED so its registries arrive as an
  explicit policy bundle, plus Increment 7's own nine closed artifact schemas
  and its complete field classification. The locked Increment 6 module and
  export path are untouched; a test harness runs the generalized engine over
  the locked registries and the real packaged Increment 6 records and requires
  the same decision on every occurrence.

Scientific boundaries this increment does not cross
---------------------------------------------------
An absolute vertical overburden stress requires density coverage from the
relevant datum or seabed to the evaluation depth. Where that column is not
measured, Increment 7 reports a partial measured increment or a transparent
low/base/high sensitivity spread, and NEVER a single absolute curve presented
as measured truth. No density value is clipped, rescaled, smoothed, despiked,
replaced or extrapolated; invalid samples are masked, short internal gaps may
be linearly bridged in true vertical depth into a separate array whose
contribution is reported separately, and nothing else is filled. The water
column, the unresolved shallow column and standard gravity are ASSUMPTIONS,
and every reported stress is partitioned so that measured, conditioned and
assumed contributions stay separately visible.

Increment 7 assigns NO named lithology. The density screening band, the gap
policy, the shallow-column bracket and the eligibility ladder are expressed in
physical and coverage terms, and none of them is justified by an assumed rock
type.

Increment 8.0.1 (this release, v0.8.1) is the final corrective patch to
Increment 8's PORE-PRESSURE DATA-GAP,
HYDROSTATIC-REFERENCE, and EFFECTIVE-STRESS SCREENING framework. It consumes
only packaged, checksum-locked Increment 6/7 outputs: no private raw file is
copied into the release. The hydrostatic reference is rho*g*TVDSS from a
mean-sea-level gauge datum under configured 1020/1025/1030 kg/m3 fluid-density
scenarios. Effective vertical stress is reported only as the transparent
scenario sigma'_v = Sv - alpha*Pp, pairing Increment 7 low/base/high vertical-
stress scenarios with configured Biot-alpha values 0.8 and 1.0. Only Boreas 1
and Poseidon 2 have an Increment 7 shallow-column stress scenario, so the
effective-stress calculation is withheld for Poseidon North 1 and Proteus
1ST2. No result is extrapolated or clipped.

The approved packaged input inventory contains zero RFT, MDT, DST, FIT, LOT,
XLOT or DFIT files; this does not prove that unprovided field data do not
exist. Candidate sonic intervals are
therefore retained only as readiness/sensitivity evidence. No normal-
compaction trend is selected or fitted; no Eaton, Bowers, equivalent-depth or
drilling-exponent transform is run; and no overpressure is inferred. A
hydrostatic reference is not a measured formation pressure, and the
effective-stress values are uncalibrated screening scenarios rather than
field-ready estimates.

Subsequent increments (elastic-property calculation, strength, horizontal
stress, and wellbore-stability screening) are added one validated phase at a
time and are intentionally absent from this version. Increment 9 has NOT been
started: no dynamic/static elastic-property model, rock-strength model,
horizontal-stress calculation, stress calibration, mud-window calculation,
or wellbore-stability analysis exists in this package.
"""

__version__ = "0.8.1"

# Fixed project-wide assurance tier. Referenced by later modules (reporting,
# plotting) so that every generated output can stamp its own classification
# without each module re-declaring the string. This value must not be
# changed without a documented calibration event (e.g. a verified RFT/MDT,
# LOT/XLOT, or core-calibrated log tie) recorded in the method-and-citation
# register.
ASSURANCE_TIER = "Tier C - Screening-Level / Uncalibrated Educational"

__all__ = ["__version__", "ASSURANCE_TIER"]


In [ ]:
%%writefile README.md
# Poseidon 2 — 1D Mechanical Earth Model

**Author:** Mikael Elgo

**Project classification:** Tier C — Screening-Level / Uncalibrated Educational 1D Mechanical Earth Model (MEM)

> **This project is screening-level, uncalibrated, and educational in nature. It is NOT validated against independent field measurements (no confirmed RFT/MDT pressure points, LOT/XLOT tests, or core-calibrated log ties are currently incorporated), and it must NOT be used for operational drilling, well-design, or any real-world decision-making. It exists to demonstrate a technically defensible, transparent, modular geomechanics workflow — not to produce field-ready predictions.**

---

## Purpose and technical scope

This repository implements a modular, reproducible 1D Mechanical Earth Model workflow for the Poseidon 2 well, built from well logs, deviation surveys, checkshot data, formation tops, and Vp/Vs data supplied for the project. The intended end-to-end scope (delivered incrementally, one validated phase at a time) covers:

- data quality control and depth alignment across LAS logs, deviation surveys, and checkshot data
- pore-pressure prediction (Eaton-family methods, contingent on a defensible normal compaction trend)
- elastic properties (dynamic Vp/Vs-derived Poisson's ratio, and density-dependent moduli where density coverage permits)
- rock-strength estimation
- vertical-stress (overburden) modelling
- horizontal-stress and wellbore-stability screening (Kirsch elastic wall-stress equations with Mohr–Coulomb/Mogi–Coulomb failure criteria)
- uncertainty treatment via deterministic low/base/high scenarios and one-at-a-time sensitivity (tornado) analysis, rather than unsupported probabilistic distributions

Every empirical or correlation-based relationship used anywhere in this project (Eaton, Bowers, Gardner, Castagna, etc.) is required to have a recorded source, stated units, applicability range, and calibration status in the project's method-and-citation register *before* it is implemented in code. Nothing is fabricated or assumed silently: missing measurements, missing calibration points, and unavailable data are always reported as unavailable rather than filled in.

This is a personal portfolio project intended to demonstrate scientific rigor, reproducibility, and honest handling of data limitations — not a commercial or operational deliverable.

## Current implementation status

**Increment 8.0.1 (this release, v0.8.1): final corrective release of the Pore-Pressure Data-Gap, Hydrostatic-Reference, and Effective-Stress Screening Framework.** It binds all seven upstream artifacts to their exact approved SHA-256 values, closes the configured scenario values, validates complete shallow-stress and sonic/checkshot identities, validates exact output coverage and cross-artifact identities, and publishes CSV/JSON plus figures as one failure-atomic bundle. It also replaces the unnecessary `tests` package workaround with a local fixture import, declares Matplotlib as the runtime dependency required by the default plotting path, and makes Colab bootstrap use a verified versioned tree or a fresh safe extraction without overwriting an incomplete folder. The approved packaged input set contains zero RFT/MDT/DST/FIT/LOT/XLOT/DFIT files; this is not a claim that unprovided field data do not exist. The hydrostatic values use one constant-density gradient from mean sea level—not a separately modelled seawater-plus-formation-fluid column—and remain configured references rather than measured pressure. No NCT is fitted, no overpressure transform is run, and no overpressure is inferred. See `INCREMENT_08_0_1_MANIFEST.md`. **Increment 9 has not been started.**

**Increment 8 (previous release, v0.8.0): Pore-Pressure Data-Gap, Hydrostatic-Reference, and Effective-Stress Screening Framework.** Increment 8 consumes only the packaged Increment 6/7 outputs and therefore runs without redistributing private raw inputs. It publishes an explicit zero-count approved-input inventory; low/base/high hydrostatic gauge-pressure references from MSL using configured fluid densities of 1020/1025/1030 kg/m3; and `sigma'_v = Sv - alpha*Pp` sensitivity for configured Biot-alpha values 0.8 and 1.0. Boreas 1 and Poseidon 2 receive effective-stress scenarios because Increment 7 has shallow-column vertical-stress scenarios for those wells; Poseidon North 1 and Proteus 1ST2 remain hydrostatic-reference-only because no corresponding absolute vertical-stress scenario exists. Candidate sonic-NCT interval thickness varies by factors of 7.93, 3.51, and 2.46 across the nine strict-no-gap configurations for Poseidon 2, Poseidon North 1, and Proteus 1ST2 respectively. With no pressure calibration or independently established normal-compaction interval, **no NCT is selected or fitted, no overpressure transform is run, and no overpressure is inferred**. See `INCREMENT_08_MANIFEST.md`; that record is retained as historical and is superseded where `INCREMENT_08_0_1_MANIFEST.md` says so.

**Increment 7 / 7.0.1 / 7.0.2 / 7.0.3 / 7.0.4 (LOCKED, v0.7.4): Density QC, Density-Coverage Qualification, and the Vertical Overburden-Stress Framework.** Builds on the LOCKED Increment 6.1.7 baseline by adding bulk-density availability and coverage measurement, twelve explicit per-sample validity masks, gap classification and conditioning, trapezoidal integration of `rho*g*dz` in TRUE VERTICAL DEPTH, an evidence-derived eligibility ladder, and transparent shallow-column sensitivity scenarios. **No approved well supports an absolute vertical overburden-stress curve.** Boreas 1 and Poseidon 2 have quantified unmeasured seabed-to-log columns of 3,486.54 m and 3,567.14 m TVD respectively; Poseidon North 1 and Proteus 1ST2 have no approved formation tops, so their seabed, water-column and shallow-gap thicknesses are not determinable. Increment 7.0.1 corrects the fail-closed profile rule and hardens YAML-loaded configuration. Increment 7.0.2 corrects the notebook gate and closes direct-constructor configuration bypasses. Increment 7.0.3 closes the P05 configuration-label contract. Increment 7.0.4 closes five remaining end-to-end contracts: the high endpoint uses eligible-population P05, public threshold overrides and threshold-bearing records fail closed on invalid values, scenario fractions separately sum assumed + conditioned + measured to one, locked seabed ingestion rejects missing/malformed/duplicate/non-finite matching data, and JSON is serialized as explicit UTF-8 LF bytes on every platform. The four approved wells' measured values remain unchanged because their finite and eligible RHOB populations coincide, their published scenario wells have zero bridged contribution, and the locked seabed table is valid and unique. The 10 m bridging threshold remains a sensitivity-tested project heuristic—not a rigorous endpoint-derived error bound—and the shallow-column low/high values remain conditional illustrative scenarios, not physical bounds. See `INCREMENT_07_0_4_MANIFEST.md`; prior records remain locked historical records.

**Increment 6 / 6.1 / … / 6.1.7 (this release, v0.6.8): Gamma-Ray QC, Shale-Proxy Sensitivity, Well-Frame Assembly, and Method-Eligibility Framework.** Increment 6.1.7 replaces value-driven output discovery with exact schemas for all eight artifacts. Artifact inventory, ordered CSV columns, JSON keys, requiredness, strict types and finite numeric values are validated independently of prose authorization; every declared string occurrence is then authorized, including empty and numeric-looking strings. Export writes a complete candidate set to an isolated staging directory, re-reads and re-validates it, and compares every typed field and row before failure-atomic publication. Missing/unknown fields, type substitution, authorized-to-authorized mutation, row reordering, stale artifacts, unknown well keys and serializer failures now fail closed without changing the official destination. This corrects assurance/export behavior only; no scientific calculation, threshold, disposition or result changes. See `INCREMENT_06_1_7_MANIFEST.md`.

**Increment 6.1.6 (previous corrective patch, v0.6.7): Gamma-Ray QC, Shale-Proxy Sensitivity, Well-Frame Assembly, and Method-Eligibility Framework.** Increment 6.1.6 enforces authorization at the **emission boundary** — see `INCREMENT_06_1_6_MANIFEST.md`. Increment 6.1.5's model was sound but validated a *reconstructed* scope rather than the records written to disk, so an endpoint `description` carrying an unauthorized sentence was persisted verbatim without ever entering that scope. Every string column and JSON path of all eight artifacts now carries a declared policy classification, labels are authorized by `(field_kind, value)` together, and export is two-stage: authorize the exact records, write, then re-authorize the written bytes. Eleven of twelve outputs are byte-identical to 6.1.5; the manifest changes only inside `lithology_validation`, where the derivation is corrected and the coverage metrics become truthful. Increment 6.1.5 is the preceding corrective patch — see `INCREMENT_06_1_5_MANIFEST.md`. It replaces blacklist-based acceptance with **positive authorization**: every persisted label must be an approved typed value, and every persisted interpretive, method and explanatory statement must resolve to a registered `statement_id` or a controlled `template_id` with strictly typed substitutions. Unregistered free text is rejected unconditionally, so an unknown lithology — or an invented word — cannot enter a controlled field. `PROHIBITED_LITHOLOGY_TERMS` is demoted to a supplementary diagnostic linter that authorizes nothing. All twelve outputs/figures are byte-identical to 6.1.4 and no scientific value changed. Increment 6.1.4 completes the architecture Increment 6.1.3 began — see `INCREMENT_06_1_4_MANIFEST.md`. 6.1.3 closed three of four validation scopes by membership or exact equality but left explanatory scope decided by a finite token list of project references, which omitted ordinary stratigraphic nouns (`member`, `group`, `package`, `play`, `prospect`, `target`) and therefore admitted assertions built on them. That rule is inverted and closed by registry, failing closed; the three token lists are deleted. All four scopes are now closed, all twelve outputs/figures are byte-identical to 6.1.3, and no scientific value changed. Increment 6.1.3 is the preceding architectural corrective patch, applied after an independent audit of Increment 6.1.2 — see `INCREMENT_06_1_3_MANIFEST.md` and the "Increment 6.1.3 update" bullet under Scientific limitations. It stops trying to detect geological assertions in free text at all: every validation scope is now decided by set membership or exact equality, project-specific prose has **zero allowance**, and method wording is admitted only by exact membership of a closed five-entry provenance registry. It also hardens the interval-record type gate and brings three previously unvalidated exported strings into scope. No scientific threshold, tolerance, policy, disposition or result changed; eleven of twelve outputs/figures are byte-identical to Increment 6.1.2 and the twelfth differs by one scalar. Increment 6.1.2 is the preceding corrective patch applied after an independent audit of Increment 6.1.1 — see `INCREMENT_06_1_2_MANIFEST.md` and the "Increment 6.1.2 update" bullet under Scientific limitations. It redesigns the named-lithology validator so the method-phrase allowance is decided per occurrence from local grammar rather than from a sentence-wide cue list and a fixed look-behind window (closing demonstrated false negatives AND demonstrated false positives), and enforces the interval-record interruption-count invariants completely at construction. No scientific threshold, tolerance, policy, disposition or result changed, and all twelve outputs/figures are byte-identical to Increment 6.1.1. Increment 6.1.1 is the preceding corrective patch applied after an independent audit of Increment 6.1 — see `INCREMENT_06_1_1_MANIFEST.md` and the "Increment 6.1.1 update" bullet under Scientific limitations. It closes a named-lithology validator bypass, corrects a stale Vp/Vs description in the active module documentation, replaces ambiguous interruption counts with three separately named quantities, corrects the notebook's output-file count, and supersedes an incorrect Increment 6.1 delta statement. Increment 6.1 is the preceding corrective patch applied after an independent geomechanics/rock-physics and software audit — see `INCREMENT_06_1_MANIFEST.md` and the "Increment 6.1 update" bullet under Scientific limitations. It corrects Vp/Vs domain terminology and boundaries, the Poseidon North 1 depth-tied status, unsupported lithology/correlation claims and a circular named-lithology gate, and gross-versus-net interval-thickness reporting. No architecture changed and no new phase was started. Builds on the LOCKED Increment 1-5.1.2 foundation (no locked module, config, test, fixture, notebook, output, or figure is modified) by adding an auditable per-well frame assembly, factual gamma-ray-family QC, an endpoint-sensitivity framework for a dimensionless screening proxy, and three method-eligibility masks with contiguous-interval registers. See `INCREMENT_06_MANIFEST.md` for the full technical design, the independently recomputed real-data findings, and the complete verification record; see "Increment 6" below for the scientific boundaries it deliberately does not cross. **Increment 7 has not been started.**

**Increment 5 / 5.1 / 5.1.1 / 5.1.2 (previous release, v0.5.2): Formation-Top Ingestion, Source Reconciliation, and Survey-Corrected Stratigraphic Depth Framework.** Builds on the LOCKED Increment 4.1.2 checkshot/time-depth layer, the LOCKED Increment 3.1.1 deviation-survey/depth-mapping layer, and the LOCKED Increment 2.1.1 LAS-ingestion layer (all unmodified) by adding contract-driven formation-top file ingestion for the two approved wells with formation-top data (Poseidon 2, Boreas 1), explicit reconciliation between each well's two independently supplied top-file representations, and mapping of reconciled marker depths through the locked survey trajectory to produce a corrected, auditable stratigraphic marker table. See `INCREMENT_05_MANIFEST.md` for the full technical design, the independently recomputed real-data findings, and the complete verification record; `INCREMENT_05_1_MANIFEST.md` for the Increment 5.1 corrective patch (a zero-common-marker reconciliation defect, readable-file absolute-path leakage, incomplete in-memory numerical validation, and a manifest-language correction — see "Increment 5.1 update" below); and `INCREMENT_05_1_1_MANIFEST.md` for the Increment 5.1.1 corrective patch (a remaining one-empty-source zero-common-marker edge case, plus three documentation-accuracy corrections — see "Increment 5.1.1 update" below). Formation-top ingestion/reconciliation/depth-correction only — no gamma-ray normalization, shale-volume calculation, named lithology classification, petrophysical interpretation, method-eligibility masks, shallow-density modelling, overburden-stress integration, NCT fitting, pore-pressure prediction, elastic properties, rock strength, horizontal stresses, or wellbore-stability analysis is performed in this increment.

New in Increment 6:
- `p2mem/wellframe_models.py` / `p2mem/wellframe.py` — a typed, auditable per-well assembly of the LOCKED canonical LAS curve arrays alongside the LOCKED MD→TVD/TVDSS mapping, carrying per-sample validity masks, per-curve provenance (source curve name, raw mnemonic, raw and canonical unit, conversion function, source filename), QC flags, and an evidence classification. Sample count and original file order are preserved exactly; an invalid sample is expressed only through a mask, never deleted, filled, interpolated, or reordered; every curve array is exposed read-only so a downstream consumer cannot mutate a locked loader's data through a frame. **No depth is ever extrapolated:** the locked `map_las_md_to_tvd_tvdss` is deliberately all-or-nothing, so this layer computes the in-coverage mask from the locked trajectory's own MD range and calls the locked mapper on the in-coverage subset only, leaving out-of-coverage samples as NaN with `depth_valid_mask == False` (`n_extrapolated` is 0 by construction, and the honest coverage gap is reported as `n_depth_unmapped` instead).
- `config/petrophysics_eligibility.yml` — the human-authored, human-reviewable per-well gamma-ray-family disposition, endpoint-sensitivity policy, physical-plausibility bounds, contiguity tolerances, and eligibility rules. The four wells carry four DISTINCT GR-family curves (`GR_api`, `GRD_api`, `ECGR_api`, and a second, independent `GR_api`) that are never merged, renamed, rescaled, or treated as geologically equivalent — Poseidon 2's and Proteus 1ST2's curves canonicalize to the same name but remain different tools in different wells with no cross-well calibration tie, so endpoints are always estimated per well from that well's own samples (a single universal cross-well endpoint pair is rejected at config-load time).
- `p2mem/petrophysics_models.py` / `p2mem/petrophysics.py` — factual GR-family QC statistics for EVERY well including excluded ones (the numbers justifying an exclusion must themselves be published), per-well low/base/high endpoint scenarios whose MEASURED endpoint values are recorded, and the dimensionless GR index `IGR = (GR − GR_low)/(GR_high − GR_low)` with clipped and unclipped results retained side by side so the amount of clipping — i.e. how far real data fell outside the assumed bracket — stays visible. Every endpoint carries `evidence_class = "assumed_configured"` and an explicit uncalibrated status; no code path can promote one to calibrated. The only permitted shale-proxy transform is the linear identity of the clipped index, exported under the self-labelling name `VSH_GR_linear_proxy_frac`.
- `p2mem/method_eligibility.py` — three input-admissibility masks (`eligible_density_for_sv`, `eligible_dynamic_elastic`, `eligible_sonic_nct_candidate`) with per-criterion pass counts and a named limiting criterion, plus contiguous-interval registers reported on MD, TVD and TVDSS. Gap bridging requires BOTH a small sample gap AND a small physical depth span, so sample-count continuity is never confused with physical-depth continuity; bridged samples are always disclosed separately from genuinely eligible ones.
- `p2mem/io/petrophysics_inventory.py` — deterministic summary/scenario/interval/manifest builders that never export a per-sample real-data array (which would effectively reproduce the private source logs) and never emit an absolute path, sanitizing every failure message against BOTH candidate source paths.
- **Key real-data findings (independently measured, not asserted):** Boreas 1's ECGR carries an unresolved scale/acquisition anomaly — median **8.40 API** against 36.06 / 36.37 / 41.41 API for the other three wells (4.3×–4.9× lower), a range of **[−0.0001, 519.18] API** including 4 negative and 42 exactly-zero samples, and **2,059 samples above the well's own declared seabed marker**. With no independent tool header, calibration record, or environmental-correction metadata available to adjudicate the cause, Boreas 1 is formally EXCLUDED (`BOREAS_ECGR_SCALE_UNRESOLVED`) from every lithology-dependent, GR-normalized, shale-proxy and NCT-candidate calculation — **excluded, never corrected**, since any shift, gain, or normalization would fabricate a calibration that does not exist and would silently propagate into every downstream result. Zero endpoint scenarios, zero proxies and zero lithology-dependent masks are computed for it; it remains available for factual raw-GR QC display and availability reporting only. Endpoint choice alone moves the screening-proxy median by up to 0.12 (dimensionless), and endpoint × threshold choice moves the qualifying sonic-NCT-CANDIDATE thickness by a factor of **8.0** in Poseidon 2 (144 m to 1,159 m), 3.5× in Poseidon North 1 and 2.3× in Proteus 1ST2 — quantifying exactly how much a later NCT result would depend on choices no data in this project constrains.
- **No named lithology is assigned anywhere in Increment 6, and the available data do not support assigning one.** Low-GR intervals occur independently in each of the three GR-eligible wells; cross-well stratigraphic persistence is NOT established, and cannot be, because Poseidon North 1 and Proteus 1ST2 have no approved formation tops to correlate within. Gamma-ray response is not uniquely diagnostic of rock type, and no core, cuttings description, image log, spectral GR, or calibrated multi-mineral solution exists in this project to adjudicate it. Every classification this increment produces (`GR_PROXY_HIGH` / `GR_PROXY_INTERMEDIATE` / `GR_PROXY_LOW` / `GR_NOT_AVAILABLE` / `GR_EXCLUDED_UNRESOLVED_SCALE`) describes DATA AND PROXY CONFIDENCE ONLY, and is machine-checked against an enforced prohibited-rock-name vocabulary.
- Still not implemented: named lithology interpretation, environmental GR correction, Boreas ECGR rescaling, neutron-density crossplot interpretation, any nonlinear Vsh transform (Larionov, Clavier, Stieber — all DEFERRED pending a retrieved, verified primary-source method record), shallow-density reconstruction, density extrapolation, vertical-stress integration, hydrostatic-pressure modelling, NCT fitting, Eaton/Bowers pore-pressure prediction, dynamic or static elastic-property calculation, rock-strength or friction-angle correlation, Shmin/SHmax modelling, stress-polygon construction, wellbore-stability analysis, and mud-weight recommendation.


New in Increment 5:
- `p2mem/top_models.py` — typed, frozen dataclasses for every formation-top header/contract/raw-row/reconciliation/corrected-marker result object, mirroring the checkshot/deviation layers' design philosophy. Two source representations exist per well — an "HRS" file (`Top_Name`/`MDRT_m` only, no well name in the file body) and a "selected readable" file (`TOP_NAME`/`MDRT_M`/`TVDSS_M`/`NOTE`, `#`-comment headers, a dashed separator line) — and every raw (`_source_`) value from both is preserved separately; nothing is ever silently overwritten, sorted, deduplicated, or repaired.
- `p2mem/io/tops.py` — an auditable formation-top parser, per-file contract resolver, and HRS-versus-readable source reconciler for the four approved files, keyed by their exact literal source filenames. Markers are matched between the two files by exact name, whitespace-normalized name, or an explicit human-authored alias contract (`config/formation_top_contracts.yml`'s `marker_name_aliases` — currently empty; no fuzzy/similarity matching of any kind is ever performed). MDRT is cross-checked between the two sources within a documented 0.005 m tolerance; a marker on which the two sources disagree beyond that tolerance is registered (`mdrt_status = "MISMATCH"`) and excluded from depth mapping (`mdrt_authority_basis = "disagreement_unresolved"`, `mapping_status = "not_mapped_mdrt_unresolved"`) rather than resolved by silently picking one file's value. Reconciled MDRT is mapped through the LOCKED Increment 3.1.1 `petrel_source_trace` survey trajectory using the existing, unmodified `p2mem.depth_mapping.map_las_md_to_tvd_tvdss` — reused unmodified, never reimplemented — called once per marker so that one out-of-coverage marker never blocks mapping of the rest of the well, and extrapolation is never performed (a marker outside survey MD coverage is reported as `mapping_status = "rejected_outside_coverage"`, never extrapolated).
- `p2mem/io/tops_inventory.py` — deterministic, metadata-only inventory/QC-table builders for the Increment 5 outputs (file inventory, marker/source register, HRS-versus-readable reconciliation table, survey-corrected marker table, ingestion issues, formation-top availability, JSON manifest) — never raw per-marker arrays beyond a single scalar per field, and never a full environment-dependent build path, only a basename.
- `config/formation_top_contracts.yml` — the human-authored, human-reviewable per-file formation-top contract for each of the four approved files, including each file's `well_identity_evidence_status` (both representations for both wells are `inferred_unverified` — the HRS files carry no well name in their body at all, association resting on the filename alone, and the readable files' well name appears only in a project-supplied `#` comment, not independently verified content; this is a deliberately MORE conservative classification than Increment 4's checkshot files, explicitly justified in `INCREMENT_05_MANIFEST.md`).
- **Key real-data findings (disclosed):** every one of Poseidon 2's 9 "selected readable" TVDSS values equals `MDRT_M - 21.8 m` EXACTLY (21.8 m is the well's own rotary-table elevation) — a literal vertical-well-assumption depth-reference defect, since a real TVDSS should differ from MD by more than a constant datum shift once a well deviates. Survey-corrected residuals (`TVDSS_source_m - TVDSS_survey_corrected_m`) reproduce Sea Bed at ≈0 m, Plover Fm (Top Reservoir) at ≈+1.68 m, and TD at ≈+2.49 m — confirming the anticipated defect and correcting it in the derived `TVDSS_survey_corrected_m` representation while `TVDSS_source_m` itself is preserved unmodified. Boreas 1's supplied TVDSS does NOT follow the `MDRT - 21.8` pattern and its maximum absolute source-versus-survey residual across all 10 markers is 0.0416 m, below the approved 0.05 m tolerance — confirming it was generated from the well's real surveyed trajectory, and it is therefore NOT "corrected" the way Poseidon 2 is; evidence is applied per file, per well, never by analogy. Poseidon North 1 and Proteus 1ST2 have no approved formation-top file and are recorded as `formation_top_availability: NOT_AVAILABLE`, a factual data gap, never substituted with another well's tops or correlated by depth alone.
- Still not implemented: gamma-ray normalization, shale-volume calculation, named lithology classification, petrophysical interpretation, method-eligibility masks, shallow-density modelling, overburden-stress integration, NCT fitting, pore-pressure prediction, elastic-property calculation, rock-strength estimation, horizontal stresses, or wellbore-stability calculations. Those remain explicitly out of scope for this increment.

New in the Increment 5.1 corrective patch (see "Increment 5.1 update" under Scientific limitations for the full defect list): `reconcile_formation_top_sources` now correctly rejects two disjoint, non-empty marker sets as a fatal `NO_COMMON_MARKERS` ERROR (previously silently accepted); `TopIngestionFailure` now carries an explicit `failure_origin` and both `hrs_path`/`readable_path`, so a failure originating from the "selected readable" file can no longer leak that file's own absolute path into any exported CSV/JSON field; and `reconcile_formation_top_sources` now fully validates its numeric-array and `mdrt_agreement_tolerance_m` inputs before any comparison, with deliberate typed exceptions rather than an incidental `IndexError`/`TypeError`. None of the real Poseidon 2 / Boreas 1 formation-top results above changed.

New in the Increment 5.1.1 corrective patch (see "Increment 5.1.1 update" under Scientific limitations for the full defect list): `reconcile_formation_top_sources` now rejects the one remaining zero-common-markers configuration Increment 5.1 missed — exactly one source entirely empty, the other non-empty — via a single, strictly correct intersection check (`if not common_markers`) that covers every zero-common-markers case at once, while a legitimate one-sided marker alongside at least one genuinely shared marker remains the pre-existing, non-fatal `NOT_COMPARABLE` case. `INCREMENT_05_1_MANIFEST.md`'s baseline-hash, packaged-outputs, and packaged-LAS-fixture wording is also corrected (documentation-only; see "Increment 5.1.1 update"). None of the real Poseidon 2 / Boreas 1 formation-top results above changed.

**Increment 4 / 4.1 / 4.1.1 / 4.1.2 (v0.4.1.1, LOCKED as of Increment 5): Checkshot (Velocity Survey) Ingestion, Duplicate-Tie Conditioning, and Time–Depth Framework.** Builds on the LOCKED Increment 3.1.1 deviation-survey/depth-mapping layer and the LOCKED Increment 2.1.1 LAS-ingestion layer (both unmodified - see below) by adding auditable checkshot parsing with explicit per-file contracts, raw-vs-conditioned duplicate-tie handling, average/interval velocity diagnostics, checkshot-vs-locked-survey depth-reference comparison, a coverage-masked forward/inverse piecewise-linear time-depth interpolation layer, LAS MD-to-checkshot-time mapping within validated coverage only, and a Poseidon-2-only sonic-checkshot drift diagnostic. See `INCREMENT_04_MANIFEST.md` for the full technical design and the independently recomputed real-data statistics, `INCREMENT_04_1_MANIFEST.md` for the Increment 4.1 corrective patch (order-invariant axis-tie conditioning for TVDSS↔OWT/TWT inversion, replacing an order-dependent defect; numerical-validation hardening — see "Increment 4.1 update" below), `INCREMENT_04_1_1_MANIFEST.md` for the Increment 4.1.1 numerical-validation corrective patch (a hidden-reversal grouping defect, incomplete full-MD validation, an untyped zero-coverage crash, missing batch isolation for numerical failures, and a unit-helper input-safety gap — see "Increment 4.1.1 update" below), and `INCREMENT_04_1_2_MANIFEST.md` for the Increment 4.1.2 packaging-only corrective patch (Colab line-ending reproducibility and a truthful notebook completion gate; no scientific, numerical, or version change). Checkshot data QC, duplicate-tie conditioning, and time-depth interpolation only — no formation-top correction, lithology interpretation, density modelling, pore-pressure prediction, elastic properties, rock strength, stress modelling, or wellbore-stability analysis is performed in this increment.

New in Increment 4:
- `p2mem/checkshot_models.py` — typed, frozen dataclasses for every checkshot header/contract/raw-row/duplicate-tie/velocity-diagnostic/depth-comparison/time-mapping/sonic-drift result object, mirroring the deviation-survey layer's design philosophy. Raw (`_source_`) values are always kept explicitly separate from conditioned (`_conditioned_`) values — never overwritten, never mixed.
- `p2mem/io/checkshot.py` — an auditable checkshot (velocity-survey) parser and per-file contract resolver for the three approved checkshot files (`Poseidon2-Checkshot.txt`, `Boreas1-Checkshot.txt`, `Proteus1-Checkshot.txt`), keyed by their exact literal source filenames. File-identity checks (filename, SHA-256, header/survey-statement text, column order, row width, numeric structure) are enforced as blocking `ERROR`s before any time-depth computation is attempted. Raises disclosure `WARNING`s including `DEPTH_BASIS_NOT_EXPLICITLY_DECLARED` (the source file's first column is labelled only `Depth`, never assumed to be MD without evidence) and, for Proteus 1ST2, `WELL_IDENTITY_INFERRED_UNVERIFIED` (the file carries no embedded well identifier tying it to Proteus 1ST2).
- `p2mem/time_depth.py` — the numerical time-depth layer: duplicate/repeated-tie detection and deterministic, disclosed median-based conditioning (raw rows always preserved and separately registered; ties never silently averaged, never force-monotonized with artificial epsilon increments); average velocity (`Vavg = TVDSS / OWT`) and interval velocity (`Vint = ΔTVDSS / ΔOWT`, NaN — never infinite or negative — for any invalid, zero, or non-increasing interval); checkshot-vs-locked-survey depth-reference comparison (residual = survey-interpolated TVDSS − checkshot-supplied TVDSS); forward/inverse piecewise-linear time-depth interpolation with explicit coverage masking (points outside validated checkshot coverage are reported as not-mapped, never extrapolated) via an order-invariant, median-based axis-tie-conditioned lookup table for TVDSS↔OWT/TWT inversion (Increment 4.1 — every tied value on the axis being inverted is grouped and registered, never resolved by an order-dependent "first wins" tie-break); LAS MD-to-checkshot-time mapping for Poseidon 2 within validated coverage only; and a Poseidon-2-only sonic-checkshot drift diagnostic (a local, version-independent trapezoidal integration of sonic slowness over the longest valid continuous MD interval, compared against the checkshot-interpolated OWT increment over the same interval — this diagnostic never modifies `VP_m_s`, `DTCO`, checkshot OWT, or the time-depth curve itself).
- `p2mem/io/checkshot_inventory.py` — deterministic, metadata-only inventory/QC-table builders for the Increment 4 / 4.1 outputs (file inventory, ingestion issues, Depth-axis duplicate-tie register, the Increment 4.1 TVDSS/OWT-axis tie register, depth-tie QC, velocity summary, sonic-checkshot drift summary, time-depth mapping summary, JSON manifest) — never raw per-sample checkshot or LAS arrays, and never a full environment-dependent build path, only a basename.
- `config/checkshot_contracts.yml` — the human-authored, human-reviewable per-file checkshot contract for each of the three approved files, including each well's `model_use_status` (`primary_model` for Poseidon 2 only; `qc_only` for Boreas 1 and Proteus 1ST2) and `identity_evidence_status` (`verified` for Poseidon 2 and Boreas 1; `inferred_unverified` for Proteus 1ST2).
- **Key real-data findings (disclosed):** Poseidon 2's raw checkshot file contains 5 repeated Depth ties, 6 non-increasing TVDSS steps, and 4 non-increasing OWT steps, all independently detected and conditioned (never silently smoothed); its checkshot-vs-survey TVDSS comparison shows a maximum absolute residual of ≈0.089 m with no systematic offset pattern. Boreas 1 shows 3 repeated Depth ties and 4 non-increasing TVDSS steps (OWT strictly increasing throughout) and a checkshot-vs-survey comparison with a near-constant offset of ≈−0.69 m, reported as an observed datum-like offset pattern — not a proven datum error. Proteus 1ST2's file is strictly increasing in all three columns with no repeated ties, and shows a near-constant offset of ≈+0.30 m against the locked survey trajectory; its association with Proteus 1ST2 remains `inferred_unverified` throughout every output. Poseidon 2's sonic-checkshot drift over its longest valid continuous sonic interval (≈MD 2449–4064 m) is ≈+16.7 ms (≈+4.4% of the checkshot-interpolated one-way transit time (OWT) increment over that interval — NOT two-way time; the diagnostic compares the integrated sonic transit time directly against the checkshot's OWT increment over the identical interval), reported as a diagnostic only. Poseidon North 1 has no approved checkshot file; this is recorded as `checkshot_availability: NOT_AVAILABLE`, a factual data gap, not an ingestion failure. Depth-tie conditioning alone does NOT guarantee TVDSS or OWT is itself strictly increasing (required for TVDSS↔OWT/TWT inversion) — see the Increment 4.1 update below for the corrected, order-invariant handling of this. See `INCREMENT_04_MANIFEST.md` for the full statistics, tables, and figures.
- Still not implemented: formation-top correction, lithology interpretation, density modelling, sonic/checkshot drift *correction*, synthetic extension of the time-depth relationship beyond measured checkshot coverage, pore-pressure prediction, elastic-property calculation, rock-strength estimation, overburden/horizontal stresses, or wellbore-stability calculations. Those remain explicitly out of scope for this increment.

**Increment 3 / 3.1 / 3.1.1: Deviation-Survey Ingestion, Minimum-Curvature Validation, and MD–TVD–TVDSS Depth Framework.** Builds on the LOCKED Increment 2.1.1 LAS-ingestion layer (unmodified - see below) by adding Petrel deviation-survey parsing, an explicit per-file survey contract, a standard minimum-curvature trajectory engine, an explicit depth-reference (MD/TVD/TVDSS) framework, and MD-to-TVD/TVDSS mapping of the existing LAS `MD_m` arrays. See `INCREMENT_03_MANIFEST.md` for the Increment 3 technical design and real four-well integration results, `INCREMENT_03_1_MANIFEST.md` for the Increment 3.1 corrective patch (four audit findings: source-filenames-with-spaces, absolute-path leakage, DLS-normalization disclosure, dogleg numerical stability — see "Increment 3.1 update" below), and `INCREMENT_03_1_1_MANIFEST.md` for the Increment 3.1.1 packaging-only corrective patch (notebook/source `%%writefile` synchronization; no scientific or numerical change). The Proteus 1ST2 trajectory-discrepancy finding is disclosed, not resolved, in any of these releases. This layer, and the Increment 2.1.1 LAS-ingestion layer beneath it, are LOCKED as of Increment 4 and reused unmodified.

Locked from Increment 2.1.1 (unmodified in Increment 3 unless a blocking defect is documented - none was found):
- `p2mem/units.py` — an explicit, NumPy-based unit-conversion layer (no external unit-registry dependency such as Pint) implementing 21 public conversion functions between oilfield and SI-internal units. Unchanged since Increment 1.1. See the module docstring and `tests/test_units.py`.
- `p2mem/models.py`, `p2mem/io/las.py`, `p2mem/io/inventory.py`, `config/las_curve_contracts.yml` — the auditable LAS 2.0 parser, per-file curve-contract resolver, and inventory builders for the four approved wells, corrected and independently re-verified through Increment 2.1.1 (158 tests passing, 4/4 real wells loading with zero ingestion errors). See `INCREMENT_02_v2.1.1_MANIFEST.md`.

New in Increment 3:
- `p2mem/deviation_models.py` — typed, frozen dataclasses for every deviation-survey/trajectory/depth-mapping result object (header info, per-file contract, raw station data, minimum-curvature result, trajectory-validation residuals, depth-basis selection, typed batch failures, LAS depth-mapping result), mirroring the LAS layer's design philosophy. Every source (`_source_`) array is kept explicitly separate from every independently computed (`_mc_`) array — never overwritten, never mixed.
- `p2mem/io/deviation.py` — an auditable Petrel deviation-survey (well-trace) parser and per-file contract resolver for the same four wells, keyed by their exact, literal source filenames (which contain spaces, e.g. `"Poseidon 2_dev.txt"` — corrected in Increment 3.1; see below). Extracts and preserves the full header block (well/survey identity, wellhead X/Y, datum and its MSL reference, coordinate-reference-system text, declared angle/depth/coordinate conventions) and the exact 11-column station table, with file-identity checks (filename, SHA-256, well/survey identifier, wellhead/datum values, coordinate system, column order, station count, MD coverage) all enforced as blocking `ERROR`s before any trajectory computation is attempted. Also raises two disclosure `WARNING`s for every successfully loaded file: the pre-existing `MD_UNIT_NOT_EXPLICITLY_DECLARED`, and the Increment 3.1 `DLS_NORMALIZATION_INFERRED_AS_DEG_PER_30M` (the supplied `DLS` column's degrees-per-30-m normalization is inferred, not header-declared, and is independently verified per file against a recomputation from that file's own inclination/azimuth).
- `p2mem/trajectory.py` — the standard minimum-curvature method (a numerically stable `arctan2(||cross||, dot)` dogleg-angle formulation — Increment 3.1 correction, see below — a ratio factor with an explicit Taylor-series limit as the dogleg approaches zero, TVD/northing/easting displacement, dogleg severity in degrees per 30 m), implemented explicitly and transparently with no third-party survey-computation library.
- `p2mem/depth_mapping.py` — MD-to-TVD/TVDSS interpolation of the locked LAS `MD_m` array against the explicitly selected depth-trajectory basis, using a documented, deterministic piecewise-linear station interpolation (never a per-sample minimum-curvature recomputation) that never extrapolates silently.
- `p2mem/io/deviation_inventory.py` — deterministic, metadata-only inventory-table builders for the Increment 3 outputs (file inventory, trajectory-validation summary, depth-reference register, LAS depth-mapping summary, ingestion issues, JSON manifest) — never raw per-sample station or LAS arrays, and (Increment 3.1 correction) never a full environment-dependent build path, only a basename.
- `config/deviation_survey_contracts.yml` — the human-authored, human-reviewable per-file deviation-survey contract for each of the four wells, keyed by the exact literal source filename (`"Poseidon 2_dev.txt"`, `"Boreas 1_dev.txt"`, `"Poseidon North 1_dev.txt"`, `"Proteus 1ST2_dev.txt"` — corrected in Increment 3.1), including an explicit, uniformly applied `depth_basis_policy` (`petrel_source_trace`, the conservative default given the Proteus 1ST2 finding below) and residual-comparison tolerances declared once and applied identically to every well (never tuned per well to force a pass/fail outcome).
- **Key real-data finding (disclosed, not resolved):** independent minimum-curvature reconstruction of Poseidon 2, Boreas 1, and Poseidon North 1 agrees with their Petrel-supplied TVD to approximately millimetre scale. Proteus 1ST2 shows a materially larger discrepancy (~0.19 m TVD, ~1.6 m easting at maximum) concentrated in its deeper section (below ~MD 4200 m), even though its own supplied dogleg-severity column is internally consistent with an independent recomputation from its own inclination/azimuth at every station. This is reported as a visible trajectory-validation `WARNING`, not corrected, hidden, or used to justify loosening every well's tolerance — see `INCREMENT_03_MANIFEST.md` Section 6 for the full investigation and the evidence pattern observed. Unaffected by the Increment 3.1 patch.
- Still not implemented (as of Increment 3/3.1/3.1.1): checkshot ingestion, time-depth conversion, formation-top correction, petrophysical interpretation, gamma-ray normalization, shale-volume calculation, lithology classification, normal-compaction-trend fitting, pore-pressure prediction, elastic-property calculation, rock-strength estimation, overburden/horizontal stresses, or wellbore-stability calculations. Those were explicitly out of scope for this increment. Checkshot ingestion and the time-depth framework were subsequently added in Increment 4 (see above); the remainder are added one gated increment at a time in later releases.

## Installation

Requires Python 3.9 or later.

```bash
# from the project root (the directory containing pyproject.toml)
pip install -e .
```

This installs the `p2mem` package in editable mode along with its runtime dependencies: NumPy (`numpy>=1.24`) and, as of Increment 2, PyYAML (`pyyaml>=6.0`) — used for parsing the human-authored curve contracts in `config/las_curve_contracts.yml`, `config/deviation_survey_contracts.yml` (Increment 3), `config/checkshot_contracts.yml` (Increment 4), and (new in Increment 5) `config/formation_top_contracts.yml`. No new runtime dependency was added in Increment 3, 4, or 5: the minimum-curvature engine, depth-mapping interpolation, duplicate-tie conditioning, velocity diagnostics, time-depth interpolation, and formation-top reconciliation/mapping all use only NumPy (including a local, version-independent trapezoidal-integration helper in `p2mem/time_depth.py`, added because `numpy.trapz`/`numpy.trapezoid` are not consistently available across supported NumPy versions). Matplotlib and pandas are used only for notebook display and QC-figure generation (`run_integration_03.py`/`run_integration_04.py`/`run_integration_05.py`, the Increment 3/4/5 notebooks) — never imported by the installable `p2mem` package itself. To also install the test dependency:

```bash
pip install -e ".[dev]"
```

## Running the tests

```bash
pytest -v
```

The suite in `tests/test_units.py` validates `p2mem/units.py` (unchanged since Increment 1.1) against analytical reference values, round-trip consistency, scalar/array inputs, NaN preservation, and rejection of invalid/nonphysical/ambiguous inputs. The suite in `tests/test_las.py` validates `p2mem/io/las.py` (locked since Increment 2.1.1) against small synthetic LAS fixtures. The suites in `tests/test_trajectory.py`, `tests/test_deviation.py`, and `tests/test_depth_mapping.py` (new in Increment 3; extended in Increment 3.1 with dogleg numerical-stability, DLS-normalization-disclosure, and real-filename-with-spaces regression/negative tests) validate the minimum-curvature engine, the Petrel deviation-survey parser/contract resolver, and the MD-to-TVD/TVDSS mapping respectively, against small synthetic fixtures under `tests/fixtures/` and in-memory synthetic data (this layer is locked, unmodified, as of Increment 4). `tests/test_deviation_inventory.py` (new in Increment 3.1) validates that no exported inventory/issues/manifest row, for a successful or a failed well, ever embeds a full environment-dependent build path. The suites in `tests/test_checkshot.py`, `tests/test_time_depth.py`, and `tests/test_checkshot_inventory.py` (new in Increment 4) validate the checkshot parser/contract resolver, the duplicate-tie conditioning/velocity-diagnostic/time-depth-interpolation/sonic-drift numerical layer, and the deterministic inventory/QC-table builders respectively, against small synthetic fixtures under `tests/fixtures/` (including a CRLF fixture used to verify line-ending detection) and in-memory synthetic/analytic data — including an analytic constant-velocity case used to independently verify the trapezoidal-integration helper. The suites in `tests/test_tops.py` and `tests/test_tops_inventory.py` (new in Increment 5) validate the formation-top parser/contract resolver/HRS-versus-readable reconciliation logic and the deterministic inventory/QC-table builders respectively, against small synthetic fixtures under `tests/fixtures/` and in-memory synthetic data, including synthetic analogs of both real regression findings (a vertical-assumption depth-reference defect with a growing residual, and a survey-consistent well with a near-zero residual). None of these suites require the private/raw project LAS, deviation, checkshot, or formation-top files, so the full suite runs the same way for anyone who clones this repository. Run the command above and read the reported pass/fail count directly — this document does not assert a fixed expected count, since that must always be read from the actual `pytest` output for the code currently on disk. Real integration validation (which DOES require the raw LAS/deviation/checkshot/formation-top files, not included in this repository) is a separate notebook run — see `02_LAS_Ingestion_and_Curve_Contracts.ipynb`, `03_Deviation_Survey_and_Depth_Framework.ipynb`, `04_Checkshot_QC_and_Time_Depth_Framework.ipynb`, and `05_Formation_Tops_and_Stratigraphic_Depth_Framework.ipynb`.

## Directory structure

```
Poseidon_1D_MEM/
├── README.md
├── pyproject.toml
├── p2mem/
│   ├── __init__.py
│   ├── models.py
│   ├── units.py
│   ├── deviation_models.py
│   ├── trajectory.py
│   ├── depth_mapping.py
│   ├── checkshot_models.py
│   ├── time_depth.py
│   ├── top_models.py
│   ├── wellframe_models.py
│   ├── wellframe.py
│   ├── petrophysics_models.py
│   ├── petrophysics.py
│   ├── method_eligibility.py
│   ├── overburden_models.py        (Increment 7)
│   ├── density_qc.py               (Increment 7)
│   ├── overburden.py               (Increment 7)
│   ├── pore_pressure_models.py      (Increment 8)
│   ├── pore_pressure.py             (Increment 8)
│   └── io/
│       ├── __init__.py
│       ├── las.py
│       ├── inventory.py
│       ├── deviation.py
│       ├── deviation_inventory.py
│       ├── checkshot.py
│       ├── checkshot_inventory.py
│       ├── tops.py
│       ├── tops_inventory.py
│       ├── output_policy.py
│       ├── petrophysics_inventory.py
│       ├── overburden_policy.py    (Increment 7)
│       ├── overburden_registry.py  (Increment 7)
│       ├── overburden_inventory.py (Increment 7)
│       ├── overburden_workflow.py  (Increment 7)
│       ├── pore_pressure_inventory.py (Increment 8)
│       └── pore_pressure_workflow.py  (Increment 8)
├── tests/
│   ├── test_units.py
│   ├── test_las.py
│   ├── test_trajectory.py
│   ├── test_deviation.py
│   ├── test_depth_mapping.py
│   ├── test_deviation_inventory.py
│   ├── test_checkshot.py
│   ├── test_time_depth.py
│   ├── test_checkshot_inventory.py
│   ├── test_tops.py
│   ├── test_tops_inventory.py
│   ├── test_wellframe.py
│   ├── test_petrophysics.py
│   ├── test_method_eligibility.py
│   ├── synthetic_inc6.py
│   ├── synthetic_inc7.py           (Increment 7; fictional in-memory frames only)
│   ├── helpers_inc7.py             (Increment 7)
│   ├── test_density_qc.py          (Increment 7)
│   ├── test_overburden.py          (Increment 7)
│   ├── test_overburden_policy.py   (Increment 7)
│   ├── test_overburden_inventory.py (Increment 7)
│   ├── synthetic_inc8.py           (Increment 8; fictional only)
│   ├── test_pore_pressure.py       (Increment 8)
│   ├── test_pore_pressure_inventory.py (Increment 8)
│   ├── test_pore_pressure_workflow.py (Increment 8)
│   └── fixtures/         (small synthetic LAS + deviation-survey + checkshot + formation-top files; no project raw data)
├── config/
│   ├── las_curve_contracts.yml
│   ├── deviation_survey_contracts.yml
│   ├── checkshot_contracts.yml
│   ├── formation_top_contracts.yml
│   ├── petrophysics_eligibility.yml
│   ├── overburden_stress.yml       (Increment 7)
│   └── pore_pressure.yml           (Increment 8)
├── data/
│   └── raw/
│       ├── logs/         (the four raw LAS files - NOT included in this repository; immutable inputs)
│       ├── deviation/    (the four raw deviation-survey files, exact filenames contain spaces, e.g. "Poseidon 2_dev.txt" - NOT included in this repository; immutable inputs)
│       ├── checkshot/    (the three raw checkshot files, exact filenames e.g. "Poseidon2-Checkshot.txt" - NOT included in this repository; immutable inputs, never rewritten/renamed/"cleaned")
│       └── tops/         (the four raw formation-top files, e.g. "Poseidon_2_HRS_tops_no_wellname_MDRT.txt" - NOT included in this repository; immutable inputs)
├── notebooks/  (reserved for later increments)
└── outputs/
    ├── 02_las_inventory/            (Increment 2.1.1 real four-well run: CSV/JSON metadata only, no raw log samples)
    ├── 03_deviation_depth/          (Increment 3 real four-well run: CSV/JSON metadata + QC figures, no raw station/log samples)
    ├── 04_checkshot_time_depth/     (Increment 4 real three-file checkshot run: CSV/JSON metadata + QC figures, no raw per-sample checkshot/LAS arrays)
    ├── 05_formation_tops/           (Increment 5 real four-file formation-top run: CSV/JSON metadata + QC figures, no raw per-marker arrays beyond scalar fields)
    ├── 06_petrophysics_eligibility/ (Increment 6 real four-well run: CSV/JSON metadata + QC figures, no raw per-sample arrays)
    ├── 07_density_overburden/       (Increment 7 real four-well run: 9 closed-schema CSV/JSON artifacts + 4 QC figures; the stress profile is a DECIMATED selection of existing samples, never a raw log dump)
    └── 08_pore_pressure_effective_stress/ (Increment 8: 7 CSV/JSON artifacts + 4 figures, derived only from packaged locked outputs)
```

`notebooks/` is created empty by the project-setup notebook cell and is not yet populated in-repo (the increment notebooks themselves are delivered as top-level files, e.g. `02_LAS_Ingestion_and_Curve_Contracts.ipynb`, `03_Deviation_Survey_and_Depth_Framework.ipynb`, `04_Checkshot_QC_and_Time_Depth_Framework.ipynb`, `05_Formation_Tops_and_Stratigraphic_Depth_Framework.ipynb`, `06_GR_QC_Shale_Proxy_and_Method_Eligibility.ipynb`, `07_Density_QC_and_Overburden_Stress_Framework.ipynb`, and are meant to be run from Google Drive per their own directory-setup cells; the Increment 7 notebook additionally runs unchanged from any working directory that is itself the project root).

## Scientific limitations

These limitations are specific to the Poseidon 2 dataset and this project's current increment, and are carried forward here so they are visible outside the conversation in which they were identified:

- **RHOB (bulk density) coverage in Poseidon 2 ends at approximately 5,296.85 m MD.** Sonic and other curves continue deeper, so Vp/Vs and dynamic Poisson's ratio remain computable below that depth, but density-dependent properties (Young's modulus, shear modulus, bulk modulus, acoustic impedance, shear impedance) are unavailable below it unless density is explicitly estimated and flagged as such — never silently substituted.
- **No reliable shale-based normal compaction trend (NCT) exists from Poseidon 2 alone.** A provisional, transferred candidate NCT identified in offset well Poseidon North 1 is a *candidate*, not a validated trend, and must not be presented as calibrated.
- **Independently measured Vp/Vs quality flags:** approximately 3.38% of Poseidon 2 Vp/Vs values fall below 1.5, and approximately 0.53% fall below the physical validity cutoff of √2 (≈1.4142) required for a non-negative dynamic Poisson's ratio.
- **No independent calibration data (RFT/MDT pressure points, LOT/XLOT tests, or core data) has been supplied or incorporated.** Any pore-pressure or stress output in later increments must be presented as a bounded or theoretical estimate, not a validated field prediction.
- **Empirical/correlation equations are not implemented until their governing equation, units, applicability range, and calibration status are recorded in the project's method-and-citation register.** Several candidate methods remain in "pending" status and are intentionally absent from the codebase for that reason, not because they were overlooked.
- Additional open items (offset-well GR/ECGR scale adjudication, missing formation tops for one offset well) are tracked in the project's design-review documentation and gate specific later phases (lithology and pore-pressure), not this increment.
- **Increment 2 update:** LAS ingestion independently reconfirms (does not newly discover, and does not act on) two previously-flagged anomalies from the Rev 1 design review: Boreas 1's ECGR curve (canonical name `ECGR_api`) ranges from approximately −0.0001 to 519.18 API (vs. roughly 5–205 API for the other three wells' GR-family curves) with 96.98% valid coverage; and Proteus 1ST2's LAS log file places its neutron-porosity curve (canonical name `NPHI_pct`) at column position 5 rather than the last position (8) used by the other three wells. Both are reported as ingestion facts (see `outputs/02_las_inventory/`); neither is rescaled, reinterpreted, or otherwise acted on by this increment.
- **Increment 2.1 / 2.1.1 update:** corrective patches addressing independent audits' naming, reporting, contract-validation, and notebook/source-synchronization findings — see `INCREMENT_02_v2.1_MANIFEST.md` and `INCREMENT_02_v2.1.1_MANIFEST.md`. No new scientific finding was made in either patch; the two anomalies above are unaffected and remain open items for a later, explicitly-scoped increment.
- **Increment 3 update:** deviation-survey ingestion independently reconfirms the Petrel-supplied trajectory for Poseidon 2, Boreas 1, and Poseidon North 1 to approximately millimetre scale via minimum curvature, and additionally DISCOVERS (not merely reconfirms) a real trajectory-reconstruction discrepancy in Proteus 1ST2's deeper section (~0.19 m TVD, ~1.6 m easting at maximum, concentrated below ~MD 4200 m) — see "Current implementation status" above and `INCREMENT_03_MANIFEST.md` Section 6 for the full investigation. This is disclosed as an open item, not corrected or hidden; downstream MD-to-TVD/TVDSS mapping for Proteus 1ST2 conservatively uses the Petrel-supplied source trajectory (not the disagreeing minimum-curvature trajectory) as a result.
- **Increment 3.1 update:** a corrective patch addressing an independent audit's findings on source-filename handling, output environment-independence, an undisclosed normalization inference, and dogleg-angle numerical conditioning — see `INCREMENT_03_1_MANIFEST.md` for the full audit and re-verification record. No new scientific finding was made in this patch; the Proteus 1ST2 discrepancy above is unaffected, remains disclosed exactly as before, and was neither corrected nor concealed. The only numerical changes are at the floating-point noise floor of the diagnostic `dogleg_deg`/`dls_deg_per_30m`/TVD-and-offset-residual fields (at most ~9×10⁻¹³ m for TVD, ~7×10⁻¹⁵ m for easting/northing, across all four real wells), with zero change to any well's PASS/WARNING status.
- **Increment 3.1.1 update:** a packaging-only corrective patch (three notebook `%%writefile` cells that had drifted from their packaged source files, mirroring the earlier Increment 2.1.1 finding). No scientific, numerical, or real-data change of any kind — see `INCREMENT_03_1_1_MANIFEST.md`.
- **Increment 4 update:** checkshot ingestion and the time-depth framework independently reproduce every raw-data anomaly the project design anticipated (Poseidon 2: 5 repeated Depth ties, 6 non-increasing TVDSS steps, 4 non-increasing OWT steps, and a ≈257 m gap in Depth coverage between 1313.1 m and 1570.1 m; Boreas 1: 3 repeated Depth ties and 4 non-increasing TVDSS steps with OWT strictly increasing; Proteus 1ST2: strictly increasing in all three raw columns) and DISCLOSES (not resolves) two further items: (1) Boreas 1's and Proteus 1ST2's checkshot-vs-locked-survey TVDSS comparisons each show a near-constant offset (≈−0.69 m and ≈+0.30 m respectively) — reported as an observed datum-like offset pattern, not a proven datum error, with both source references preserved unmodified; (2) `Proteus1-Checkshot.txt` carries no embedded well identifier, so its association with Proteus 1ST2 is recorded as `identity_status: inferred_unverified` and used for QC only, never as a substitute time-depth model for any other well. Poseidon 2's sonic-checkshot drift over its longest valid continuous sonic interval is a diagnostic finding only (≈+16.7 ms, ≈+4.4%) and does not trigger any correction of `VP_m_s`, `DTCO`, checkshot OWT, or the time-depth curve. Poseidon North 1 has no approved checkshot file and is recorded as a factual data gap (`checkshot_availability: NOT_AVAILABLE`), not substituted with another well's data. See `INCREMENT_04_MANIFEST.md` for the full statistics, tables, and figures. **NOTE:** `INCREMENT_04_MANIFEST.md` incorrectly stated TVDSS is strictly increasing after Depth-tie conditioning for all three wells; this was corrected by Increment 4.1 (see below) — do not rely on that original statement.
- **Increment 4.1 update:** a narrowly scoped corrective patch to Increment 4, applied after an independent numerical-method audit, that does NOT begin Increment 5 or any later-phase work. It corrected an order-dependent tie-break: `tvdss_to_owt`/`owt_to_tvdss` (and their TWT equivalents) previously resolved a repeated value on the axis being inverted by silently keeping whichever tied row was encountered first in the Depth-conditioned table and discarding the other, with no audit trail beyond a bare count. This is replaced by an explicit, order-invariant policy (`p2mem.time_depth.build_axis_conditioned_lookup_table`/`build_axis_conditioned_tables_for_well`): every tied value is grouped by exact equality regardless of parse order, every tied row is registered in a new audit register (`checkshot_time_axis_tie_register.csv` — separate from, and never confused with, the pre-existing Depth-axis `checkshot_duplicate_tie_register.csv`), and the group's dependent-value MEDIAN becomes the conditioned representative (order-invariant; a screening-level choice, not proof the original TVDSS↔OWT relationship was single-valued at that tied value). A genuine reversal (not a tie) raises a typed error rather than being sorted or forced monotonic. Independently reproduced real-data counts: Poseidon 2 has two TVDSS-axis and two OWT-axis tie groups after Depth-tie conditioning; Boreas 1 has one TVDSS-axis tie group and zero OWT-axis tie groups; Proteus 1ST2 has none of either — correcting `INCREMENT_04_MANIFEST.md`'s original, incorrect "strictly increasing for all three wells" statement. This patch also hardens `trapezoidal_integrate` and `compute_sonic_checkshot_drift` to validate their numerical preconditions (finite, one-dimensional, equal-length, strictly increasing MD/x where required) — most notably, a decreasing or duplicate MD run can no longer silently produce a physically invalid negative transit time; it now raises a typed error instead. None of this hardening changes the already-verified real Poseidon 2 sonic-drift result, which is bit-for-bit unchanged. See `INCREMENT_04_1_MANIFEST.md` for the full audit, corrected statistics, and re-verification record.

- **Increment 4.1.1 update:** a narrowly scoped numerical-validation corrective patch to Increment 4.1, applied after an independent numerical-method/software-QA audit, that does NOT begin Increment 5 or any later-phase work, and does NOT alter any previously verified real Poseidon 2/Boreas 1/Proteus 1ST2 result. It corrected four blocking defects and one input-safety gap. (1) `build_axis_conditioned_lookup_table` grouped ALL occurrences of an identical axis value together GLOBALLY before checking for a reversal, so a reversal that returned to an already-seen value (e.g. `[100.0, 200.0, 100.0]`) was silently hidden rather than raising a typed error; it now evaluates the ORIGINAL, ungrouped sequence's successive differences for negativity BEFORE any grouping is attempted — provably equivalent to the 4.1 behavior for every legitimate adjacent tie, and strictly stronger against a non-adjacent reversal. (2) `compute_sonic_checkshot_drift`/`find_longest_finite_positive_run` validated MD monotonicity only within the selected finite-positive-VP run, so a decreasing or duplicate MD value outside that run (e.g. at a NaN-VP station) could pass silently; the COMPLETE canonical `md_m` array is now required finite and strictly increasing before run-selection (the real Poseidon 2 MD array, 31,897 samples, was independently re-verified to already satisfy this — the real sonic-drift result is bit-for-bit unchanged). (3) `compare_checkshot_to_survey` reached an untyped NumPy `ValueError` ("zero-size array to reduction operation") if every checkshot Depth row fell outside the locked survey's own MD coverage; it now raises a typed error naming the well, the checkshot Depth range, and the survey MD coverage. (4) `p2mem.io.checkshot.load_checkshot_surveys` did not catch the typed numerical-conditioning error, so a defect in one well's data could stop the entire batch; it is now caught per well (never via a blanket exception handler) and recorded as a typed, isolated per-well failure, exactly like every other expected failure mode. (5) `seconds_to_milliseconds`/`milliseconds_to_seconds` coerced their input directly, unlike `p2mem.units`'s Increment-1 input-safety policy, so a boolean, numeric-looking string, or complex value would be silently reinterpreted rather than rejected; both now reject such input with a typed error via a local, documented copy of `p2mem.units`'s identical private dtype check (`p2mem/units.py` itself remains LOCKED and unmodified). See `INCREMENT_04_1_1_MANIFEST.md` for the full audit, the regression-test list, and the re-verification record.

- **Increment 5 update:** formation-top ingestion independently reproduces both required regression findings from the real approved files. Poseidon 2's "selected readable" file's supplied TVDSS equals `MDRT - 21.8 m` EXACTLY for every one of its 9 markers (21.8 m is the well's own rotary-table elevation) — a vertical-well-assumption depth-reference defect that is corrected in the derived `TVDSS_survey_corrected_m` representation, reproducing residuals of ≈0 m at Sea Bed, ≈+1.68 m at Plover Fm (Top Reservoir), and ≈+2.49 m at TD. Boreas 1's supplied TVDSS does not follow that pattern (maximum absolute residual 0.0416 m, below the 0.05 m tolerance) and is therefore NOT corrected — this is applied per file, per well, never by analogy from Poseidon 2's defect. Poseidon North 1 and Proteus 1ST2 have no approved formation-top file and are recorded as a factual data gap (`formation_top_availability: NOT_AVAILABLE`), never substituted with another well's tops or correlated by depth alone. Both formation-top file representations for both wells are classified `well_identity_evidence_status: inferred_unverified` — a deliberately more conservative classification than Increment 4's checkshot files, since neither the HRS files (no well name in the body) nor the readable files (well name only in a project-supplied comment) constitute independently verified file content; see `INCREMENT_05_MANIFEST.md` Section 2 for the full rationale. No lithology interpretation, petrophysical calculation, or geological correlation is performed on these markers — the Increment 5 marker-depth comparison figures are explicitly labelled as depth comparisons only.

- **Increment 5.1 update:** a narrowly scoped corrective patch to Increment 5, applied after an independent technical/software-QA audit, that does NOT begin Increment 6 or any later-phase work, and does NOT alter any previously verified real Poseidon 2/Boreas 1 formation-top result above. It corrected three defects and one documentation-accuracy gap. (1) `reconcile_formation_top_sources` documented "zero canonical markers in common between the two sources" as a fatal `NO_COMMON_MARKERS` ERROR, but actually tested the emptiness of the UNION of both sources' canonical names, so two entirely disjoint, non-empty marker sets were silently accepted as one-sided `NOT_COMPARABLE` entries instead of being rejected; the check now explicitly evaluates the INTERSECTION of the two sources' canonical names, and `load_formation_top_well` consequently raises `TopSourceReconciliationError` for this condition, while a legitimate one-sided marker (with at least one marker genuinely shared) remains the pre-existing, non-fatal `NOT_COMPARABLE` case. (2) `load_formation_top_surveys` recorded only the HRS file's path in every `TopIngestionFailure.source_path`, regardless of which file/stage actually failed, so a failure originating from the "selected readable" file (a missing file, malformed content, or a contract mismatch) could leak that file's own absolute path, unsanitized, into exported issues/availability/manifest rows; `TopIngestionFailure` now carries an explicit `failure_origin` and both `hrs_path`/`readable_path` fields, and every exporting function in `p2mem.io.tops_inventory` now sanitizes both candidate paths (literal substring replacement only, never a regex). (3) `reconcile_formation_top_sources` — the public in-memory API, as distinct from the file parsers, which already enforced this — did not validate its own documented contract before any numerical comparison: non-finite or negative MDRT/TVDSS values, a shorter `NOTE_source` tuple (previously reaching an untyped `IndexError`), a non-1-dimensional array, and an unvalidated `mdrt_agreement_tolerance_m` keyword (previously accepting NaN, a negative value, or a boolean) were all silently accepted or reached an undocumented incidental error; all are now rejected before any comparison, with deliberate, documented exceptions. (4) `INCREMENT_05_MANIFEST.md`'s statement that no absolute path exists ANYWHERE in the package was overbroad and inaccurate, since existing synthetic tests and historical documentation intentionally contain fake absolute-path strings as test inputs; the precise, narrowly scoped claim ("no environment-dependent build path appears in exported CSV/JSON outputs") is now stated explicitly in `INCREMENT_05_1_MANIFEST.md`, which also acknowledges the prior wording was overbroad — `INCREMENT_05_MANIFEST.md` itself is a locked historical record and is NOT rewritten. See `INCREMENT_05_1_MANIFEST.md` for the full audit, the regression-test list, and the real-data non-regression verification record.

- **Increment 5.1.1 update:** a further narrowly scoped corrective patch to Increment 5.1, applied after an independent technical/software-QA audit, that does NOT begin Increment 6 or any later-phase work, and does NOT alter any scientific result, tolerance, depth-mapping method, formation-top contract, or real-data output. It corrected one remaining reconciliation edge case and three documentation-accuracy gaps. (1) Increment 5.1 correctly rejected two non-empty, disjoint marker sets, but its condition ("both sources non-empty AND intersection empty", plus a separate both-empty case) still silently accepted the one remaining zero-common-markers configuration: exactly one source entirely empty and the other non-empty (the intersection of an empty set with anything is itself empty, so this was still a zero-common-markers condition). `reconcile_formation_top_sources` now uses the single, strictly correct check `if not common_markers` — empty in every zero-common-markers case (both empty, either side alone empty, or both non-empty and disjoint) and never empty whenever at least one canonical marker is genuinely shared, so a legitimate one-sided marker alongside at least one shared marker remains the pre-existing, non-fatal `NOT_COMPARABLE` case. (2) `INCREMENT_05_1_MANIFEST.md` stated that the real Increment 5 baseline ZIP's SHA-256 "matches" a governing-instruction-supplied hash that was, in fact, a two-character truncation of the real 64-character value — a mathematical impossibility, now stated transparently as a documentation/input typo, never as baseline corruption or uncertainty. (3) `INCREMENT_05_1_MANIFEST.md` Section 7 stated `outputs/` is excluded from the delivered ZIP; this was false — the delivered Increment 5.1 ZIP packages the `outputs/` tree (including the byte-identical Increment 5 formation-top outputs/figures), exactly as every prior increment's ZIP has; only dev-only regenerated-comparison directories and private raw source files are excluded, now stated accurately. (4) the same manifest's clean-room section stated no `*.las` file exists in the package; this was false because small, intentionally packaged synthetic LAS/checkshot/deviation/top fixtures exist under `tests/fixtures/` for portable testing — the corrected wording distinguishes these fictional fixtures from the genuinely excluded real/private project source files. See `INCREMENT_05_1_1_MANIFEST.md` for the full audit, the regression-test list, and the real-data non-regression verification record.

- **Increment 6 update:** the gamma-ray QC, screening-proxy sensitivity, well-frame and method-eligibility framework adds NO calibration and NO interpretation. Four specific limitations are newly quantified and disclosed. (1) **Boreas 1's ECGR is excluded, not corrected.** Its independently measured median (8.40 API) sits 4.3×–4.9× below the other three wells' GR medians, its range spans [−0.0001, 519.18] API including values at and below zero, and 2,059 of its samples lie above the well's own declared seabed marker. No tool header, calibration record, or environmental-correction metadata exists to adjudicate whether this is a scale, unit, tool-type, or acquisition problem, so the curve is formally excluded from every GR-derived calculation under the machine-readable reason `BOREAS_ECGR_SCALE_UNRESOLVED`. Rescaling it would fabricate a calibration this project does not have. (2) **The screening proxy is not a shale volume and is strongly endpoint-dependent.** `VSH_GR_linear_proxy_frac` is the linear identity of a clipped GR index computed under ASSUMED per-well percentile endpoints; across the three configured scenarios the proxy median moves by up to 0.12 dimensionless, and the qualifying sonic-NCT-candidate thickness moves by a factor of 8.0 in Poseidon 2 (144 m to 1,159 m), 3.5 in Poseidon North 1 and 2.3 in Proteus 1ST2. Any later NCT or pore-pressure result built on a single endpoint/threshold choice would inherit that full range as unquantified uncertainty. (3) **Eligibility is not validity.** The three masks record only whether a sample is technically admissible as INPUT to a later method; none of those methods is implemented, fitted, or validated here, and a sonic-NCT-candidate interval is emphatically not evidence that any interval is normally compacted, is a selected donor interval, or is any named lithology. Density and dynamic-elastic eligibility in all four wells begins only around 3,900–4,800 m TVDSS, so a later overburden integration cannot be supported from surface by these logs alone regardless of per-sample eligibility. (4) **Poseidon North 1 and Proteus 1ST2 have no approved formation tops**, so their above-seabed sample counts are reported as *not determinable* (never as 0) and all their results remain depth-tied and stratigraphically unvalidated. See `INCREMENT_06_MANIFEST.md` for the full measured statistics, sensitivity tables, and verification record.

- **Increment 6.1 update:** a narrowly scoped corrective patch to Increment 6, applied after an independent geomechanics/rock-physics and software audit. It does NOT start Increment 7 and implements no pore pressure, NCT fitting, elastic-property calculation, rock strength, stress, or wellbore-stability work. Four findings were corrected. (1) **Vp/Vs terminology and boundaries were scientifically inaccurate.** Increment 6 called every excluded Vp/Vs ratio "non-physical" and used a strictly exclusive √2 bound. With *r* = Vp/Vs, ν = (r²−2)/(2(r²−1)) and K = ρ(Vp²−(4/3)Vs²): *r* = √2 gives ν = 0 **exactly** and must be accepted by a non-negative-ν policy, so the bound is now **inclusive**; √(4/3) < *r* < √2 gives a **positive** bulk modulus with a negative Poisson's ratio — unusual and outside this project's conservative policy but *not* physically impossible; only *r* ≤ √(4/3) implies a non-positive bulk modulus and is genuinely outside the isotropic elastic model; and *r* > 4 is a **configured plausibility limit**, not a Poisson-domain boundary (ν ≈ 0.467 there). The condition is renamed a *configured non-negative-Poisson-ratio applicability screen*, the three bounds are separately named in config, and exclusions are diagnosed **by regime** rather than aggregated. Independently re-measured on the real data, the previously aggregated counts decompose as: Boreas 1 — 1 positive-K/negative-ν, 0 non-positive-K; Poseidon 2 — 22 and 0; Poseidon North 1 — **3 and 4** (the former single count of 7); Proteus 1ST2 — 3 and 0. No real sample sits exactly at √2, so the inclusive-bound correction changes no eligible count in this dataset — it corrects the policy, not the numbers. (2) **Poseidon North 1's machine-readable `use_status` contradicted its own data.** It read `screening_proxy_allowed` while the well has no approved formation tops; it is now `screening_proxy_allowed_depth_tied`, and a config invariant makes any contradictory has-tops/use-status pairing fail config loading loudly. (3) **Unsupported lithology and correlation claims were removed, and the named-lithology gate is no longer circular.** Project-specific "clastic-dominated" assertions are gone (a disclaimer does not undo an assertion), and the "regionally persistent" cross-well claim is replaced by the factual statement that low-GR intervals occur *independently* in each of the three GR-eligible wells with cross-well stratigraphic persistence **not established** — it cannot be, since two of those wells have no approved tops. `named_lithology_assigned` is now **derived** from an actual validation pass over every persisted label, per-well note, mask name and manifest statement rather than hardcoded; injecting a prohibited term makes the validation, the manifest flag and the completion gate fail together. (4) **Gross versus net interval thickness was ambiguous.** The reported "qualifying thickness" summed block endpoint spans that may contain explicitly bridged ineligible samples. Thickness is now exported only as `gross_thickness_*_m` (endpoint span, bridging-inclusive) or `net_thickness_*_m` (bridged gaps removed), every mask is additionally decomposed under a strict-no-gap policy, every sensitivity case reports its bridged-sample and interrupted-block counts, and Figure 4 distinguishes qualifying from rejected sub-threshold blocks with the counted population stated. Measured for Poseidon 2: configured **gross** 144.4–1,159.2 m (factor 8.03) versus **strict-no-gap** 140.0–1,110.2 m (factor 7.93). See `INCREMENT_06_1_MANIFEST.md` for the full audit and re-verification record.
- **Increment 6.1.1 update:** a second, narrower corrective patch, applied after an independent audit of Increment 6.1. It does NOT start Increment 7 and implements no pore pressure, NCT fitting, elastic-property calculation, rock strength, stress, or wellbore-stability work; it changes no configured tolerance, threshold, or qualifying-block policy. Five findings were corrected. (1) **The named-lithology validator could be bypassed.** The allowlist that lets per-well prose name a *method* contained `"shale gas"`, which is not a method-category phrase but can be a direct geological/hydrocarbon assertion — `"This interval contains shale gas."` passed with zero violations. The allowlist is reduced to `shale proxy` and `shale volume`, the only two phrases that are genuine method/quantity names required to state this project's boundaries, and the allowance is now evaluated **per sentence** and suppressed wherever the sentence carries a geological-assertion cue (`contains`, `comprises`, `bearing`, `facies`, …) or the phrase is immediately preceded by a quantity cue (`has a high shale volume`). Explanatory use of `shale proxy` / `shale volume` as a method or quantity name is preserved; use in any label remains prohibited outright. Regression tests prove the allowlist can no longer hide a geological assertion, and that injecting either bypass case into manifest content makes `named_lithology_assigned` true, `lithology_validation.n_violations` non-zero, and the completion gate fail. (2) **The active Vp/Vs description was stale.** The live top-level docstring of `p2mem.method_eligibility` still described the screen as keeping samples "inside the Poisson domain" under a strictly exclusive `Vp/Vs > √2`, contradicting the implementation corrected in 6.1. It now states the *configured non-negative-Poisson-ratio applicability screen* with an **inclusive** `Vp/Vs ≥ √2` bound and says explicitly that this is neither a physical-possibility test nor a boundary of the mathematical Poisson domain; a regression test asserts the corrected wording against the live module documentation. Superseded wording is retained only where it is clearly preserved as historical record. (3) **Interruption counts were ambiguous or wrong.** `n_interruptions` was a boolean-like flag reported as a count — all 327 bridged blocks reported exactly 1 — and `n_interrupted_subruns` did not describe what it counted. Interval records now carry `n_bridged_samples` (ineligible samples absorbed inside the gross block), `n_bridged_gaps` (distinct bridged runs) and `n_eligible_subruns` (strictly contiguous eligible sub-runs), with `n_bridged_gaps = n_eligible_subruns − 1` enforced at construction; a block with no gap reports 0/0/1 and a block with two separate bridged gaps reports 2 gaps and 3 sub-runs. The sensitivity summary distinguishes `n_bridged_samples_in_qualifying_blocks`, `n_bridged_gaps_in_qualifying_blocks` and `n_interrupted_qualifying_blocks`. No ambiguous alias survives in any active export. (4) **The notebook's output count was wrong.** It creates and checks eight deterministic CSV/JSON outputs while its gate text and printed label said seven; both now say eight and the gate asserts the declared count and the existence of all eight. Four figures remain separate — twelve outputs/figures in total. (5) **The Increment 6.1 delta arithmetic was incorrect.** That record stated "Changed (12)"; a clean recursive comparison gives **18 changed, 3 added, 0 removed (21 path differences)**. The locked 6.1 record is not rewritten; the statement is explicitly superseded, and both deltas are reported file by file, in `INCREMENT_06_1_1_MANIFEST.md`.
- **Increment 6.1.2 update:** a third, narrower corrective patch, applied after an independent audit of Increment 6.1.1. It does NOT start Increment 7, implements no pore pressure, NCT fitting, elastic-property calculation, rock strength, stress, or wellbore-stability work, changes no scientific threshold, tolerance, endpoint scenario, contiguity policy, GR disposition, depth-mapping rule, or real-data interpretation, and adds no runtime dependency. Two residual findings were corrected. (1) **The named-lithology validator was wrong in both directions.** Increment 6.1.1 decided the method-phrase allowance from a sentence-wide assertion-cue list plus a fixed look-behind window. Because nothing to the *right* of an allowed phrase was inspected, and because the normalizer erased possessives, `"Poseidon 2's shale volume is high."`, `"Poseidon 2's shale volume is 70 percent."` and `"The interval's shale volume exceeds 60 percent."` all passed with **zero violations** — the assertion lives in the predicate and the possessive, neither of which was ever read. Conversely, because a sentence-wide cue fires without asking what the cue word is predicated *of*, `"This method contains a shale proxy calculation."` and `"The analysis shows no shale volume was computed."` were reported as **false** lithological assertions. The allowance is now a property of each *occurrence*, decided from its own local grammar: a possessive geological entity, an adjacent magnitude modifier, a right-hand magnitude or dominance predicate, or a composition verb whose resolved subject head is a geological entity each withdraw it. A bare copula does not — `"the shale proxy is dimensionless"` describes a method, not an amount of rock. Apostrophes are parsed rather than erased, line breaks end sentences, ambiguous project-specific sentences fail closed, and labels keep zero latitude. A table-driven matrix of 59 discrimination rows across all three scopes, plus gate-path injections, pins the behaviour in both directions. (2) **Interval-record invariants were only partially enforced.** The 6.1.1 constructor checked only `n_bridged_gaps == n_eligible_subruns − 1`, and skipped even that whenever a value was `None` or `n_eligible_subruns` was 0 — so records with missing, negative, boolean or contradictory counts were constructible. All three counts are now validated together: required, strictly integral (booleans, strings, complex values and fractional floats rejected, never coerced), non-negative, `n_eligible_subruns ≥ 1`, `n_bridged_gaps = n_eligible_subruns − 1`, `n_bridged_samples = 0` **iff** `n_bridged_gaps = 0`, and `n_bridged_samples ≥ n_bridged_gaps`. Removed legacy field names are rejected outright rather than ignored. See `INCREMENT_06_1_2_MANIFEST.md` for the full audit and re-verification record.
- **Increment 6.1.3 update:** the fourth corrective patch, and an architectural one. It does NOT start Increment 7, changes no scientific threshold, tolerance, endpoint scenario, contiguity policy, GR disposition, depth-mapping rule, or real-data interpretation, and adds no runtime dependency. **(1) The validator no longer parses English.** Increments 6.1.1 and 6.1.2 each tried to decide from grammar whether a sentence containing a rock name named a *method* or asserted *geology*. Both were audited; both failed in both directions. 6.1.2 still missed `"The interval has a shale volume."`, `"The shale volume in Poseidon 2 exceeds 60 percent."` and `"Poseidon 2 shale volume was determined to be high."` — a missing predicate verb, a prepositional possessor, and a magnitude beyond the look-ahead window — while wrongly rejecting `"The well contains no shale volume estimate."` Each is a list gap or a window edge, and no finite grammar closes an unbounded space. Every scope is now decided by membership or equality: labels and project-specific prose have **zero allowance** (the `allow_method_phrases` parameter is deleted, so there is no exemption path left to bypass); method and limitation wording is admitted only when the text is **exactly equal** to a member of the closed, provenance-tagged `METHOD_STATEMENTS` registry, whose five members were *measured* by scanning every string literal in active packaged source under a zero-allowance rule; and explanatory text may use a rock name generically but never in a sentence referring to a project well **or a project rock body** (6.1.2 checked well names alone). All the grammar machinery — cue lists, magnitude vocabularies, look-behind and look-ahead windows, subject resolution, occurrence classification — is **deleted**, and a test asserts none of it is importable. Correctness is proved by **closure** over every prohibited term in every position inside carrier prose built from the exact constructions that defeated both previous implementations, not by a list of example sentences. **(2) Three exported strings had never been validated at all** — including a `calibration_status` value literally containing `not_a_shale_volume`, written into every row of `gr_proxy_sensitivity_summary.csv`. All three are now scanned, raising `n_fields_checked` from 138 to 143; that single scalar is the only difference in any output. **(3) The interval-record type gate is hardened**: keyword acceptance is a whitelist over declared slots (a misspelled count name no longer leaves the real count silently unset), floats are rejected outright including whole-valued ones, and `NaN`/`Inf` raise a typed `PetrophysicsInputError` before any conversion is attempted. See `INCREMENT_06_1_3_MANIFEST.md` for the full audit and re-verification record.
- **Increment 6.1.4 update:** a deliberately narrow completion of the Increment 6.1.3 architecture — one scope rule, no other change. It does NOT start Increment 7 and changes no scientific threshold, tolerance, policy, disposition or result. 6.1.3 closed labels and project-specific prose by removing every exemption, and closed method wording by exact registry membership — but left **explanatory** scope decided by a finite token list of project references. That is the same shape of rule that failed in 6.1.1 and 6.1.2, and it failed the same way: ordinary stratigraphic and exploration nouns were missing from the list, so `"The member is shale."`, `"The group is limestone."`, `"The package is a clean sandstone."`, `"The play is shale-dominated."`, `"The prospect is carbonate."` and `"The target is sandstone."` all passed — while `"The upper member is a clean sandstone reservoir."` failed only because `reservoir` happened to be listed, an accident that shows what a list-shaped rule produces. The rule is **inverted**: explanatory text carrying a rock term is a violation unless the text is a member of the closed `GENERIC_EXPLANATORY_STATEMENTS` registry. It **fails closed**, so no vocabulary gap can admit anything, and `PROJECT_WELL_NAME_TOKENS`, `PROJECT_ROCK_BODY_TOKENS` and `PROJECT_REFERENCE_TOKENS` are deleted. The generic registry is **empty** as a measured fact — this project persists exactly one explanatory field and it carries no rock name — but the mechanism is live and tested, because later increments explaining gamma-ray non-uniqueness will need it. **All four scopes are now closed by membership or exact equality**, and the closure proof covers every one of them. See `INCREMENT_06_1_4_MANIFEST.md` for the full record.
- **Increment 6.1.5 update:** the final Increment 6 corrective patch, and the one that changes the security model rather than the recognizer. Increments 6.1–6.1.4 all asked *"does this text contain a geological assertion?"* and all four shared one assumption — content is acceptable by default and becomes unacceptable only when a recognizer fires. All four were defeated, finally by a word no recognizer had been given: in the 6.1.4 package `"The interval is chalk."`, `"…halite."`, `"…gypsum."`, `"…conglomerate."`, `"…chert."`, `"…tuff."`, `"…basalt."`, `"…dolostone."`, `"…lignite."`, `"…calcareous."` all passed label, interpretive and explanatory scope, and so did `"The interval is qxzite."` — a word that does not exist. (They already failed in *method* scope, the one scope that then required registration; that contrast is what this patch generalises.) **A longer blacklist is not the fix and is not the mechanism the assurance claim rests on.** The model is inverted: labels must be members of a typed `APPROVED_LABELS` registry; interpretive, method and explanatory statements must resolve to a registered `statement_id`, or — interpretive only — a reviewed `template_id` whose substitutions are restricted to declared types. Id and exact rendered text are validated together, so unknown ids, id/text mismatches, case changes, near-misses, duplicate ids and undeclared template fields all fail. The registries were **measured** from the actual persisted export: 17 approved labels, 17 registered statements, 1 controlled template. `PROHIBITED_LITHOLOGY_TERMS` remains only as a diagnostic linter that authorizes nothing — every rejection listed above occurs with that linter returning **empty**. **The defensible assurance statement**, superseding the wording of every earlier Increment 6 manifest: *every persisted project-specific classification and interpretive statement is generated from an approved typed value, controlled template, or registered statement; arbitrary free text cannot enter these controlled fields.* This is **not** a claim that the software understands or exhaustively recognizes natural-language lithology — it does not, and no earlier version did. Free-form notebook narrative lies outside these controlled fields and remains subject to manual scientific review. See `INCREMENT_06_1_5_MANIFEST.md`.
- **Increment 6.1.6 update:** authorization is now enforced where records are actually EMITTED. 6.1.5 closed the unit validator but validated a scope the manifest builder *reconstructed*; a `GrEndpointScenario` with `description="The interval is chalk."` was persisted verbatim while absent from that 143-field scope, so it could not move `named_lithology_assigned`. Five findings, all reproduced first: the exported endpoint-description bypass; incomplete output coverage (endpoint `description`, `limitations`, `purpose`, `population_statement`, `limiting_criterion`, calibration/evidence labels, diagnostic text and manifest prose were not consistently validated); a field-kind mismatch letting `use_status="GR"` pass because `APPROVED_LABELS` was keyed by value alone; a stale exported `derivation` still describing prohibited-term validation; and a duplicated `named_lithology_statement` literal that could diverge from its registered copy while the gate stayed green. `p2mem.io.output_policy` now declares a category for **every** string column and JSON path of all eight artifacts (105 entries, zero unclassified) across eight closed categories, with **no** general category admitting arbitrary prose; labels are authorized by `(field_kind, value)`; the manifest statement comes from its registered id; and export authorizes the exact pre-serialization records, writes, then **re-authorizes the written bytes** so a post-authorization mutation is caught. Assurance metrics say what they count — `n_fields_checked=143` is replaced by `scope_object_fields_checked` plus an `emitted_field_coverage` block reporting total emitted occurrences, controlled occurrences, structural occurrences, occurrences outside the guarantee, unclassified fields, unauthorized fields and field-kind mismatches. Machine diagnostics and operator-facing issue text are counted separately and are explicitly outside the controlled-interpretation guarantee. See `INCREMENT_06_1_6_MANIFEST.md`.
- **Increment 6.1.7 update:** the 6.1.6 gate still inferred field presence from non-empty string values and wrote to the official directory before its post-write check. Exact schemas now validate all eight artifacts independently of content, including inventory, ordered CSV columns, JSON keys, requiredness, strict types, finite numerics and approved dynamic well identifiers. Every declared string occurrence is authorized even when empty or numeric-looking. The complete candidate set is written to an isolated sibling directory, re-read, re-validated and compared field-by-field and row-by-row using a typed canonical representation before publication. Missing/unknown fields, post-write type changes, row reordering, authorized-to-authorized substitutions, stale artifacts and serializer errors fail closed without altering the official destination. Publication is failure-atomic for handled process errors; no claim is made about multi-file atomicity across power loss or operating-system failure. No scientific calculation, threshold, disposition or result changed. See `INCREMENT_06_1_7_MANIFEST.md`.

- **Increment 7 / 7.0.1 / 7.0.2 / 7.0.3 update:** the density QC, coverage-qualification and vertical overburden-stress framework adds NO calibration. (1) **No approved well supports an absolute vertical overburden-stress curve.** All four wells carry a contract-resolved `RHOB_kg_m3` curve and are fully depth-mapped within survey coverage. Boreas 1 and Poseidon 2 have quantified unmeasured seabed-to-log columns of 3,486.54 m and 3,567.14 m TVD; Poseidon North 1 and Proteus 1ST2 have no determinable seabed or shallow-gap thickness because no approved formation-top file exists for either. (2) The low shallow-density endpoint is conditional on a fully saturated column, the high endpoint is an illustrative same-well P05 scenario motivated by monotonic-compaction reasoning, and the midpoint has no evidentiary support. None is a physical bound or best estimate, and 57–81% of the scenario totals is assumed. (3) The 10 m short-gap threshold is a reviewable heuristic evaluated through the complete 0/2/5/10/20/30 m sensitivity—not a rigorous error bound inferred from endpoint contrast. (4) **Increment 7.0.1 closes a fail-closed defect:** profile construction now truncates at every unresolved internal gap, including short or unmapped gaps; it no longer checks only the `long_internal_gap` class. The real approved workflow still bridges Boreas 1's 7.12 m gap, while its 15.64 m gap truncates the column and excludes 2,511 deeper eligible samples. (5) **Increment 7.0.2 closes two assurance defects:** the notebook gate now reads the schema-declared `assumed_density_basis` field, and the public frozen config constructor enforces loader-equivalent strict typing and invariants for direct construction and `dataclasses.replace()`. (6) **Increment 7.0.3 closes the remaining percentile-provenance defect:** the configured high percentile must be exactly 5.0 because the implemented statistic and scenario are explicitly P05; an inconsistent label now fails at both YAML loading and public construction. The packaged value was already 5.0, so no scientific output changes. See `INCREMENT_07_0_3_MANIFEST.md` for the corrective verification record.

- **Increment 7.0.4 update:** a final corrective patch after end-to-end audit. The high scenario's P05 is now calculated over `eligible_for_measured_integration`, matching its documented population; the distinct all-finite P05 remains available as factual QC. Every public threshold override and threshold-bearing result rejects boolean, textual, complex, negative, NaN and infinite values. Scenario accounting now exports three exhaustive fractions—assumed, conditioned and measured—whose constructor-enforced sum is one. The locked seabed reader fails closed on a missing table and malformed, duplicate or non-finite matching rows while still representing a valid header-only table as an explicit empty mapping. JSON output uses explicit UTF-8 bytes with LF newlines, making regeneration byte-identical on Windows and Linux/Colab. Approved real-data stress values, statuses, thresholds and figures remain unchanged; the QC/scenario CSV schemas and assurance JSON change additively to expose the corrected populations and fraction accounting. See `INCREMENT_07_0_4_MANIFEST.md`.

## Screening-level statement

**This 1D Mechanical Earth Model is a screening-level, uncalibrated, educational work product.** It has not been validated against independent field measurements and does not carry the assurance level required for drilling engineering, well design, casing/mud-weight selection, or any other operational decision. Any numerical result produced by this codebase should be read as illustrative of a defensible methodology applied to the available data, not as a certified or field-ready prediction.


In [ ]:
%%writefile config/pore_pressure.yml
# Poseidon 2 1D MEM — Increment 8
# Pore-pressure and effective-stress screening policy.
# Tier C — Screening-Level / Uncalibrated Educational.
#
# All values below are explicit screening assumptions. They are not measured
# formation pressures and are not calibration data. No NCT, Eaton, Bowers,
# equivalent-depth, drilling-exponent, or other overpressure transform is
# enabled by this file.

schema_version: "8.0.1"
increment: 8
assurance_tier: "Tier C - Screening-Level / Uncalibrated Educational"

locked_inputs:
  vertical_stress_profile: "outputs/07_density_overburden/vertical_stress_profile.csv"
  shallow_column_scenarios: "outputs/07_density_overburden/shallow_column_scenarios.csv"
  overburden_eligibility: "outputs/07_density_overburden/overburden_eligibility_summary.csv"
  method_eligibility: "outputs/06_petrophysics_eligibility/method_eligibility_summary.csv"
  thickness_sensitivity: "outputs/06_petrophysics_eligibility/thickness_sensitivity_summary.csv"
  petrophysics_manifest: "outputs/06_petrophysics_eligibility/petrophysics_eligibility_manifest.json"
  sonic_checkshot_drift: "outputs/04_checkshot_time_depth/sonic_checkshot_drift_summary.csv"

hydrostatic_reference:
  gravity_m_s2: 9.80665
  fluid_density_low_kg_m3: 1020.0
  fluid_density_base_kg_m3: 1025.0
  fluid_density_high_kg_m3: 1030.0
  pressure_datum: "mean_sea_level_gauge_zero"
  depth_coordinate: "tvdss_m_positive_downward"

effective_stress:
  equation: "sigma_v_effective_pa = sigma_v_total_pa - biot_alpha * pore_pressure_pa"
  biot_alpha_scenarios: [0.8, 1.0]
  profile_biot_alpha: 1.0
  profile_fluid_density_kg_m3: 1025.0
  negative_values_policy: "retain_and_flag_never_clip"

pressure_calibration:
  required_types: ["RFT", "MDT", "DST", "FIT", "LOT", "XLOT", "DFIT"]
  available_file_count: 0
  cross_well_transfer_allowed: false

prediction_policy:
  nct_fit_enabled: false
  overpressure_transform_enabled: false
  pressure_calibration_required: true
  independent_normal_interval_evidence_required: true
  sonic_checkshot_drift_correction_allowed: false
  extrapolation_allowed: false

reporting:
  output_directory: "outputs/08_pore_pressure_effective_stress"
  increment_9_started: false


In [ ]:
%%writefile p2mem/pore_pressure_models.py
"""Typed records and invariants for Increment 8.

The records distinguish measured/locked inputs, configured assumptions,
hydrostatic references, and derived screening scenarios.  None represents a
calibrated formation-pressure measurement or an overpressure prediction.
"""

from dataclasses import dataclass
import math
from numbers import Integral, Real
from typing import Optional, Tuple


class PorePressureInputError(ValueError):
    """Raised when an Increment 8 public input violates its contract."""


ASSURANCE_TIER = "Tier C - Screening-Level / Uncalibrated Educational"
PACKAGE_VERSION = "0.8.1"
CONFIG_SCHEMA_VERSION = "8.0.1"
APPROVED_GRAVITY_M_S2 = 9.80665
APPROVED_FLUID_DENSITIES_KG_M3: Tuple[float, ...] = (1020.0, 1025.0, 1030.0)
APPROVED_BIOT_ALPHA_SCENARIOS: Tuple[float, ...] = (0.8, 1.0)
APPROVED_PROFILE_BIOT_ALPHA = 1.0
APPROVED_PROFILE_FLUID_DENSITY_KG_M3 = 1025.0
WELL_KEYS: Tuple[str, ...] = (
    "Boreas_1", "Poseidon_2", "Poseidon_North_1", "Proteus_1ST2",
)
PRESSURE_DATA_TYPES: Tuple[str, ...] = (
    "RFT", "MDT", "DST", "FIT", "LOT", "XLOT", "DFIT",
)
NCT_STATUSES: Tuple[str, ...] = (
    "NOT_ELIGIBLE_INPUT_QC_EXCLUSION",
    "WITHHELD_NO_PRESSURE_CALIBRATION_OR_INDEPENDENT_NORMAL_INTERVAL",
)
OVERBURDEN_STATUSES: Tuple[str, ...] = (
    "screening_sensitivity_only", "partial_measured_increment_only",
    "not_eligible",
)
SV_SCENARIOS: Tuple[str, ...] = ("low", "base", "high")
FLUID_SCENARIOS: Tuple[str, ...] = ("low", "base", "high")


def finite_real(value, name: str, *, minimum: Optional[float] = None,
                maximum: Optional[float] = None) -> float:
    """Return a strict finite real, rejecting bool, strings and complex."""
    if isinstance(value, bool) or not isinstance(value, Real):
        raise PorePressureInputError(f"{name} must be a finite real number.")
    out = float(value)
    if not math.isfinite(out):
        raise PorePressureInputError(f"{name} must be finite.")
    if minimum is not None and out < minimum:
        raise PorePressureInputError(f"{name} must be >= {minimum}.")
    if maximum is not None and out > maximum:
        raise PorePressureInputError(f"{name} must be <= {maximum}.")
    return out


def strict_int(value, name: str, *, minimum: int = 0) -> int:
    """Return an integer without coercion; bool is never an integer here."""
    if isinstance(value, bool) or not isinstance(value, Integral):
        raise PorePressureInputError(f"{name} must be an integer.")
    out = int(value)
    if out < minimum:
        raise PorePressureInputError(f"{name} must be >= {minimum}.")
    return out


def exact_bool(value, name: str) -> bool:
    if type(value) is not bool:
        raise PorePressureInputError(f"{name} must be exactly bool.")
    return value


def enum(value, name: str, allowed: Tuple[str, ...]) -> str:
    if not isinstance(value, str) or value not in allowed:
        raise PorePressureInputError(f"{name} must be one of {allowed!r}.")
    return value


@dataclass(frozen=True)
class PressureConfig:
    schema_version: str
    assurance_tier: str
    gravity_m_s2: float
    fluid_density_low_kg_m3: float
    fluid_density_base_kg_m3: float
    fluid_density_high_kg_m3: float
    biot_alpha_scenarios: Tuple[float, ...]
    profile_biot_alpha: float
    profile_fluid_density_kg_m3: float
    pressure_calibration_types: Tuple[str, ...]
    pressure_calibration_file_count: int
    cross_well_transfer_allowed: bool
    nct_fit_enabled: bool
    overpressure_transform_enabled: bool
    pressure_calibration_required: bool
    independent_normal_interval_evidence_required: bool
    sonic_checkshot_drift_correction_allowed: bool
    extrapolation_allowed: bool
    increment_9_started: bool

    def __post_init__(self) -> None:
        if self.schema_version != CONFIG_SCHEMA_VERSION:
            raise PorePressureInputError(
                f"schema_version must be {CONFIG_SCHEMA_VERSION!r}.")
        if self.assurance_tier != ASSURANCE_TIER:
            raise PorePressureInputError("Unexpected assurance_tier.")
        g = finite_real(self.gravity_m_s2, "gravity_m_s2", minimum=0.0)
        if g == 0.0:
            raise PorePressureInputError("gravity_m_s2 must be > 0.")
        lo = finite_real(self.fluid_density_low_kg_m3, "fluid_density_low_kg_m3", minimum=0.0)
        base = finite_real(self.fluid_density_base_kg_m3, "fluid_density_base_kg_m3", minimum=0.0)
        hi = finite_real(self.fluid_density_high_kg_m3, "fluid_density_high_kg_m3", minimum=0.0)
        if not lo < base < hi:
            raise PorePressureInputError("Fluid-density scenarios must satisfy low < base < high.")
        if not isinstance(self.biot_alpha_scenarios, tuple) or not self.biot_alpha_scenarios:
            raise PorePressureInputError("biot_alpha_scenarios must be a non-empty tuple.")
        alphas = tuple(finite_real(v, "biot_alpha", minimum=0.0, maximum=1.0)
                       for v in self.biot_alpha_scenarios)
        if len(set(alphas)) != len(alphas):
            raise PorePressureInputError("biot_alpha_scenarios must be unique.")
        finite_real(self.profile_biot_alpha, "profile_biot_alpha", minimum=0.0, maximum=1.0)
        finite_real(self.profile_fluid_density_kg_m3, "profile_fluid_density_kg_m3", minimum=0.0)
        if g != APPROVED_GRAVITY_M_S2:
            raise PorePressureInputError(
                "gravity_m_s2 must equal the approved Increment 8 value.")
        if (lo, base, hi) != APPROVED_FLUID_DENSITIES_KG_M3:
            raise PorePressureInputError(
                "Fluid-density scenarios must equal the approved Increment 8 values.")
        if alphas != APPROVED_BIOT_ALPHA_SCENARIOS:
            raise PorePressureInputError(
                "biot_alpha_scenarios must equal the approved Increment 8 values.")
        if self.profile_biot_alpha != APPROVED_PROFILE_BIOT_ALPHA:
            raise PorePressureInputError(
                "profile_biot_alpha must equal the approved Increment 8 value.")
        if self.profile_fluid_density_kg_m3 != APPROVED_PROFILE_FLUID_DENSITY_KG_M3:
            raise PorePressureInputError(
                "profile_fluid_density_kg_m3 must equal the approved Increment 8 value.")
        if tuple(self.pressure_calibration_types) != PRESSURE_DATA_TYPES:
            raise PorePressureInputError("pressure_calibration_types must match the closed required set.")
        strict_int(self.pressure_calibration_file_count, "pressure_calibration_file_count")
        for name in (
            "nct_fit_enabled", "overpressure_transform_enabled",
            "pressure_calibration_required",
            "independent_normal_interval_evidence_required",
            "cross_well_transfer_allowed",
            "sonic_checkshot_drift_correction_allowed",
            "extrapolation_allowed",
            "increment_9_started",
        ):
            exact_bool(getattr(self, name), name)
        if self.pressure_calibration_file_count != 0:
            raise PorePressureInputError("Increment 8 has no approved pressure-calibration file.")
        if (self.nct_fit_enabled or self.overpressure_transform_enabled
                or self.cross_well_transfer_allowed
                or self.sonic_checkshot_drift_correction_allowed
                or self.extrapolation_allowed or self.increment_9_started):
            raise PorePressureInputError("Prediction and extrapolation must remain disabled in Increment 8.")
        if not self.pressure_calibration_required or not self.independent_normal_interval_evidence_required:
            raise PorePressureInputError("Both prediction evidence requirements must remain enabled.")


@dataclass(frozen=True)
class PressureDataAvailability:
    well_key: str
    rft_count: int
    mdt_count: int
    dst_count: int
    fit_count: int
    lot_count: int
    xlot_count: int
    dfit_count: int
    availability_status: str

    def __post_init__(self) -> None:
        enum(self.well_key, "well_key", WELL_KEYS)
        counts = [strict_int(getattr(self, f), f) for f in (
            "rft_count", "mdt_count", "dst_count", "fit_count",
            "lot_count", "xlot_count", "dfit_count",
        )]
        if self.availability_status != "NOT_AVAILABLE" or any(counts):
            raise PorePressureInputError(
                "No approved pressure-calibration measurement is available in Increment 8.")


@dataclass(frozen=True)
class NctReadiness:
    well_key: str
    status: str
    candidate_configurations: int
    qualifying_thickness_min_tvd_m: Optional[float]
    qualifying_thickness_max_tvd_m: Optional[float]
    thickness_sensitivity_factor: Optional[float]
    pressure_calibration_available: bool
    independent_normal_interval_evidence_available: bool
    nct_fit_performed: bool
    overpressure_inferred: bool

    def __post_init__(self) -> None:
        enum(self.well_key, "well_key", WELL_KEYS)
        enum(self.status, "status", NCT_STATUSES)
        n = strict_int(self.candidate_configurations, "candidate_configurations")
        for f in (
            "pressure_calibration_available", "independent_normal_interval_evidence_available",
            "nct_fit_performed", "overpressure_inferred",
        ):
            exact_bool(getattr(self, f), f)
        if any((self.pressure_calibration_available,
                self.independent_normal_interval_evidence_available,
                self.nct_fit_performed, self.overpressure_inferred)):
            raise PorePressureInputError("Increment 8 readiness flags must all remain false.")
        values = (self.qualifying_thickness_min_tvd_m,
                  self.qualifying_thickness_max_tvd_m,
                  self.thickness_sensitivity_factor)
        if n == 0:
            if self.status != NCT_STATUSES[0] or any(v is not None for v in values):
                raise PorePressureInputError("No-candidate readiness record is inconsistent.")
        else:
            if self.status != NCT_STATUSES[1] or any(v is None for v in values):
                raise PorePressureInputError("Candidate readiness record is incomplete.")
            lo = finite_real(values[0], "qualifying_thickness_min_tvd_m", minimum=0.0)
            hi = finite_real(values[1], "qualifying_thickness_max_tvd_m", minimum=0.0)
            factor = finite_real(values[2], "thickness_sensitivity_factor", minimum=1.0)
            if lo <= 0.0 or hi < lo or not math.isclose(factor, hi / lo, rel_tol=1e-12):
                raise PorePressureInputError("NCT thickness sensitivity values are inconsistent.")


@dataclass(frozen=True)
class HydrostaticReferenceNode:
    well_key: str
    node_index: int
    fluid_scenario: str
    tvdss_m: float
    fluid_density_kg_m3: float
    gravity_m_s2: float
    pressure_pa: float

    def __post_init__(self) -> None:
        enum(self.well_key, "well_key", WELL_KEYS)
        strict_int(self.node_index, "node_index")
        enum(self.fluid_scenario, "fluid_scenario", FLUID_SCENARIOS)
        z = finite_real(self.tvdss_m, "tvdss_m", minimum=0.0)
        rho = finite_real(self.fluid_density_kg_m3, "fluid_density_kg_m3", minimum=0.0)
        g = finite_real(self.gravity_m_s2, "gravity_m_s2", minimum=0.0)
        p = finite_real(self.pressure_pa, "pressure_pa", minimum=0.0)
        if not math.isclose(p, rho * g * z, rel_tol=1e-12, abs_tol=1e-8):
            raise PorePressureInputError("Hydrostatic reference identity failed.")


@dataclass(frozen=True)
class EffectiveStressScenario:
    well_key: str
    sv_scenario: str
    fluid_scenario: str
    biot_alpha: float
    evaluation_tvdss_m: float
    total_vertical_stress_pa: float
    pore_pressure_reference_pa: float
    effective_vertical_stress_pa: float
    effective_stress_nonnegative: bool

    def __post_init__(self) -> None:
        enum(self.well_key, "well_key", WELL_KEYS)
        enum(self.sv_scenario, "sv_scenario", SV_SCENARIOS)
        enum(self.fluid_scenario, "fluid_scenario", FLUID_SCENARIOS)
        alpha = finite_real(self.biot_alpha, "biot_alpha", minimum=0.0, maximum=1.0)
        finite_real(self.evaluation_tvdss_m, "evaluation_tvdss_m", minimum=0.0)
        sv = finite_real(self.total_vertical_stress_pa, "total_vertical_stress_pa", minimum=0.0)
        pp = finite_real(self.pore_pressure_reference_pa, "pore_pressure_reference_pa", minimum=0.0)
        eff = finite_real(self.effective_vertical_stress_pa, "effective_vertical_stress_pa")
        flag = exact_bool(self.effective_stress_nonnegative, "effective_stress_nonnegative")
        expected = sv - alpha * pp
        if not math.isclose(eff, expected, rel_tol=1e-12, abs_tol=1e-8):
            raise PorePressureInputError("Effective-stress identity failed.")
        if flag != (eff >= 0.0):
            raise PorePressureInputError("effective_stress_nonnegative flag is inconsistent.")


In [ ]:
%%writefile p2mem/pore_pressure.py
"""Hydrostatic-reference and effective-stress calculations for Increment 8.

The module intentionally implements no NCT fit and no overpressure transform.
Hydrostatic pressure is a configured reference, not a measured formation
pressure. Effective stress is reported only as a scenario paired with the
Increment 7 vertical-stress scenarios.
"""

import math
from pathlib import Path
from typing import Iterable, Mapping, Sequence, Tuple

import yaml

from p2mem.pore_pressure_models import (
    ASSURANCE_TIER, PRESSURE_DATA_TYPES, EffectiveStressScenario,
    HydrostaticReferenceNode, NctReadiness, PorePressureInputError,
    PressureConfig, PressureDataAvailability,
)


class _NoDuplicateKeySafeLoader(yaml.SafeLoader):
    """Safe YAML loader that refuses silent duplicate-key overwrites."""


def _construct_mapping_no_duplicates(loader, node, deep=False):
    loader.flatten_mapping(node)
    mapping = {}
    for key_node, value_node in node.value:
        key = loader.construct_object(key_node, deep=deep)
        if key in mapping:
            raise ValueError(
                f"duplicate configuration key {key!r} at line "
                f"{key_node.start_mark.line + 1}")
        mapping[key] = loader.construct_object(value_node, deep=deep)
    return mapping


_NoDuplicateKeySafeLoader.add_constructor(
    yaml.resolver.BaseResolver.DEFAULT_MAPPING_TAG,
    _construct_mapping_no_duplicates,
)


def _locked_float_token(row: Mapping[str, str], field: str, context: str) -> float:
    try:
        token = row[field]
        if not isinstance(token, str) or not token.strip():
            raise ValueError("required non-empty string token")
        value = float(token)
    except (KeyError, TypeError, ValueError, OverflowError) as exc:
        raise PorePressureInputError(
            f"{context}: {field!r} must be a finite numeric token.") from exc
    if not math.isfinite(value):
        raise PorePressureInputError(
            f"{context}: {field!r} must be finite.")
    return value


def _locked_int_token(row: Mapping[str, str], field: str, context: str) -> int:
    try:
        token = row[field]
        if not isinstance(token, str) or not token.isdigit():
            raise ValueError("canonical non-negative integer token required")
        return int(token)
    except (KeyError, TypeError, ValueError, OverflowError) as exc:
        raise PorePressureInputError(
            f"{context}: {field!r} must be a canonical integer token.") from exc


def load_pressure_config(path) -> PressureConfig:
    """Load and strictly validate the human-authored Increment 8 policy."""
    p = Path(path)
    if not p.is_file():
        raise PorePressureInputError(f"Pressure policy is missing: {p.name!r}.")
    try:
        raw = yaml.load(
            p.read_text(encoding="utf-8"), Loader=_NoDuplicateKeySafeLoader)
        if not isinstance(raw, dict):
            raise TypeError("top-level mapping required")
        expected_top = {
            "schema_version", "increment", "assurance_tier", "locked_inputs",
            "hydrostatic_reference", "effective_stress", "pressure_calibration",
            "prediction_policy", "reporting",
        }
        if set(raw) != expected_top:
            raise KeyError("unknown or missing top-level field")
        locked = raw["locked_inputs"]
        h = raw["hydrostatic_reference"]
        e = raw["effective_stress"]
        c = raw["pressure_calibration"]
        pred = raw["prediction_policy"]
        reporting = raw["reporting"]
        if any(not isinstance(section, dict) for section in (locked, h, e, c, pred, reporting)):
            raise TypeError("all policy sections must be mappings")
        expected_locked = {
            "vertical_stress_profile": "outputs/07_density_overburden/vertical_stress_profile.csv",
            "shallow_column_scenarios": "outputs/07_density_overburden/shallow_column_scenarios.csv",
            "overburden_eligibility": "outputs/07_density_overburden/overburden_eligibility_summary.csv",
            "method_eligibility": "outputs/06_petrophysics_eligibility/method_eligibility_summary.csv",
            "thickness_sensitivity": "outputs/06_petrophysics_eligibility/thickness_sensitivity_summary.csv",
            "petrophysics_manifest": "outputs/06_petrophysics_eligibility/petrophysics_eligibility_manifest.json",
            "sonic_checkshot_drift": "outputs/04_checkshot_time_depth/sonic_checkshot_drift_summary.csv",
        }
        if locked != expected_locked:
            raise KeyError("locked input registry changed")
        if set(h) != {"gravity_m_s2", "fluid_density_low_kg_m3",
                      "fluid_density_base_kg_m3", "fluid_density_high_kg_m3",
                      "pressure_datum", "depth_coordinate"}:
            raise KeyError("hydrostatic policy fields changed")
        if (h["pressure_datum"] != "mean_sea_level_gauge_zero"
                or h["depth_coordinate"] != "tvdss_m_positive_downward"):
            raise ValueError("hydrostatic datum or coordinate changed")
        if set(e) != {"equation", "biot_alpha_scenarios", "profile_biot_alpha",
                      "profile_fluid_density_kg_m3", "negative_values_policy"}:
            raise KeyError("effective-stress policy fields changed")
        if (e["equation"] != "sigma_v_effective_pa = sigma_v_total_pa - biot_alpha * pore_pressure_pa"
                or e["negative_values_policy"] != "retain_and_flag_never_clip"):
            raise ValueError("effective-stress equation or negative-value policy changed")
        if set(c) != {"required_types", "available_file_count", "cross_well_transfer_allowed"}:
            raise KeyError("calibration policy fields changed")
        if set(pred) != {"nct_fit_enabled", "overpressure_transform_enabled",
                         "pressure_calibration_required",
                         "independent_normal_interval_evidence_required",
                         "sonic_checkshot_drift_correction_allowed", "extrapolation_allowed"}:
            raise KeyError("prediction policy fields changed")
        if set(reporting) != {"output_directory", "increment_9_started"}:
            raise KeyError("reporting policy fields changed")
        if reporting["output_directory"] != "outputs/08_pore_pressure_effective_stress":
            raise ValueError("output directory changed")
    except (OSError, TypeError, KeyError, ValueError, yaml.YAMLError) as exc:
        raise PorePressureInputError("Pressure policy is malformed.") from exc
    if not isinstance(raw, dict) or raw.get("increment") != 8:
        raise PorePressureInputError("Pressure policy must declare increment: 8.")
    return PressureConfig(
        schema_version=raw.get("schema_version"),
        assurance_tier=raw.get("assurance_tier"),
        gravity_m_s2=h.get("gravity_m_s2"),
        fluid_density_low_kg_m3=h.get("fluid_density_low_kg_m3"),
        fluid_density_base_kg_m3=h.get("fluid_density_base_kg_m3"),
        fluid_density_high_kg_m3=h.get("fluid_density_high_kg_m3"),
        biot_alpha_scenarios=tuple(e.get("biot_alpha_scenarios", ())),
        profile_biot_alpha=e.get("profile_biot_alpha"),
        profile_fluid_density_kg_m3=e.get("profile_fluid_density_kg_m3"),
        pressure_calibration_types=tuple(c.get("required_types", ())),
        pressure_calibration_file_count=c.get("available_file_count"),
        cross_well_transfer_allowed=c.get("cross_well_transfer_allowed"),
        nct_fit_enabled=pred.get("nct_fit_enabled"),
        overpressure_transform_enabled=pred.get("overpressure_transform_enabled"),
        pressure_calibration_required=pred.get("pressure_calibration_required"),
        independent_normal_interval_evidence_required=pred.get(
            "independent_normal_interval_evidence_required"),
        sonic_checkshot_drift_correction_allowed=pred.get(
            "sonic_checkshot_drift_correction_allowed"),
        extrapolation_allowed=pred.get("extrapolation_allowed"),
        increment_9_started=reporting.get("increment_9_started"),
    )


def hydrostatic_reference_pressure_pa(tvdss_m, fluid_density_kg_m3,
                                      gravity_m_s2) -> float:
    """Return gauge pressure from MSL: ``rho * g * TVDSS``.

    The domain is at or below mean sea level (TVDSS >= 0). Inputs are strict
    finite real scalars; booleans, strings, complex values, NaN and infinities
    are rejected with :class:`PorePressureInputError`.
    """
    from p2mem.pore_pressure_models import finite_real
    z = finite_real(tvdss_m, "tvdss_m", minimum=0.0)
    rho = finite_real(fluid_density_kg_m3, "fluid_density_kg_m3", minimum=0.0)
    g = finite_real(gravity_m_s2, "gravity_m_s2", minimum=0.0)
    if rho == 0.0 or g == 0.0:
        raise PorePressureInputError("fluid density and gravity must be > 0.")
    return rho * g * z


def effective_vertical_stress_pa(total_vertical_stress_pa,
                                 pore_pressure_pa, biot_alpha) -> float:
    """Return ``Sv - alpha*Pp`` without clipping negative scenario values."""
    from p2mem.pore_pressure_models import finite_real
    sv = finite_real(total_vertical_stress_pa, "total_vertical_stress_pa", minimum=0.0)
    pp = finite_real(pore_pressure_pa, "pore_pressure_pa", minimum=0.0)
    alpha = finite_real(biot_alpha, "biot_alpha", minimum=0.0, maximum=1.0)
    return sv - alpha * pp


def build_pressure_data_inventory(well_keys: Iterable[str]) -> Tuple[PressureDataAvailability, ...]:
    """Return the explicit zero-count *approved-input* inventory for each well."""
    return tuple(PressureDataAvailability(
        well_key=wk, rft_count=0, mdt_count=0, dst_count=0, fit_count=0,
        lot_count=0, xlot_count=0, dfit_count=0,
        availability_status="NOT_AVAILABLE",
    ) for wk in sorted(well_keys))


def derive_nct_readiness(well_key: str, thickness_rows: Sequence[Mapping[str, str]]) -> NctReadiness:
    """Derive whether a real NCT fit is defensible from locked sensitivity rows.

    Candidate configurations are data-eligibility cases, not independent
    measurements. Even when they exist, fitting is withheld because the
    project has neither pressure calibration nor an independently established
    normally compacted reference interval.
    """
    selected = [r for r in thickness_rows
                if r.get("well_key") == well_key
                and r.get("mask_name") == "eligible_sonic_nct_candidate"
                and r.get("contiguity_policy") == "strict_no_gap"]
    if not selected:
        return NctReadiness(
            well_key=well_key, status="NOT_ELIGIBLE_INPUT_QC_EXCLUSION",
            candidate_configurations=0,
            qualifying_thickness_min_tvd_m=None,
            qualifying_thickness_max_tvd_m=None,
            thickness_sensitivity_factor=None,
            pressure_calibration_available=False,
            independent_normal_interval_evidence_available=False,
            nct_fit_performed=False, overpressure_inferred=False,
        )
    expected_cases = {(scenario, threshold) for scenario in ("low", "base", "high")
                      for threshold in ("0.5", "0.6", "0.7")}
    cases = [(r.get("scenario_name"), r.get("proxy_threshold")) for r in selected]
    if len(cases) != len(set(cases)) or set(cases) != expected_cases:
        raise PorePressureInputError(
            "Locked strict-no-gap NCT sensitivity must contain exactly nine unique cases.")
    thickness = [
        _locked_float_token(
            r, "gross_qualifying_thickness_tvdss_m",
            "Locked thickness-sensitivity row")
        for r in selected
    ]
    if any(not math.isfinite(v) or v <= 0.0 for v in thickness):
        raise PorePressureInputError("Locked qualifying thickness must be finite and > 0.")
    lo, hi = min(thickness), max(thickness)
    return NctReadiness(
        well_key=well_key,
        status="WITHHELD_NO_PRESSURE_CALIBRATION_OR_INDEPENDENT_NORMAL_INTERVAL",
        candidate_configurations=len(selected),
        qualifying_thickness_min_tvd_m=lo,
        qualifying_thickness_max_tvd_m=hi,
        thickness_sensitivity_factor=hi / lo,
        pressure_calibration_available=False,
        independent_normal_interval_evidence_available=False,
        nct_fit_performed=False, overpressure_inferred=False,
    )


def build_hydrostatic_nodes(profile_rows: Sequence[Mapping[str, str]],
                            config: PressureConfig) -> Tuple[HydrostaticReferenceNode, ...]:
    """Build low/base/high hydrostatic references at locked profile nodes."""
    densities = (
        ("low", config.fluid_density_low_kg_m3),
        ("base", config.fluid_density_base_kg_m3),
        ("high", config.fluid_density_high_kg_m3),
    )
    out = []
    seen = set()
    last_by_well = {}
    for row in profile_rows:
        try:
            wk = row["well_key"]
        except (KeyError, TypeError) as exc:
            raise PorePressureInputError(
                "Locked vertical-stress profile row has no well key.") from exc
        if not isinstance(wk, str):
            raise PorePressureInputError(
                "Locked vertical-stress profile well key must be a string.")
        idx = _locked_int_token(
            row, "node_index", "Locked vertical-stress profile row")
        z = _locked_float_token(
            row, "tvdss_m", "Locked vertical-stress profile row")
        key = (wk, idx)
        if key in seen:
            raise PorePressureInputError("Locked profile contains a duplicate well/node index.")
        seen.add(key)
        previous = last_by_well.get(wk)
        if previous is None:
            if idx != 0:
                raise PorePressureInputError("Locked profile node index must begin at zero.")
        elif idx != previous[0] + 1 or z <= previous[1]:
            raise PorePressureInputError(
                "Locked profile indices must be contiguous and TVDSS strictly increasing.")
        last_by_well[wk] = (idx, z)
        for scenario, rho in densities:
            p = hydrostatic_reference_pressure_pa(z, rho, config.gravity_m_s2)
            out.append(HydrostaticReferenceNode(
                well_key=wk, node_index=idx, fluid_scenario=scenario,
                tvdss_m=z, fluid_density_kg_m3=rho,
                gravity_m_s2=config.gravity_m_s2, pressure_pa=p,
            ))
    return tuple(out)


def make_effective_scenario(*, well_key: str, sv_scenario: str,
                            fluid_scenario: str, biot_alpha: float,
                            evaluation_tvdss_m: float,
                            total_vertical_stress_pa: float,
                            fluid_density_kg_m3: float,
                            gravity_m_s2: float) -> EffectiveStressScenario:
    pp = hydrostatic_reference_pressure_pa(
        evaluation_tvdss_m, fluid_density_kg_m3, gravity_m_s2)
    eff = effective_vertical_stress_pa(total_vertical_stress_pa, pp, biot_alpha)
    return EffectiveStressScenario(
        well_key=well_key, sv_scenario=sv_scenario,
        fluid_scenario=fluid_scenario, biot_alpha=biot_alpha,
        evaluation_tvdss_m=evaluation_tvdss_m,
        total_vertical_stress_pa=total_vertical_stress_pa,
        pore_pressure_reference_pa=pp,
        effective_vertical_stress_pa=eff,
        effective_stress_nonnegative=eff >= 0.0,
    )


def scenario_initial_stress_pa(total_stress_pa, measured_only_stress_pa,
                               bridged_stress_pa) -> float:
    """Return the scenario stress above the first profile node.

    Increment 7's cumulative profile contains measured plus conditioned
    (bridged) density. Both components must therefore be removed from the
    terminal total before that cumulative profile is added back node by node.
    """
    from p2mem.pore_pressure_models import finite_real
    total = finite_real(total_stress_pa, "total_stress_pa", minimum=0.0)
    measured = finite_real(measured_only_stress_pa, "measured_only_stress_pa", minimum=0.0)
    bridged = finite_real(bridged_stress_pa, "bridged_stress_pa", minimum=0.0)
    initial = total - measured - bridged
    if initial < 0.0:
        raise PorePressureInputError("Scenario components exceed total stress.")
    return initial


In [ ]:
%%writefile p2mem/io/pore_pressure_inventory.py
"""Deterministic Increment 8 inventory builders and transactional export."""

from dataclasses import asdict
import csv
import hashlib
import json
import math
import os
from pathlib import Path
import shutil
import tempfile
import uuid
from typing import Dict, Iterable, Mapping, Sequence, Tuple

from p2mem.pore_pressure import (
    build_hydrostatic_nodes, build_pressure_data_inventory,
    derive_nct_readiness, hydrostatic_reference_pressure_pa,
    make_effective_scenario, scenario_initial_stress_pa,
)
from p2mem.pore_pressure_models import (
    APPROVED_BIOT_ALPHA_SCENARIOS, APPROVED_FLUID_DENSITIES_KG_M3,
    APPROVED_GRAVITY_M_S2, APPROVED_PROFILE_BIOT_ALPHA,
    APPROVED_PROFILE_FLUID_DENSITY_KG_M3, ASSURANCE_TIER, PACKAGE_VERSION,
    PRESSURE_DATA_TYPES, SV_SCENARIOS, WELL_KEYS,
    EffectiveStressScenario, HydrostaticReferenceNode, NctReadiness,
    PorePressureInputError, PressureConfig, PressureDataAvailability,
)


OUTPUT_NAMES = (
    "pressure_data_inventory.csv",
    "nct_readiness_summary.csv",
    "hydrostatic_reference_profile.csv",
    "effective_stress_scenarios.csv",
    "effective_stress_profile.csv",
    "pore_pressure_issues.csv",
    "pore_pressure_manifest.json",
)

LOCKED_SOURCE_NAMES = (
    "vertical_stress_profile.csv", "shallow_column_scenarios.csv",
    "overburden_eligibility_summary.csv", "method_eligibility_summary.csv",
    "thickness_sensitivity_summary.csv", "petrophysics_eligibility_manifest.json",
    "sonic_checkshot_drift_summary.csv",
)

# These hashes identify the exact approved Increment 6.1.7 / 7.0.4 artifacts.
# A syntactically valid but altered upstream file is not a locked source.
EXPECTED_LOCKED_SOURCE_SHA256 = {
    "vertical_stress_profile.csv":
        "bfe2bd37d31793285f06db34fda6e3e715dd8f085a6685620bca08167941ad52",
    "shallow_column_scenarios.csv":
        "356a5e83bca131ed1046c83d0c27d5624bafe8b2510cdff8a500871514aca7a8",
    "overburden_eligibility_summary.csv":
        "8bd03850d1ea8a58b620810f65af18c1d4115b3cd385c542949d7ba22e45cf12",
    "method_eligibility_summary.csv":
        "7fa715459de2eaa2007cdffe1af30a9c3eb424c78650f618ba34738d9a5c64b9",
    "thickness_sensitivity_summary.csv":
        "0c953df792a474960760e4bc491a05e2ffa448a8909589accadc14c78f2aec3e",
    "petrophysics_eligibility_manifest.json":
        "00e05aaa35454aef62bb7541adfbccca5a348ff8ccb2ecbc26b9a8093877a58e",
    "sonic_checkshot_drift_summary.csv":
        "17c84fcf9c9c320d9de665023d5d19ab6c5112bdf0d408378a289c9250886874",
}

FIGURE_NAMES = (
    "fig01_hydrostatic_reference_profile.png",
    "fig02_nct_readiness_and_sensitivity.png",
    "fig03_terminal_effective_stress_scenarios.png",
    "fig04_pressure_calibration_data_gap.png",
)

SHALLOW_SCENARIOS = (
    "low", "base", "high", "base_seawater_low", "base_seawater_high",
)

EFFECTIVE_STRESS_WELLS = ("Boreas_1", "Poseidon_2")
PROFILE_NODE_COUNTS = {
    "Boreas_1": 80,
    "Poseidon_2": 122,
    "Poseidon_North_1": 116,
    "Proteus_1ST2": 33,
}
EXPECTED_ROW_COUNTS = {
    "pressure_data_inventory.csv": 4,
    "nct_readiness_summary.csv": 4,
    "hydrostatic_reference_profile.csv": 1053,
    "effective_stress_scenarios.csv": 36,
    "effective_stress_profile.csv": 606,
    "pore_pressure_issues.csv": 11,
}
FLUID_DENSITY_BY_SCENARIO = dict(
    zip(("low", "base", "high"), APPROVED_FLUID_DENSITIES_KG_M3))

MANIFEST_TITLE = "Pore-Pressure Data-Gap, Hydrostatic Reference, and Effective-Stress Screening Framework"
MANIFEST_LIMITATIONS = (
    "Hydrostatic values use one configured constant-density gradient from mean sea level; they are references, not measured formation pressures or a separately modelled seawater-plus-formation-fluid column.",
    "Vertical-stress inputs are Increment 7 screening scenarios, not calibrated absolute stress measurements.",
    "Biot coefficients are configured sensitivity values; no laboratory measurement is available.",
    "Candidate sonic intervals do not establish normal compaction; no NCT was selected or fitted.",
    "No overpressure transform was run and no overpressure is inferred.",
    "No value is extrapolated beyond a locked source profile node.",
    "The zero-count pressure-data inventory describes approved packaged project inputs; it does not prove that no unprovided field data exist.",
)

CSV_COLUMNS = {
    "pressure_data_inventory.csv": (
        "well_key", "rft_count", "mdt_count", "dst_count", "fit_count",
        "lot_count", "xlot_count", "dfit_count", "availability_status",
        "evidence_class", "calibration_status", "assurance_tier",
    ),
    "nct_readiness_summary.csv": (
        "well_key", "status", "candidate_configurations",
        "qualifying_thickness_min_tvd_m", "qualifying_thickness_max_tvd_m",
        "thickness_sensitivity_factor", "pressure_calibration_available",
        "independent_normal_interval_evidence_available", "nct_fit_performed",
        "overpressure_inferred", "evidence_class", "calibration_status",
        "assurance_tier",
    ),
    "hydrostatic_reference_profile.csv": (
        "well_key", "node_index", "fluid_scenario", "tvdss_m",
        "fluid_density_kg_m3", "gravity_m_s2", "pressure_pa", "pressure_mpa",
        "pressure_basis", "evidence_class", "calibration_status", "assurance_tier",
    ),
    "effective_stress_scenarios.csv": (
        "well_key", "sv_scenario", "fluid_scenario", "biot_alpha",
        "evaluation_tvdss_m", "total_vertical_stress_pa",
        "total_vertical_stress_mpa", "pore_pressure_reference_pa",
        "pore_pressure_reference_mpa", "effective_vertical_stress_pa",
        "effective_vertical_stress_mpa", "effective_stress_nonnegative",
        "equation", "evidence_class", "calibration_status", "assurance_tier",
    ),
    "effective_stress_profile.csv": (
        "well_key", "node_index", "sv_scenario", "tvdss_m", "biot_alpha",
        "fluid_density_kg_m3", "total_vertical_stress_pa",
        "pore_pressure_reference_pa", "effective_vertical_stress_pa",
        "total_vertical_stress_mpa", "pore_pressure_reference_mpa",
        "effective_vertical_stress_mpa", "effective_stress_nonnegative",
        "evidence_class", "calibration_status", "assurance_tier",
    ),
    "pore_pressure_issues.csv": (
        "well_key", "severity", "issue_code", "measured_value", "unit",
        "disposition", "assurance_tier",
    ),
}

CSV_INTEGER_FIELDS = {
    "pressure_data_inventory.csv": {
        "rft_count", "mdt_count", "dst_count", "fit_count", "lot_count",
        "xlot_count", "dfit_count",
    },
    "nct_readiness_summary.csv": {"candidate_configurations"},
    "hydrostatic_reference_profile.csv": {"node_index"},
    "effective_stress_scenarios.csv": set(),
    "effective_stress_profile.csv": {"node_index"},
    "pore_pressure_issues.csv": set(),
}

CSV_NUMERIC_FIELDS = {
    "pressure_data_inventory.csv": set(),
    "nct_readiness_summary.csv": {
        "qualifying_thickness_min_tvd_m", "qualifying_thickness_max_tvd_m",
        "thickness_sensitivity_factor",
    },
    "hydrostatic_reference_profile.csv": {
        "tvdss_m", "fluid_density_kg_m3", "gravity_m_s2", "pressure_pa", "pressure_mpa",
    },
    "effective_stress_scenarios.csv": {
        "biot_alpha", "evaluation_tvdss_m", "total_vertical_stress_pa",
        "total_vertical_stress_mpa", "pore_pressure_reference_pa",
        "pore_pressure_reference_mpa", "effective_vertical_stress_pa",
        "effective_vertical_stress_mpa",
    },
    "effective_stress_profile.csv": {
        "tvdss_m", "biot_alpha", "fluid_density_kg_m3",
        "total_vertical_stress_pa", "pore_pressure_reference_pa",
        "effective_vertical_stress_pa", "total_vertical_stress_mpa",
        "pore_pressure_reference_mpa", "effective_vertical_stress_mpa",
    },
    "pore_pressure_issues.csv": {"measured_value"},
}

ALLOWED_STRING_VALUES = {
    ("pressure_data_inventory.csv", "well_key"): set(WELL_KEYS),
    ("pressure_data_inventory.csv", "availability_status"): {"NOT_AVAILABLE"},
    ("pressure_data_inventory.csv", "evidence_class"): {
        "factual_approved_input_absence_inventory"},
    ("pressure_data_inventory.csv", "calibration_status"): {
        "uncalibrated_no_approved_pressure_measurements"},
    ("nct_readiness_summary.csv", "well_key"): set(WELL_KEYS),
    ("nct_readiness_summary.csv", "status"): {
        "NOT_ELIGIBLE_INPUT_QC_EXCLUSION",
        "WITHHELD_NO_PRESSURE_CALIBRATION_OR_INDEPENDENT_NORMAL_INTERVAL",
    },
    ("nct_readiness_summary.csv", "pressure_calibration_available"): {"false"},
    ("nct_readiness_summary.csv", "independent_normal_interval_evidence_available"): {"false"},
    ("nct_readiness_summary.csv", "nct_fit_performed"): {"false"},
    ("nct_readiness_summary.csv", "overpressure_inferred"): {"false"},
    ("nct_readiness_summary.csv", "evidence_class"): {"eligibility_derived_not_prediction"},
    ("nct_readiness_summary.csv", "calibration_status"): {"uncalibrated_fit_withheld"},
    ("hydrostatic_reference_profile.csv", "well_key"): set(WELL_KEYS),
    ("hydrostatic_reference_profile.csv", "fluid_scenario"): {"low", "base", "high"},
    ("hydrostatic_reference_profile.csv", "pressure_basis"): {"configured_hydrostatic_reference_from_msl"},
    ("hydrostatic_reference_profile.csv", "evidence_class"): {"assumed_reference_not_measurement"},
    ("hydrostatic_reference_profile.csv", "calibration_status"): {"uncalibrated_reference_only"},
    ("effective_stress_scenarios.csv", "well_key"): {"Boreas_1", "Poseidon_2"},
    ("effective_stress_scenarios.csv", "sv_scenario"): {"low", "base", "high"},
    ("effective_stress_scenarios.csv", "fluid_scenario"): {"low", "base", "high"},
    ("effective_stress_scenarios.csv", "effective_stress_nonnegative"): {"true", "false"},
    ("effective_stress_scenarios.csv", "equation"): {
        "sigma_v_effective_equals_sigma_v_total_minus_alpha_times_pressure"},
    ("effective_stress_scenarios.csv", "evidence_class"): {"derived_screening_scenario"},
    ("effective_stress_scenarios.csv", "calibration_status"): {"uncalibrated_sensitivity_only"},
    ("effective_stress_profile.csv", "well_key"): {"Boreas_1", "Poseidon_2"},
    ("effective_stress_profile.csv", "sv_scenario"): {"low", "base", "high"},
    ("effective_stress_profile.csv", "effective_stress_nonnegative"): {"true", "false"},
    ("effective_stress_profile.csv", "evidence_class"): {"derived_screening_scenario"},
    ("effective_stress_profile.csv", "calibration_status"): {"uncalibrated_sensitivity_only"},
    ("pore_pressure_issues.csv", "well_key"): set(WELL_KEYS),
    ("pore_pressure_issues.csv", "severity"): {"WARNING"},
    ("pore_pressure_issues.csv", "issue_code"): {
        "PRESSURE_CALIBRATION_NOT_AVAILABLE", "NCT_AND_OVERPRESSURE_MODEL_NOT_RUN",
        "ABSOLUTE_VERTICAL_STRESS_SCENARIO_NOT_AVAILABLE",
        "SONIC_CHECKSHOT_DRIFT_DIAGNOSTIC_UNCORRECTED",
    },
    ("pore_pressure_issues.csv", "unit"): {"approved_files", "not_applicable", "percent"},
    ("pore_pressure_issues.csv", "disposition"): {
        "prediction_withheld", "effective_stress_not_computed",
        "diagnostic_only_no_curve_correction",
    },
}
for _artifact in CSV_COLUMNS:
    ALLOWED_STRING_VALUES[(_artifact, "assurance_tier")] = {ASSURANCE_TIER}


REQUIRED_LOCKED_COLUMNS = {
    "vertical_stress_profile.csv": (
        "well_key", "node_index", "tvdss_m", "cumulative_measured_increment_pa",
    ),
    "shallow_column_scenarios.csv": (
        "well_key", "scenario_name", "measured_formation_stress_pa",
        "bridged_gap_stress_pa", "water_column_stress_pa",
        "unresolved_shallow_stress_pa", "total_stress_pa",
    ),
    "overburden_eligibility_summary.csv": (
        "well_key", "overburden_status", "absolute_stress_supported",
    ),
    "method_eligibility_summary.csv": (
        "well_key", "mask_name", "use_status", "n_eligible",
    ),
    "thickness_sensitivity_summary.csv": (
        "well_key", "mask_name", "contiguity_policy", "scenario_name",
        "proxy_threshold", "gross_qualifying_thickness_tvdss_m",
    ),
    "sonic_checkshot_drift_summary.csv": (
        "well_key", "sonic_transit_time_s", "checkshot_owt_increment_s",
        "sonic_minus_checkshot_ms", "checkshot_minus_sonic_ms",
        "sonic_minus_checkshot_percent",
    ),
}


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as fh:
        for chunk in iter(lambda: fh.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def read_locked_csv(path) -> Tuple[dict, ...]:
    """Read a locked CSV fail-closed: exact header uniqueness and row width."""
    p = Path(path)
    if not p.is_file():
        raise PorePressureInputError(f"Locked input is missing: {p.name!r}.")
    with p.open(newline="", encoding="utf-8") as fh:
        reader = csv.DictReader(fh)
        fields = reader.fieldnames or []
        if not fields or len(fields) != len(set(fields)):
            raise PorePressureInputError(f"Locked input has invalid header: {p.name!r}.")
        required = REQUIRED_LOCKED_COLUMNS.get(p.name, ())
        missing = [name for name in required if name not in fields]
        if missing:
            raise PorePressureInputError(
                f"Locked input {p.name!r} is missing required columns: {missing!r}.")
        rows = []
        for number, row in enumerate(reader, start=2):
            if None in row or any(v is None for v in row.values()):
                raise PorePressureInputError(
                    f"Locked input {p.name!r} row {number} has wrong width.")
            rows.append(dict(row))
    if not rows:
        raise PorePressureInputError(f"Locked input is empty: {p.name!r}.")
    return tuple(rows)


def _bool_text(value: bool) -> str:
    return "true" if value else "false"


def _clean_record(record: Mapping[str, object]) -> dict:
    out = {}
    for key, value in record.items():
        if value is None:
            out[key] = ""
        elif type(value) is bool:
            out[key] = _bool_text(value)
        else:
            out[key] = value
    return out


def _csv_float(row: Mapping[str, str], field: str, context: str,
               *, minimum=None) -> float:
    """Parse one required CSV scalar without accepting blanks or non-finite values."""
    try:
        token = row[field]
        if not isinstance(token, str) or not token.strip():
            raise ValueError("required non-empty string token")
        value = float(token)
    except (KeyError, TypeError, ValueError, OverflowError) as exc:
        raise PorePressureInputError(
            f"{context}: {field!r} must be a finite numeric token.") from exc
    if not math.isfinite(value) or (minimum is not None and value < minimum):
        raise PorePressureInputError(
            f"{context}: {field!r} is outside its finite numeric domain.")
    return value


def _csv_int(row: Mapping[str, str], field: str, context: str,
             *, minimum=0) -> int:
    """Parse one required canonical non-negative integer CSV token."""
    try:
        token = row[field]
        if not isinstance(token, str) or not token.isdigit():
            raise ValueError("canonical integer token required")
        value = int(token)
    except (KeyError, TypeError, ValueError, OverflowError) as exc:
        raise PorePressureInputError(
            f"{context}: {field!r} must be a canonical integer token.") from exc
    if value < minimum:
        raise PorePressureInputError(
            f"{context}: {field!r} must be >= {minimum}.")
    return value


def _validate_source_hashes(source_hashes: Mapping[str, str]) -> None:
    if not isinstance(source_hashes, Mapping):
        raise PorePressureInputError("Source-artifact hash inventory is not a mapping.")
    if dict(source_hashes) != EXPECTED_LOCKED_SOURCE_SHA256:
        raise PorePressureInputError(
            "Source-artifact hashes do not match the exact approved locked inputs.")


def build_payloads(*, config: PressureConfig,
                   profile_rows: Sequence[Mapping[str, str]],
                   shallow_rows: Sequence[Mapping[str, str]],
                   overburden_rows: Sequence[Mapping[str, str]],
                   method_rows: Sequence[Mapping[str, str]],
                   thickness_rows: Sequence[Mapping[str, str]],
                   drift_rows: Sequence[Mapping[str, str]],
                   source_hashes: Mapping[str, str]) -> Dict[str, object]:
    """Build the complete deterministic Increment 8 payload set."""
    if (not isinstance(source_hashes, Mapping)
            or set(source_hashes) != set(LOCKED_SOURCE_NAMES) or any(
            not isinstance(value, str) or len(value) != 64
            or any(c not in "0123456789abcdef" for c in value)
             for value in source_hashes.values())):
        raise PorePressureInputError("Source-artifact hash inventory is not exact.")
    method_candidate_wells = set()
    for row in method_rows:
        if row.get("mask_name") != "eligible_sonic_nct_candidate":
            continue
        try:
            eligible = _csv_int(
                row, "n_eligible", "Locked method-eligibility row")
        except PorePressureInputError:
            raise
        if eligible > 0:
            method_candidate_wells.add(row.get("well_key"))
    thickness_candidate_wells = {
        row.get("well_key") for row in thickness_rows
        if row.get("mask_name") == "eligible_sonic_nct_candidate"
        and row.get("contiguity_policy") == "strict_no_gap"
    }
    if method_candidate_wells != thickness_candidate_wells:
        raise PorePressureInputError(
            "Locked method eligibility and thickness sensitivity disagree on candidate wells.")
    if not method_candidate_wells.issubset(WELL_KEYS):
        raise PorePressureInputError("Locked candidate data contain an unknown well key.")

    inventory = build_pressure_data_inventory(WELL_KEYS)
    readiness = tuple(derive_nct_readiness(wk, thickness_rows) for wk in WELL_KEYS)
    hydro = build_hydrostatic_nodes(profile_rows, config)

    availability_rows = []
    for rec in inventory:
        row = asdict(rec)
        row.update(evidence_class="factual_approved_input_absence_inventory",
                   calibration_status="uncalibrated_no_approved_pressure_measurements",
                   assurance_tier=ASSURANCE_TIER)
        availability_rows.append(_clean_record(row))

    readiness_rows = []
    for rec in readiness:
        row = asdict(rec)
        row.update(evidence_class="eligibility_derived_not_prediction",
                   calibration_status="uncalibrated_fit_withheld",
                   assurance_tier=ASSURANCE_TIER)
        readiness_rows.append(_clean_record(row))

    hydro_rows = []
    for rec in hydro:
        row = asdict(rec)
        row.update(
            pressure_mpa=rec.pressure_pa / 1e6,
            pressure_basis="configured_hydrostatic_reference_from_msl",
            evidence_class="assumed_reference_not_measurement",
            calibration_status="uncalibrated_reference_only",
            assurance_tier=ASSURANCE_TIER,
        )
        hydro_rows.append(_clean_record(row))

    by_profile: Dict[str, list] = {wk: [] for wk in WELL_KEYS}
    for row in profile_rows:
        if row.get("well_key") not in by_profile:
            raise PorePressureInputError("Locked vertical-stress profile contains an unknown well.")
        by_profile[row["well_key"]].append(row)
    for wk in by_profile:
        by_profile[wk].sort(
            key=lambda r: _csv_int(r, "node_index", "Locked vertical-stress profile"))
        cumulative_values = [
            _csv_float(r, "cumulative_measured_increment_pa",
                       "Locked vertical-stress profile", minimum=0.0)
            for r in by_profile[wk]
        ]
        if cumulative_values and not math.isclose(
                cumulative_values[0], 0.0, rel_tol=0.0, abs_tol=1e-12):
            raise PorePressureInputError(
                "Locked vertical-stress cumulative profile must begin at zero.")
        if any(b < a for a, b in zip(cumulative_values, cumulative_values[1:])):
            raise PorePressureInputError(
                "Locked vertical-stress cumulative profile must be non-decreasing.")
    if any(not rows for rows in by_profile.values()):
        raise PorePressureInputError("Locked vertical-stress profile is missing a well.")

    base_shallow = {}
    shallow_seen = set()
    for row in shallow_rows:
        wk, scenario = row.get("well_key"), row.get("scenario_name")
        if wk not in WELL_KEYS or scenario not in SHALLOW_SCENARIOS:
            raise PorePressureInputError(
                "Locked shallow-scenario inventory has an unknown well or scenario.")
        key = (wk, scenario)
        if key in shallow_seen:
            raise PorePressureInputError(f"Duplicate shallow scenario {key!r}.")
        shallow_seen.add(key)
        components = tuple(_csv_float(
            row, field, "Locked shallow-scenario row", minimum=0.0)
            for field in (
                "water_column_stress_pa", "unresolved_shallow_stress_pa",
                "measured_formation_stress_pa", "bridged_gap_stress_pa",
            ))
        total = _csv_float(
            row, "total_stress_pa", "Locked shallow-scenario row", minimum=0.0)
        if not math.isclose(total, sum(components), rel_tol=1e-12, abs_tol=1e-5):
            raise PorePressureInputError(
                "Locked shallow-scenario stress components do not sum to total stress.")
        if scenario in SV_SCENARIOS:
            base_shallow[key] = row

    terminal_records = []
    profile_out = []
    density_scenarios = (
        ("low", config.fluid_density_low_kg_m3),
        ("base", config.fluid_density_base_kg_m3),
        ("high", config.fluid_density_high_kg_m3),
    )
    for wk in WELL_KEYS:
        rows = by_profile[wk]
        has_all = all((wk, s) in base_shallow for s in SV_SCENARIOS)
        if not rows or not has_all:
            continue
        terminal = rows[-1]
        z_terminal = _csv_float(
            terminal, "tvdss_m", "Locked terminal profile", minimum=0.0)
        cumulative_terminal = _csv_float(
            terminal, "cumulative_measured_increment_pa",
            "Locked terminal profile", minimum=0.0)
        for sv_scenario in SV_SCENARIOS:
            source = base_shallow[(wk, sv_scenario)]
            measured = _csv_float(
                source, "measured_formation_stress_pa",
                "Locked shallow-scenario row", minimum=0.0)
            bridged = _csv_float(
                source, "bridged_gap_stress_pa",
                "Locked shallow-scenario row", minimum=0.0)
            total = _csv_float(
                source, "total_stress_pa", "Locked shallow-scenario row", minimum=0.0)
            if not math.isclose(cumulative_terminal, measured + bridged,
                                rel_tol=1e-10, abs_tol=1e-5):
                raise PorePressureInputError(
                    f"Locked terminal profile and scenario disagree for {wk!r}.")
            for fluid_scenario, rho in density_scenarios:
                for alpha in config.biot_alpha_scenarios:
                    rec = make_effective_scenario(
                        well_key=wk, sv_scenario=sv_scenario,
                        fluid_scenario=fluid_scenario, biot_alpha=alpha,
                        evaluation_tvdss_m=z_terminal,
                        total_vertical_stress_pa=total,
                        fluid_density_kg_m3=rho,
                        gravity_m_s2=config.gravity_m_s2,
                    )
                    row = asdict(rec)
                    row.update(
                        total_vertical_stress_mpa=rec.total_vertical_stress_pa / 1e6,
                        pore_pressure_reference_mpa=rec.pore_pressure_reference_pa / 1e6,
                        effective_vertical_stress_mpa=rec.effective_vertical_stress_pa / 1e6,
                        equation="sigma_v_effective_equals_sigma_v_total_minus_alpha_times_pressure",
                        evidence_class="derived_screening_scenario",
                        calibration_status="uncalibrated_sensitivity_only",
                        assurance_tier=ASSURANCE_TIER,
                    )
                    terminal_records.append(_clean_record(row))

            initial = scenario_initial_stress_pa(total, measured, bridged)
            for node in rows:
                idx = _csv_int(node, "node_index", "Locked vertical-stress profile")
                z = _csv_float(
                    node, "tvdss_m", "Locked vertical-stress profile", minimum=0.0)
                cumulative = _csv_float(
                    node, "cumulative_measured_increment_pa",
                    "Locked vertical-stress profile", minimum=0.0)
                sv = initial + cumulative
                pp = hydrostatic_reference_pressure_pa(
                    z, config.profile_fluid_density_kg_m3, config.gravity_m_s2)
                eff = sv - config.profile_biot_alpha * pp
                profile_out.append(_clean_record({
                    "well_key": wk, "node_index": idx, "sv_scenario": sv_scenario,
                    "tvdss_m": z, "biot_alpha": config.profile_biot_alpha,
                    "fluid_density_kg_m3": config.profile_fluid_density_kg_m3,
                    "total_vertical_stress_pa": sv,
                    "pore_pressure_reference_pa": pp,
                    "effective_vertical_stress_pa": eff,
                    "total_vertical_stress_mpa": sv / 1e6,
                    "pore_pressure_reference_mpa": pp / 1e6,
                    "effective_vertical_stress_mpa": eff / 1e6,
                    "effective_stress_nonnegative": eff >= 0.0,
                    "evidence_class": "derived_screening_scenario",
                    "calibration_status": "uncalibrated_sensitivity_only",
                    "assurance_tier": ASSURANCE_TIER,
                }))

    overburden_status = {}
    for row in overburden_rows:
        wk = row.get("well_key")
        if wk not in WELL_KEYS or wk in overburden_status:
            raise PorePressureInputError("Locked overburden inventory has an unknown or duplicate well.")
        if row.get("absolute_stress_supported") not in ("False", "false"):
            raise PorePressureInputError(
                "Increment 8 expects no well to support measured absolute vertical stress.")
        status = row.get("overburden_status")
        if status not in {"screening_sensitivity_only", "partial_measured_increment_only", "not_eligible"}:
            raise PorePressureInputError("Locked overburden status is outside the closed vocabulary.")
        overburden_status[wk] = status
    if set(overburden_status) != set(WELL_KEYS):
        raise PorePressureInputError("Locked overburden inventory is incomplete.")
    for wk, status in overburden_status.items():
        expected_scenarios = (set(SHALLOW_SCENARIOS)
                              if status == "screening_sensitivity_only" else set())
        observed_scenarios = {scenario for well, scenario in shallow_seen if well == wk}
        if observed_scenarios != expected_scenarios:
            raise PorePressureInputError(
                "Locked overburden status and exact shallow-scenario inventory disagree.")
        has_scenarios = all((wk, s) in base_shallow for s in SV_SCENARIOS)
        if has_scenarios != (status == "screening_sensitivity_only"):
            raise PorePressureInputError(
                "Locked overburden status and shallow-scenario availability disagree.")
    issues = []
    for wk in WELL_KEYS:
        issues.extend((
            {"well_key": wk, "severity": "WARNING",
             "issue_code": "PRESSURE_CALIBRATION_NOT_AVAILABLE",
             "measured_value": 0, "unit": "approved_files",
             "disposition": "prediction_withheld", "assurance_tier": ASSURANCE_TIER},
            {"well_key": wk, "severity": "WARNING",
             "issue_code": "NCT_AND_OVERPRESSURE_MODEL_NOT_RUN",
             "measured_value": "", "unit": "not_applicable",
             "disposition": "prediction_withheld", "assurance_tier": ASSURANCE_TIER},
        ))
        if overburden_status.get(wk) != "screening_sensitivity_only":
            issues.append({"well_key": wk, "severity": "WARNING",
                           "issue_code": "ABSOLUTE_VERTICAL_STRESS_SCENARIO_NOT_AVAILABLE",
                           "measured_value": "", "unit": "not_applicable",
                           "disposition": "effective_stress_not_computed",
                           "assurance_tier": ASSURANCE_TIER})
    poseidon_drift = [row for row in drift_rows if row.get("well_key") == "Poseidon_2"]
    if len(poseidon_drift) != 1 or len(drift_rows) != 1:
        raise PorePressureInputError("Locked sonic-checkshot drift inventory must contain one Poseidon 2 row.")
    for row in poseidon_drift:
        sonic_s = _csv_float(
            row, "sonic_transit_time_s", "Locked sonic-checkshot drift", minimum=0.0)
        checkshot_s = _csv_float(
            row, "checkshot_owt_increment_s", "Locked sonic-checkshot drift", minimum=0.0)
        drift_ms = _csv_float(
            row, "sonic_minus_checkshot_ms", "Locked sonic-checkshot drift")
        reverse_ms = _csv_float(
            row, "checkshot_minus_sonic_ms", "Locked sonic-checkshot drift")
        drift_percent = _csv_float(
            row, "sonic_minus_checkshot_percent", "Locked sonic-checkshot drift")
        if sonic_s <= 0.0 or checkshot_s <= 0.0:
            raise PorePressureInputError(
                "Locked sonic and checkshot transit times must be positive.")
        if (not math.isclose(drift_ms, (sonic_s - checkshot_s) * 1000.0,
                             rel_tol=1e-12, abs_tol=1e-12)
                or not math.isclose(reverse_ms, -drift_ms,
                                    rel_tol=1e-12, abs_tol=1e-12)
                or not math.isclose(
                    drift_percent, (sonic_s - checkshot_s) / checkshot_s * 100.0,
                    rel_tol=1e-12, abs_tol=1e-12)):
            raise PorePressureInputError(
                "Locked sonic-checkshot drift identities are inconsistent.")
        if row.get("well_key") == "Poseidon_2":
            issues.append({"well_key": "Poseidon_2", "severity": "WARNING",
                           "issue_code": "SONIC_CHECKSHOT_DRIFT_DIAGNOSTIC_UNCORRECTED",
                           "measured_value": drift_percent,
                           "unit": "percent",
                           "disposition": "diagnostic_only_no_curve_correction",
                           "assurance_tier": ASSURANCE_TIER})

    raw_csv = {
        "pressure_data_inventory.csv": availability_rows,
        "nct_readiness_summary.csv": readiness_rows,
        "hydrostatic_reference_profile.csv": hydro_rows,
        "effective_stress_scenarios.csv": terminal_records,
        "effective_stress_profile.csv": profile_out,
        "pore_pressure_issues.csv": issues,
    }
    ordered_csv = {}
    for name, rows in raw_csv.items():
        columns = CSV_COLUMNS[name]
        ordered = []
        for index, row in enumerate(rows):
            if set(row) != set(columns):
                raise PorePressureInputError(
                    f"{name} row {index} has unknown or missing fields.")
            ordered.append({column: row[column] for column in columns})
        ordered_csv[name] = ordered

    row_counts = {
        name: len(rows) for name, rows in ordered_csv.items()
    }
    _validate_source_hashes(source_hashes)
    manifest = {
        "package_version": PACKAGE_VERSION,
        "increment": 8,
        "increment_title": MANIFEST_TITLE,
        "assurance_tier": ASSURANCE_TIER,
        "source_artifact_sha256": dict(sorted(source_hashes.items())),
        "row_counts": row_counts,
        "pressure_calibration": {
            "approved_file_count": 0,
            "required_types": list(config.pressure_calibration_types),
            "cross_well_transfer_performed": False,
        },
        "prediction_gate": {
            "nct_fit_performed": False,
            "overpressure_transform_performed": False,
            "overpressure_inferred": False,
            "extrapolated_samples": 0,
        },
        "hydrostatic_reference": {
            "pressure_datum": "mean_sea_level_gauge_zero",
            "equation": "pressure_pa = fluid_density_kg_m3 * gravity_m_s2 * tvdss_m",
            "fluid_density_scenarios_kg_m3": [
                config.fluid_density_low_kg_m3,
                config.fluid_density_base_kg_m3,
                config.fluid_density_high_kg_m3,
            ],
            "status": "configured_reference_not_measured_formation_pressure",
        },
        "effective_stress": {
            "equation": "sigma_v_effective_pa = sigma_v_total_pa - biot_alpha * pore_pressure_pa",
            "biot_alpha_scenarios": list(config.biot_alpha_scenarios),
            "wells_with_scenarios": sorted({r["well_key"] for r in terminal_records}),
            "negative_values_clipped": 0,
        },
        "nct_readiness": [asdict(r) for r in readiness],
        "limitations": list(MANIFEST_LIMITATIONS),
        "increment_9_started": False,
    }
    return {
        **ordered_csv,
        "pore_pressure_manifest.json": manifest,
    }


def validate_payloads(payloads: Mapping[str, object]) -> None:
    if tuple(payloads) != OUTPUT_NAMES:
        raise PorePressureInputError("Increment 8 output inventory is not exact.")
    for name, columns in CSV_COLUMNS.items():
        rows = payloads[name]
        if not isinstance(rows, list):
            raise PorePressureInputError(f"{name} payload must be a list.")
        for index, row in enumerate(rows):
            if tuple(row) != columns:
                raise PorePressureInputError(
                    f"{name} row {index} has unknown, missing, or misordered fields.")
            for field, value in row.items():
                if field in CSV_INTEGER_FIELDS[name]:
                    if isinstance(value, bool) or not isinstance(value, int):
                        raise PorePressureInputError(f"{name}:{field} must be a strict integer.")
                    if value < 0:
                        raise PorePressureInputError(f"{name}:{field} must be non-negative.")
                elif field in CSV_NUMERIC_FIELDS[name]:
                    if value == "" and name in {"nct_readiness_summary.csv", "pore_pressure_issues.csv"}:
                        continue
                    if isinstance(value, bool) or not isinstance(value, (int, float)):
                        raise PorePressureInputError(f"{name}:{field} must be numeric.")
                    if not math.isfinite(float(value)):
                        raise PorePressureInputError(f"{name}:{field} must be finite.")
                else:
                    if not isinstance(value, str):
                        raise PorePressureInputError(f"{name}:{field} must be a string.")
                    allowed = ALLOWED_STRING_VALUES.get((name, field))
                    if allowed is None or value not in allowed:
                        raise PorePressureInputError(
                            f"{name}:{field} contains an unauthorized value.")
    manifest = payloads["pore_pressure_manifest.json"]
    expected_top = {
        "package_version", "increment", "increment_title", "assurance_tier",
        "source_artifact_sha256", "row_counts", "pressure_calibration",
        "prediction_gate", "hydrostatic_reference", "effective_stress",
        "nct_readiness", "limitations", "increment_9_started",
    }
    if not isinstance(manifest, dict) or set(manifest) != expected_top:
        raise PorePressureInputError("Increment 8 manifest payload is malformed.")
    if (manifest.get("package_version") != PACKAGE_VERSION or manifest.get("increment") != 8
            or manifest.get("increment_title") != MANIFEST_TITLE
            or manifest.get("assurance_tier") != ASSURANCE_TIER
            or manifest.get("limitations") != list(MANIFEST_LIMITATIONS)
            or manifest.get("increment_9_started") is not False):
        raise PorePressureInputError("Increment 8 manifest identity or limitation text changed.")
    hashes = manifest.get("source_artifact_sha256")
    _validate_source_hashes(hashes)
    if manifest.get("prediction_gate") != {
            "nct_fit_performed": False, "overpressure_transform_performed": False,
            "overpressure_inferred": False, "extrapolated_samples": 0}:
        raise PorePressureInputError("Manifest prediction gate is not closed.")
    if manifest.get("pressure_calibration") != {
            "approved_file_count": 0, "required_types": list(PRESSURE_DATA_TYPES),
            "cross_well_transfer_performed": False}:
        raise PorePressureInputError("Manifest calibration inventory is inconsistent.")
    if manifest.get("hydrostatic_reference") != {
            "pressure_datum": "mean_sea_level_gauge_zero",
            "equation": "pressure_pa = fluid_density_kg_m3 * gravity_m_s2 * tvdss_m",
            "fluid_density_scenarios_kg_m3": list(APPROVED_FLUID_DENSITIES_KG_M3),
            "status": "configured_reference_not_measured_formation_pressure"}:
        raise PorePressureInputError("Manifest hydrostatic-reference policy changed.")
    if manifest.get("effective_stress") != {
            "equation": "sigma_v_effective_pa = sigma_v_total_pa - biot_alpha * pore_pressure_pa",
            "biot_alpha_scenarios": list(APPROVED_BIOT_ALPHA_SCENARIOS),
            "wells_with_scenarios": ["Boreas_1", "Poseidon_2"],
            "negative_values_clipped": 0}:
        raise PorePressureInputError("Manifest effective-stress policy changed.")
    counts = manifest.get("row_counts")
    if (not isinstance(counts, dict) or set(counts) != set(CSV_COLUMNS)
            or any(isinstance(v, bool) or not isinstance(v, int) or v < 0
                   for v in counts.values())):
        raise PorePressureInputError("Manifest row-count registry is malformed.")
    if counts != EXPECTED_ROW_COUNTS:
        raise PorePressureInputError(
            "Manifest row counts do not match the locked Increment 8 inventory.")
    nct_manifest = manifest.get("nct_readiness")
    if not isinstance(nct_manifest, list) or len(nct_manifest) != 4:
        raise PorePressureInputError("Manifest NCT-readiness registry is malformed.")
    nct_keys = set(NctReadiness.__dataclass_fields__)
    if any(not isinstance(row, dict) or set(row) != nct_keys for row in nct_manifest):
        raise PorePressureInputError("Manifest NCT-readiness record shape changed.")
    if {row["well_key"] for row in nct_manifest} != set(WELL_KEYS):
        raise PorePressureInputError("Manifest NCT-readiness well inventory changed.")
    for row in nct_manifest:
        NctReadiness(**row)
    for name in CSV_COLUMNS:
        if manifest["row_counts"].get(name) != len(payloads[name]):
            raise PorePressureInputError(f"Manifest count disagrees for {name}.")

    primary_keys = {
        "pressure_data_inventory.csv": ("well_key",),
        "nct_readiness_summary.csv": ("well_key",),
        "hydrostatic_reference_profile.csv": ("well_key", "node_index", "fluid_scenario"),
        "effective_stress_scenarios.csv": ("well_key", "sv_scenario", "fluid_scenario", "biot_alpha"),
        "effective_stress_profile.csv": ("well_key", "node_index", "sv_scenario"),
        "pore_pressure_issues.csv": ("well_key", "issue_code"),
    }
    for name, keys in primary_keys.items():
        seen = set()
        for row in payloads[name]:
            key = tuple(row[k] for k in keys)
            if key in seen:
                raise PorePressureInputError(f"{name} contains a duplicate primary key.")
            seen.add(key)

    if {row["well_key"] for row in payloads["pressure_data_inventory.csv"]} != set(WELL_KEYS):
        raise PorePressureInputError("Pressure-data well inventory is incomplete.")
    if {row["well_key"] for row in payloads["nct_readiness_summary.csv"]} != set(WELL_KEYS):
        raise PorePressureInputError("NCT-readiness well inventory is incomplete.")

    nct_csv_as_records = []
    for row in payloads["nct_readiness_summary.csv"]:
        nct_csv_as_records.append({
            "well_key": row["well_key"],
            "status": row["status"],
            "candidate_configurations": row["candidate_configurations"],
            "qualifying_thickness_min_tvd_m": (
                None if row["qualifying_thickness_min_tvd_m"] == ""
                else row["qualifying_thickness_min_tvd_m"]),
            "qualifying_thickness_max_tvd_m": (
                None if row["qualifying_thickness_max_tvd_m"] == ""
                else row["qualifying_thickness_max_tvd_m"]),
            "thickness_sensitivity_factor": (
                None if row["thickness_sensitivity_factor"] == ""
                else row["thickness_sensitivity_factor"]),
            "pressure_calibration_available":
                row["pressure_calibration_available"] == "true",
            "independent_normal_interval_evidence_available":
                row["independent_normal_interval_evidence_available"] == "true",
            "nct_fit_performed": row["nct_fit_performed"] == "true",
            "overpressure_inferred": row["overpressure_inferred"] == "true",
        })
    if sorted(nct_csv_as_records, key=lambda r: r["well_key"]) != sorted(
            nct_manifest, key=lambda r: r["well_key"]):
        raise PorePressureInputError(
            "Manifest and CSV NCT-readiness records disagree.")

    hydro_by_key = {}
    for row in payloads["hydrostatic_reference_profile.csv"]:
        key = (row["well_key"], row["node_index"], row["fluid_scenario"])
        hydro_by_key[key] = row
        expected_density = FLUID_DENSITY_BY_SCENARIO[row["fluid_scenario"]]
        if (row["fluid_density_kg_m3"] != expected_density
                or row["gravity_m_s2"] != APPROVED_GRAVITY_M_S2):
            raise PorePressureInputError(
                "Hydrostatic profile does not use the approved density and gravity.")
    for well_key, count in PROFILE_NODE_COUNTS.items():
        reference_depths = None
        for scenario in ("low", "base", "high"):
            rows = sorted(
                (r for r in payloads["hydrostatic_reference_profile.csv"]
                 if r["well_key"] == well_key and r["fluid_scenario"] == scenario),
                key=lambda r: r["node_index"],
            )
            if [r["node_index"] for r in rows] != list(range(count)):
                raise PorePressureInputError(
                    "Hydrostatic profile node inventory is not contiguous and complete.")
            depths = [r["tvdss_m"] for r in rows]
            if any(b <= a for a, b in zip(depths, depths[1:])):
                raise PorePressureInputError(
                    "Hydrostatic profile TVDSS must be strictly increasing.")
            if reference_depths is None:
                reference_depths = depths
            elif depths != reference_depths:
                raise PorePressureInputError(
                    "Hydrostatic fluid scenarios do not share identical profile nodes.")

    expected_scenario_keys = {
        (well, sv, fluid, alpha)
        for well in EFFECTIVE_STRESS_WELLS
        for sv in SV_SCENARIOS
        for fluid in ("low", "base", "high")
        for alpha in APPROVED_BIOT_ALPHA_SCENARIOS
    }
    scenario_rows = payloads["effective_stress_scenarios.csv"]
    if {(r["well_key"], r["sv_scenario"], r["fluid_scenario"], r["biot_alpha"])
            for r in scenario_rows} != expected_scenario_keys:
        raise PorePressureInputError(
            "Effective-stress terminal scenario matrix is incomplete.")
    scenario_total_by_key = {}
    for row in scenario_rows:
        density = FLUID_DENSITY_BY_SCENARIO[row["fluid_scenario"]]
        expected_pressure = (
            density * APPROVED_GRAVITY_M_S2 * row["evaluation_tvdss_m"])
        if not math.isclose(
                row["pore_pressure_reference_pa"], expected_pressure,
                rel_tol=1e-12, abs_tol=1e-8):
            raise PorePressureInputError(
                "Effective-stress scenario hydrostatic identity failed.")
        key = (row["well_key"], row["sv_scenario"])
        identity = (row["evaluation_tvdss_m"], row["total_vertical_stress_pa"])
        if key in scenario_total_by_key and scenario_total_by_key[key] != identity:
            raise PorePressureInputError(
                "Effective-stress scenario total stress changes across pressure sensitivities.")
        scenario_total_by_key[key] = identity

    expected_profile_keys = {
        (well, node, sv)
        for well in EFFECTIVE_STRESS_WELLS
        for node in range(PROFILE_NODE_COUNTS[well])
        for sv in SV_SCENARIOS
    }
    profile_rows = payloads["effective_stress_profile.csv"]
    if {(r["well_key"], r["node_index"], r["sv_scenario"])
            for r in profile_rows} != expected_profile_keys:
        raise PorePressureInputError(
            "Effective-stress profile scenario/node inventory is incomplete.")
    for row in profile_rows:
        if (row["biot_alpha"] != APPROVED_PROFILE_BIOT_ALPHA
                or row["fluid_density_kg_m3"]
                != APPROVED_PROFILE_FLUID_DENSITY_KG_M3):
            raise PorePressureInputError(
                "Effective-stress profile does not use its approved reference scenario.")
        hydro_row = hydro_by_key[
            (row["well_key"], row["node_index"], "base")]
        if (row["tvdss_m"] != hydro_row["tvdss_m"]
                or not math.isclose(
                    row["pore_pressure_reference_pa"], hydro_row["pressure_pa"],
                    rel_tol=1e-12, abs_tol=1e-8)):
            raise PorePressureInputError(
                "Effective-stress and hydrostatic profiles disagree.")
    for well in EFFECTIVE_STRESS_WELLS:
        terminal_index = PROFILE_NODE_COUNTS[well] - 1
        for sv in SV_SCENARIOS:
            terminal_profile = next(
                r for r in profile_rows
                if r["well_key"] == well and r["node_index"] == terminal_index
                and r["sv_scenario"] == sv)
            terminal_scenario = scenario_total_by_key[(well, sv)]
            if (terminal_profile["tvdss_m"] != terminal_scenario[0]
                    or not math.isclose(
                        terminal_profile["total_vertical_stress_pa"],
                        terminal_scenario[1], rel_tol=1e-12, abs_tol=1e-8)):
                raise PorePressureInputError(
                    "Effective-stress profile and terminal scenario disagree.")

    expected_issue_keys = {
        *((well, "PRESSURE_CALIBRATION_NOT_AVAILABLE") for well in WELL_KEYS),
        *((well, "NCT_AND_OVERPRESSURE_MODEL_NOT_RUN") for well in WELL_KEYS),
        ("Poseidon_North_1", "ABSOLUTE_VERTICAL_STRESS_SCENARIO_NOT_AVAILABLE"),
        ("Proteus_1ST2", "ABSOLUTE_VERTICAL_STRESS_SCENARIO_NOT_AVAILABLE"),
        ("Poseidon_2", "SONIC_CHECKSHOT_DRIFT_DIAGNOSTIC_UNCORRECTED"),
    }
    if {(r["well_key"], r["issue_code"])
            for r in payloads["pore_pressure_issues.csv"]} != expected_issue_keys:
        raise PorePressureInputError("Pore-pressure issue inventory is incomplete.")

    for row in payloads["pressure_data_inventory.csv"]:
        PressureDataAvailability(
            row["well_key"], row["rft_count"], row["mdt_count"], row["dst_count"],
            row["fit_count"], row["lot_count"], row["xlot_count"], row["dfit_count"],
            row["availability_status"])
    for row in payloads["nct_readiness_summary.csv"]:
        NctReadiness(
            well_key=row["well_key"], status=row["status"],
            candidate_configurations=row["candidate_configurations"],
            qualifying_thickness_min_tvd_m=(None if row["qualifying_thickness_min_tvd_m"] == "" else row["qualifying_thickness_min_tvd_m"]),
            qualifying_thickness_max_tvd_m=(None if row["qualifying_thickness_max_tvd_m"] == "" else row["qualifying_thickness_max_tvd_m"]),
            thickness_sensitivity_factor=(None if row["thickness_sensitivity_factor"] == "" else row["thickness_sensitivity_factor"]),
            pressure_calibration_available=row["pressure_calibration_available"] == "true",
            independent_normal_interval_evidence_available=row["independent_normal_interval_evidence_available"] == "true",
            nct_fit_performed=row["nct_fit_performed"] == "true",
            overpressure_inferred=row["overpressure_inferred"] == "true")
    for row in payloads["hydrostatic_reference_profile.csv"]:
        HydrostaticReferenceNode(
            row["well_key"], row["node_index"], row["fluid_scenario"], row["tvdss_m"],
            row["fluid_density_kg_m3"], row["gravity_m_s2"], row["pressure_pa"])
        if not math.isclose(row["pressure_mpa"], row["pressure_pa"] / 1e6,
                            rel_tol=1e-12, abs_tol=1e-12):
            raise PorePressureInputError("Hydrostatic Pa/MPa identity failed.")
    for row in payloads["effective_stress_scenarios.csv"]:
        EffectiveStressScenario(
            row["well_key"], row["sv_scenario"], row["fluid_scenario"],
            row["biot_alpha"], row["evaluation_tvdss_m"],
            row["total_vertical_stress_pa"], row["pore_pressure_reference_pa"],
            row["effective_vertical_stress_pa"],
            row["effective_stress_nonnegative"] == "true")
        for pa_name, mpa_name in (
                ("total_vertical_stress_pa", "total_vertical_stress_mpa"),
                ("pore_pressure_reference_pa", "pore_pressure_reference_mpa"),
                ("effective_vertical_stress_pa", "effective_vertical_stress_mpa")):
            if not math.isclose(row[mpa_name], row[pa_name] / 1e6,
                                rel_tol=1e-12, abs_tol=1e-12):
                raise PorePressureInputError("Effective-scenario Pa/MPa identity failed.")
    for row in payloads["effective_stress_profile.csv"]:
        expected = row["total_vertical_stress_pa"] - row["biot_alpha"] * row["pore_pressure_reference_pa"]
        if (row["total_vertical_stress_pa"] < 0 or row["pore_pressure_reference_pa"] < 0
                or not 0 <= row["biot_alpha"] <= 1
                or not math.isclose(row["effective_vertical_stress_pa"], expected,
                                    rel_tol=1e-12, abs_tol=1e-8)
                or (row["effective_stress_nonnegative"] == "true") != (expected >= 0)):
            raise PorePressureInputError("Effective-profile identity failed.")
        for pa_name, mpa_name in (
                ("total_vertical_stress_pa", "total_vertical_stress_mpa"),
                ("pore_pressure_reference_pa", "pore_pressure_reference_mpa"),
                ("effective_vertical_stress_pa", "effective_vertical_stress_mpa")):
            if not math.isclose(row[mpa_name], row[pa_name] / 1e6,
                                rel_tol=1e-12, abs_tol=1e-12):
                raise PorePressureInputError("Effective-profile Pa/MPa identity failed.")


def _csv_bytes(columns: Sequence[str], rows: Sequence[Mapping[str, object]]) -> bytes:
    import io
    buf = io.StringIO(newline="")
    writer = csv.DictWriter(buf, fieldnames=columns, lineterminator="\n")
    writer.writeheader()
    writer.writerows(rows)
    return buf.getvalue().encode("utf-8")


def export_payloads(payloads: Mapping[str, object], output_dir) -> None:
    """Validate, serialize in isolation, then atomically publish the set."""
    validate_payloads(payloads)
    destination = Path(output_dir)
    destination.parent.mkdir(parents=True, exist_ok=True)
    staging = Path(tempfile.mkdtemp(prefix=f".{destination.name}.candidate-",
                                    dir=str(destination.parent)))
    backup = None
    try:
        for name in OUTPUT_NAMES:
            value = payloads[name]
            data = (_csv_bytes(CSV_COLUMNS[name], value) if name.endswith(".csv")
                    else (json.dumps(value, indent=2, sort_keys=True,
                                     ensure_ascii=True, allow_nan=False) + "\n").encode("utf-8"))
            (staging / name).write_bytes(data)
        # Re-read every staged artifact before it becomes official.
        for name in CSV_COLUMNS:
            with (staging / name).open(newline="", encoding="utf-8") as fh:
                reader = csv.DictReader(fh)
                if tuple(reader.fieldnames or ()) != CSV_COLUMNS[name]:
                    raise PorePressureInputError(f"Serialized {name} header changed.")
                observed = list(reader)
            expected = [{k: str(v) for k, v in row.items()} for row in payloads[name]]
            if observed != expected:
                raise PorePressureInputError(f"Serialized {name} records changed.")
        json.loads((staging / "pore_pressure_manifest.json").read_text(encoding="utf-8"))
        if destination.exists():
            backup = destination.parent / f".{destination.name}.backup-{uuid.uuid4().hex}"
            os.replace(destination, backup)
        try:
            os.replace(staging, destination)
        except Exception:
            if backup is not None and backup.exists() and not destination.exists():
                os.replace(backup, destination)
            raise
        if backup is not None and backup.exists():
            shutil.rmtree(backup)
    finally:
        if staging.exists():
            shutil.rmtree(staging)


def _render_figures_unpublished(
        payloads: Mapping[str, object], output_dir) -> Tuple[str, ...]:
    """Render and verify figures below an unpublished candidate directory."""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    validate_payloads(payloads)
    root = Path(output_dir) / "figures"
    root.mkdir(parents=True, exist_ok=True)
    metadata = {"Software": f"p2mem {PACKAGE_VERSION}"}

    hydro = [r for r in payloads["hydrostatic_reference_profile.csv"]
             if r["fluid_scenario"] == "base"]
    fig, ax = plt.subplots(figsize=(7, 6))
    for wk in WELL_KEYS:
        rows = [r for r in hydro if r["well_key"] == wk]
        ax.plot([r["pressure_mpa"] for r in rows], [r["tvdss_m"] for r in rows], label=wk)
    ax.invert_yaxis(); ax.grid(True, alpha=.25); ax.legend(fontsize=8)
    ax.set(xlabel="Hydrostatic reference pressure (MPa)", ylabel="TVDSS (m)",
           title="Configured hydrostatic reference — not measured pressure")
    fig.tight_layout(); fig.savefig(root / "fig01_hydrostatic_reference_profile.png", dpi=150, metadata=metadata); plt.close(fig)

    ready = payloads["nct_readiness_summary.csv"]
    fig, ax = plt.subplots(figsize=(8, 4))
    x = list(range(len(ready)))
    mins = [float(r["qualifying_thickness_min_tvd_m"] or 0.0) for r in ready]
    maxs = [float(r["qualifying_thickness_max_tvd_m"] or 0.0) for r in ready]
    ax.bar(x, maxs, color="#9ecae1", label="maximum")
    ax.bar(x, mins, color="#2171b5", label="minimum")
    ax.set_xticks(x, [r["well_key"] for r in ready], rotation=20, ha="right")
    ax.set(ylabel="Strict-no-gap qualifying thickness (m TVD)",
           title="NCT-candidate sensitivity; no NCT fitted")
    ax.legend(); ax.grid(True, axis="y", alpha=.25); fig.tight_layout()
    fig.savefig(root / "fig02_nct_readiness_and_sensitivity.png", dpi=150, metadata=metadata); plt.close(fig)

    term = [r for r in payloads["effective_stress_scenarios.csv"]
            if r["fluid_scenario"] == "base" and float(r["biot_alpha"]) == 1.0]
    fig, ax = plt.subplots(figsize=(8, 4))
    labels = [f"{r['well_key']}\n{r['sv_scenario']}" for r in term]
    vals = [r["effective_vertical_stress_mpa"] for r in term]
    ax.bar(range(len(vals)), vals, color="#31a354")
    ax.set_xticks(range(len(vals)), labels, rotation=25, ha="right")
    ax.set(ylabel="Effective vertical stress scenario (MPa)",
           title="Terminal sensitivity — hydrostatic reference, α=1.0")
    ax.grid(True, axis="y", alpha=.25); fig.tight_layout()
    fig.savefig(root / "fig03_terminal_effective_stress_scenarios.png", dpi=150, metadata=metadata); plt.close(fig)

    fig, ax = plt.subplots(figsize=(8, 3.5))
    rows = payloads["pressure_data_inventory.csv"]
    ax.imshow([[0] * 7 for _ in rows], cmap="Greys", vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(7), ["RFT", "MDT", "DST", "FIT", "LOT", "XLOT", "DFIT"])
    ax.set_yticks(range(4), [r["well_key"] for r in rows])
    for i in range(4):
        for j in range(7): ax.text(j, i, "0", ha="center", va="center")
    ax.set_title("Approved pressure-calibration measurements — factual inventory")
    fig.tight_layout(); fig.savefig(root / "fig04_pressure_calibration_data_gap.png", dpi=150, metadata=metadata); plt.close(fig)
    observed = tuple(p.name for p in sorted(root.glob("*.png")))
    if set(observed) != set(FIGURE_NAMES) or len(observed) != len(FIGURE_NAMES):
        raise PorePressureInputError("Rendered figure inventory is not exact.")
    return FIGURE_NAMES


def render_figures(payloads: Mapping[str, object], output_dir) -> Tuple[str, ...]:
    """Render four PNGs transactionally; preserve any prior complete set on failure."""
    output_root = Path(output_dir)
    output_root.mkdir(parents=True, exist_ok=True)
    destination = output_root / "figures"
    workspace = Path(tempfile.mkdtemp(
        prefix=".figures.candidate-", dir=str(output_root)))
    candidate_output = workspace / "output"
    backup = None
    try:
        names = _render_figures_unpublished(payloads, candidate_output)
        candidate = candidate_output / "figures"
        if destination.exists():
            if not destination.is_dir():
                raise PorePressureInputError(
                    "Figure destination exists but is not a directory.")
            backup = output_root / f".figures.backup-{uuid.uuid4().hex}"
            os.replace(destination, backup)
        try:
            os.replace(candidate, destination)
        except Exception:
            if backup is not None and backup.exists() and not destination.exists():
                os.replace(backup, destination)
            raise
        if backup is not None and backup.exists():
            shutil.rmtree(backup)
        return names
    finally:
        if workspace.exists():
            shutil.rmtree(workspace)


In [ ]:
%%writefile p2mem/io/pore_pressure_workflow.py
"""One deterministic, locked-output workflow for Increment 8."""

import json
import os
from pathlib import Path
import shutil
import tempfile
from typing import Dict
import uuid

from p2mem.pore_pressure import load_pressure_config
from p2mem.pore_pressure_models import PorePressureInputError
from p2mem.io.pore_pressure_inventory import (
    FIGURE_NAMES, OUTPUT_NAMES, build_payloads, export_payloads, read_locked_csv,
    render_figures, sha256_file,
)


LOCKED_PATHS = {
    "vertical_stress_profile.csv": "outputs/07_density_overburden/vertical_stress_profile.csv",
    "shallow_column_scenarios.csv": "outputs/07_density_overburden/shallow_column_scenarios.csv",
    "overburden_eligibility_summary.csv": "outputs/07_density_overburden/overburden_eligibility_summary.csv",
    "method_eligibility_summary.csv": "outputs/06_petrophysics_eligibility/method_eligibility_summary.csv",
    "thickness_sensitivity_summary.csv": "outputs/06_petrophysics_eligibility/thickness_sensitivity_summary.csv",
    "petrophysics_eligibility_manifest.json": "outputs/06_petrophysics_eligibility/petrophysics_eligibility_manifest.json",
    "sonic_checkshot_drift_summary.csv": "outputs/04_checkshot_time_depth/sonic_checkshot_drift_summary.csv",
}


def _publish_complete_bundle(payloads, destination: Path,
                             *, render_png: bool) -> tuple[str, ...]:
    """Stage data and figures together, then replace the official directory once.

    A serializer, plotting, or publication failure leaves the previous complete
    directory byte-for-byte intact and removes all temporary state.
    """
    destination.parent.mkdir(parents=True, exist_ok=True)
    workspace = Path(tempfile.mkdtemp(
        prefix=f".{destination.name}.bundle-", dir=str(destination.parent)))
    candidate = workspace / "candidate"
    backup = None
    try:
        export_payloads(payloads, candidate)
        figures = render_figures(payloads, candidate) if render_png else ()
        expected_files = set(OUTPUT_NAMES)
        if render_png:
            expected_files.update(f"figures/{name}" for name in FIGURE_NAMES)
        actual_files = {
            path.relative_to(candidate).as_posix()
            for path in candidate.rglob("*") if path.is_file()
        }
        if actual_files != expected_files:
            raise PorePressureInputError(
                "Complete Increment 8 output-bundle inventory is not exact.")
        if destination.exists():
            if not destination.is_dir():
                raise PorePressureInputError(
                    "Increment 8 output destination exists but is not a directory.")
            backup = destination.parent / (
                f".{destination.name}.backup-{uuid.uuid4().hex}")
            os.replace(destination, backup)
        try:
            os.replace(candidate, destination)
        except Exception:
            if backup is not None and backup.exists() and not destination.exists():
                os.replace(backup, destination)
            raise
        if backup is not None and backup.exists():
            shutil.rmtree(backup)
        return figures
    finally:
        if workspace.exists():
            shutil.rmtree(workspace)


def run_pore_pressure_workflow(project_root, *, render_png: bool = True) -> Dict[str, object]:
    """Build Increment 8 exclusively from packaged, locked upstream outputs."""
    root = Path(project_root)
    if not root.is_dir():
        raise PorePressureInputError("project_root must be an existing directory.")
    config_path = root / "config/pore_pressure.yml"
    config = load_pressure_config(config_path)
    paths = {name: root / rel for name, rel in LOCKED_PATHS.items()}
    for path in paths.values():
        if not path.is_file():
            raise PorePressureInputError(f"Locked source artifact is missing: {path.name!r}.")

    # Parse the upstream manifest too: a syntactically invalid locked record is
    # a hard failure even though no individual value is consumed from it.
    try:
        upstream_manifest = json.loads(
            paths["petrophysics_eligibility_manifest.json"].read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError) as exc:
        raise PorePressureInputError("Locked petrophysics manifest is malformed.") from exc
    calibration = upstream_manifest.get("calibration_data_available")
    required_flags = (
        "pressure_rft_mdt_dst", "stress_fit_lot_xlot_dfit",
        "independent_vp_vs_calibration",
    )
    if not isinstance(calibration, dict) or any(
            calibration.get(name) is not False for name in required_flags):
        raise PorePressureInputError(
            "Locked petrophysics manifest does not support the zero-calibration premise.")

    payloads = build_payloads(
        config=config,
        profile_rows=read_locked_csv(paths["vertical_stress_profile.csv"]),
        shallow_rows=read_locked_csv(paths["shallow_column_scenarios.csv"]),
        overburden_rows=read_locked_csv(paths["overburden_eligibility_summary.csv"]),
        method_rows=read_locked_csv(paths["method_eligibility_summary.csv"]),
        thickness_rows=read_locked_csv(paths["thickness_sensitivity_summary.csv"]),
        drift_rows=read_locked_csv(paths["sonic_checkshot_drift_summary.csv"]),
        source_hashes={name: sha256_file(path) for name, path in paths.items()},
    )
    output_dir = root / "outputs/08_pore_pressure_effective_stress"
    figures = _publish_complete_bundle(
        payloads, output_dir, render_png=render_png)
    return {"config": config, "payloads": payloads,
            "output_dir": output_dir, "figures": figures}


In [ ]:
%%writefile tests/synthetic_inc8.py
"""Small fictional Increment 8 fixtures; no private project data."""

from p2mem.pore_pressure_models import PressureConfig


def config() -> PressureConfig:
    return PressureConfig(
        schema_version="8.0.1",
        assurance_tier="Tier C - Screening-Level / Uncalibrated Educational",
        gravity_m_s2=9.80665,
        fluid_density_low_kg_m3=1020.0,
        fluid_density_base_kg_m3=1025.0,
        fluid_density_high_kg_m3=1030.0,
        biot_alpha_scenarios=(0.8, 1.0),
        profile_biot_alpha=1.0,
        profile_fluid_density_kg_m3=1025.0,
        pressure_calibration_types=("RFT", "MDT", "DST", "FIT", "LOT", "XLOT", "DFIT"),
        pressure_calibration_file_count=0,
        cross_well_transfer_allowed=False,
        nct_fit_enabled=False,
        overpressure_transform_enabled=False,
        pressure_calibration_required=True,
        independent_normal_interval_evidence_required=True,
        sonic_checkshot_drift_correction_allowed=False,
        extrapolation_allowed=False,
        increment_9_started=False,
    )


def thickness_rows(well="Poseidon_2"):
    rows = []
    for scenario, values in {
        "low": (100.0, 120.0, 150.0),
        "base": (80.0, 110.0, 140.0),
        "high": (50.0, 70.0, 90.0),
    }.items():
        for threshold, thickness in zip(("0.5", "0.6", "0.7"), values):
            rows.append({
                "well_key": well,
                "mask_name": "eligible_sonic_nct_candidate",
                "contiguity_policy": "strict_no_gap",
                "scenario_name": scenario,
                "proxy_threshold": threshold,
                "gross_qualifying_thickness_tvdss_m": str(thickness),
            })
    return rows


In [ ]:
%%writefile tests/test_pore_pressure.py
from dataclasses import replace
import math
from numbers import Real

import numpy as np
import pytest
import yaml

from p2mem.pore_pressure import (
    build_hydrostatic_nodes, build_pressure_data_inventory,
    derive_nct_readiness, effective_vertical_stress_pa,
    hydrostatic_reference_pressure_pa, load_pressure_config,
    make_effective_scenario, scenario_initial_stress_pa,
)
from p2mem.pore_pressure_models import (
    ASSURANCE_TIER, PRESSURE_DATA_TYPES, EffectiveStressScenario,
    HydrostaticReferenceNode, NctReadiness, PorePressureInputError,
    PressureDataAvailability,
)
from synthetic_inc8 import config, thickness_rows


INVALID_SCALARS = [True, False, "1", b"1", 1 + 0j, None, [1], np.array([1.0]),
                   float("nan"), float("inf"), float("-inf")]


@pytest.mark.parametrize("depth", [0.0, 1.0, 1000.0, np.float64(2000.0)])
@pytest.mark.parametrize("density", [1000.0, 1025.0, np.float64(1100.0)])
def test_hydrostatic_exact_identity(depth, density):
    got = hydrostatic_reference_pressure_pa(depth, density, 9.80665)
    assert got == pytest.approx(float(depth) * float(density) * 9.80665,
                                rel=1e-15, abs=1e-9)


@pytest.mark.parametrize("bad", INVALID_SCALARS)
def test_hydrostatic_rejects_invalid_depth(bad):
    with pytest.raises(PorePressureInputError):
        hydrostatic_reference_pressure_pa(bad, 1025.0, 9.80665)


@pytest.mark.parametrize("bad", INVALID_SCALARS)
def test_hydrostatic_rejects_invalid_density(bad):
    with pytest.raises(PorePressureInputError):
        hydrostatic_reference_pressure_pa(1000.0, bad, 9.80665)


@pytest.mark.parametrize("bad", INVALID_SCALARS)
def test_hydrostatic_rejects_invalid_gravity(bad):
    with pytest.raises(PorePressureInputError):
        hydrostatic_reference_pressure_pa(1000.0, 1025.0, bad)


@pytest.mark.parametrize("args", [(-1, 1025, 9.8), (1, -1, 9.8), (1, 1025, -1),
                                   (1, 0, 9.8), (1, 1025, 0)])
def test_hydrostatic_rejects_out_of_domain(args):
    with pytest.raises(PorePressureInputError):
        hydrostatic_reference_pressure_pa(*args)


@pytest.mark.parametrize("alpha,expected", [(0.0, 100.0), (0.5, 90.0), (1.0, 80.0)])
def test_effective_stress_equation(alpha, expected):
    assert effective_vertical_stress_pa(100.0, 20.0, alpha) == expected


def test_negative_effective_stress_is_retained_not_clipped():
    assert effective_vertical_stress_pa(10.0, 20.0, 1.0) == -10.0


@pytest.mark.parametrize("bad", INVALID_SCALARS)
@pytest.mark.parametrize("position", [0, 1, 2])
def test_effective_stress_rejects_invalid_scalar(bad, position):
    args = [100.0, 20.0, 1.0]
    args[position] = bad
    with pytest.raises(PorePressureInputError):
        effective_vertical_stress_pa(*args)


@pytest.mark.parametrize("args", [(-1, 0, 1), (1, -1, 1), (1, 0, -0.1), (1, 0, 1.1)])
def test_effective_stress_rejects_out_of_domain(args):
    with pytest.raises(PorePressureInputError):
        effective_vertical_stress_pa(*args)


def test_config_fixture_is_valid():
    c = config()
    assert c.schema_version == "8.0.1"
    assert c.pressure_calibration_types == PRESSURE_DATA_TYPES


@pytest.mark.parametrize("field,value", [
    ("schema_version", "8"), ("schema_version", "8.0.0"),
    ("assurance_tier", "calibrated"),
    ("gravity_m_s2", 0.0), ("gravity_m_s2", True),
    ("gravity_m_s2", 9.81),
    ("fluid_density_low_kg_m3", 1025.0),
    ("fluid_density_low_kg_m3", 1019.0),
    ("fluid_density_base_kg_m3", 1026.0),
    ("fluid_density_high_kg_m3", 1031.0),
    ("biot_alpha_scenarios", (0.8, 0.8)),
    ("biot_alpha_scenarios", (1.0, 0.8)),
    ("biot_alpha_scenarios", (1.1,)), ("profile_biot_alpha", -0.1),
    ("profile_biot_alpha", 0.8),
    ("profile_fluid_density_kg_m3", 1030.0),
    ("pressure_calibration_types", ("RFT",)),
    ("pressure_calibration_file_count", 1),
    ("pressure_calibration_file_count", 0.0),
    ("nct_fit_enabled", True), ("overpressure_transform_enabled", True),
    ("extrapolation_allowed", True), ("pressure_calibration_required", 1),
    ("pressure_calibration_required", False),
    ("independent_normal_interval_evidence_required", False),
    ("cross_well_transfer_allowed", True),
    ("sonic_checkshot_drift_correction_allowed", True),
    ("increment_9_started", True),
])
def test_config_invariants(field, value):
    with pytest.raises(PorePressureInputError):
        replace(config(), **{field: value})


def test_load_shipped_config(project_root):
    c = load_pressure_config(project_root / "config/pore_pressure.yml")
    assert c == config()


def test_load_config_missing(tmp_path):
    with pytest.raises(PorePressureInputError):
        load_pressure_config(tmp_path / "missing.yml")


def test_load_config_malformed(tmp_path):
    p = tmp_path / "x.yml"
    p.write_text("[not: valid", encoding="utf-8")
    with pytest.raises(PorePressureInputError):
        load_pressure_config(p)


def test_load_config_rejects_unknown_top_level_field(project_root, tmp_path):
    raw = yaml.safe_load((project_root / "config/pore_pressure.yml").read_text())
    raw["typo"] = 1
    p = tmp_path / "x.yml"; p.write_text(yaml.safe_dump(raw), encoding="utf-8")
    with pytest.raises(PorePressureInputError, match="malformed"):
        load_pressure_config(p)


def test_load_config_rejects_unknown_nested_field(project_root, tmp_path):
    raw = yaml.safe_load((project_root / "config/pore_pressure.yml").read_text())
    raw["prediction_policy"]["silent_bypass"] = True
    p = tmp_path / "x.yml"; p.write_text(yaml.safe_dump(raw), encoding="utf-8")
    with pytest.raises(PorePressureInputError, match="malformed"):
        load_pressure_config(p)


@pytest.mark.parametrize("needle", [
    'schema_version: "8.0.1"',
    "  gravity_m_s2: 9.80665",
])
def test_load_config_rejects_duplicate_key_at_any_depth(project_root, tmp_path, needle):
    text = (project_root / "config/pore_pressure.yml").read_text(encoding="utf-8")
    text = text.replace(needle, needle + "\n" + needle, 1)
    path = tmp_path / "duplicate.yml"
    path.write_text(text, encoding="utf-8")
    with pytest.raises(PorePressureInputError, match="malformed"):
        load_pressure_config(path)


def test_pressure_inventory_exact_four_wells():
    rows = build_pressure_data_inventory(("Poseidon_2", "Boreas_1"))
    assert [r.well_key for r in rows] == ["Boreas_1", "Poseidon_2"]
    assert all(r.availability_status == "NOT_AVAILABLE" for r in rows)


@pytest.mark.parametrize("field", ["rft_count", "mdt_count", "dst_count", "fit_count",
                                    "lot_count", "xlot_count", "dfit_count"])
@pytest.mark.parametrize("bad", [-1, 0.0, True, "0", float("nan")])
def test_pressure_availability_count_is_strict_integer(field, bad):
    good = PressureDataAvailability("Poseidon_2", 0, 0, 0, 0, 0, 0, 0, "NOT_AVAILABLE")
    with pytest.raises(PorePressureInputError):
        replace(good, **{field: bad})


def test_pressure_availability_cannot_claim_data():
    with pytest.raises(PorePressureInputError):
        PressureDataAvailability("Poseidon_2", 1, 0, 0, 0, 0, 0, 0, "NOT_AVAILABLE")


def test_nct_no_rows_means_not_eligible():
    r = derive_nct_readiness("Boreas_1", [])
    assert r.candidate_configurations == 0
    assert r.status == "NOT_ELIGIBLE_INPUT_QC_EXCLUSION"
    assert r.qualifying_thickness_min_tvd_m is None


def test_nct_candidate_sensitivity_is_measured():
    r = derive_nct_readiness("Poseidon_2", thickness_rows())
    assert r.candidate_configurations == 9
    assert r.qualifying_thickness_min_tvd_m == 50.0
    assert r.qualifying_thickness_max_tvd_m == 150.0
    assert r.thickness_sensitivity_factor == 3.0
    assert not r.nct_fit_performed and not r.overpressure_inferred


@pytest.mark.parametrize("value", ["", "nan", "inf", "0", "-1", True, 1.0])
def test_nct_rejects_invalid_qualifying_thickness(value):
    rows = thickness_rows()
    rows[0]["gross_qualifying_thickness_tvdss_m"] = value
    with pytest.raises(PorePressureInputError):
        derive_nct_readiness("Poseidon_2", rows)


def test_nct_rejects_duplicate_case():
    rows = thickness_rows(); rows[-1] = dict(rows[0])
    with pytest.raises(PorePressureInputError, match="exactly nine"):
        derive_nct_readiness("Poseidon_2", rows)


def test_nct_requires_complete_exact_strict_case_matrix():
    rows = thickness_rows()
    rows[0]["contiguity_policy"] = "configured_bridging"
    with pytest.raises(PorePressureInputError, match="exactly nine"):
        derive_nct_readiness("Poseidon_2", rows)


@pytest.mark.parametrize("field,value", [
    ("candidate_configurations", 0.0), ("candidate_configurations", True),
    ("pressure_calibration_available", 0), ("nct_fit_performed", True),
    ("overpressure_inferred", True), ("thickness_sensitivity_factor", 2.0),
])
def test_nct_record_invariants(field, value):
    good = derive_nct_readiness("Poseidon_2", thickness_rows())
    with pytest.raises(PorePressureInputError):
        replace(good, **{field: value})


def test_hydrostatic_node_constructor_checks_identity():
    with pytest.raises(PorePressureInputError):
        HydrostaticReferenceNode("Poseidon_2", 0, "base", 1000.0, 1025.0,
                                 9.80665, 1.0)


def test_build_hydrostatic_nodes_three_per_input_row():
    rows = [{"well_key": "Poseidon_2", "node_index": "0", "tvdss_m": "1000"}]
    out = build_hydrostatic_nodes(rows, config())
    assert len(out) == 3
    assert [r.fluid_scenario for r in out] == ["low", "base", "high"]


def test_hydrostatic_nodes_reject_duplicate_index():
    rows = [{"well_key": "Poseidon_2", "node_index": "0", "tvdss_m": "1000"}] * 2
    with pytest.raises(PorePressureInputError, match="duplicate"):
        build_hydrostatic_nodes(rows, config())


@pytest.mark.parametrize("field,value", [
    ("node_index", True), ("node_index", 0), ("node_index", 0.0),
    ("node_index", "0.0"), ("tvdss_m", True), ("tvdss_m", 1000.0),
])
def test_hydrostatic_nodes_require_canonical_csv_tokens(field, value):
    row = {"well_key": "Poseidon_2", "node_index": "0", "tvdss_m": "1000"}
    row[field] = value
    with pytest.raises(PorePressureInputError):
        build_hydrostatic_nodes([row], config())


@pytest.mark.parametrize("rows", [
    [{"well_key": "Poseidon_2", "node_index": "1", "tvdss_m": "1000"}],
    [{"well_key": "Poseidon_2", "node_index": "0", "tvdss_m": "1000"},
     {"well_key": "Poseidon_2", "node_index": "2", "tvdss_m": "1001"}],
    [{"well_key": "Poseidon_2", "node_index": "0", "tvdss_m": "1000"},
     {"well_key": "Poseidon_2", "node_index": "1", "tvdss_m": "999"}],
])
def test_hydrostatic_nodes_reject_noncontiguous_or_reversed(rows):
    with pytest.raises(PorePressureInputError):
        build_hydrostatic_nodes(rows, config())


def test_scenario_initial_stress_removes_measured_and_bridged_components():
    assert scenario_initial_stress_pa(100.0, 60.0, 10.0) == 30.0


def test_scenario_initial_stress_rejects_component_overrun():
    with pytest.raises(PorePressureInputError):
        scenario_initial_stress_pa(100.0, 95.0, 10.0)


def test_make_effective_scenario_sets_flag_and_identity():
    r = make_effective_scenario(
        well_key="Poseidon_2", sv_scenario="base", fluid_scenario="base",
        biot_alpha=1.0, evaluation_tvdss_m=1000.0,
        total_vertical_stress_pa=30e6, fluid_density_kg_m3=1025.0,
        gravity_m_s2=9.80665)
    assert r.effective_stress_nonnegative
    assert r.effective_vertical_stress_pa == 30e6 - 1025.0 * 9.80665 * 1000.0


def test_make_effective_scenario_retains_negative_and_flags():
    r = make_effective_scenario(
        well_key="Poseidon_2", sv_scenario="low", fluid_scenario="high",
        biot_alpha=1.0, evaluation_tvdss_m=1000.0,
        total_vertical_stress_pa=1.0, fluid_density_kg_m3=1030.0,
        gravity_m_s2=9.80665)
    assert r.effective_vertical_stress_pa < 0.0
    assert not r.effective_stress_nonnegative


@pytest.fixture
def project_root():
    from pathlib import Path
    return Path(__file__).resolve().parents[1]


In [ ]:
%%writefile tests/test_pore_pressure_inventory.py
import csv
import hashlib
import json
import math
import os
from pathlib import Path
import shutil

import pytest

from p2mem.io import pore_pressure_inventory as inv
from p2mem.io.pore_pressure_workflow import LOCKED_PATHS, run_pore_pressure_workflow
from p2mem.pore_pressure_models import PorePressureInputError, WELL_KEYS
from p2mem.wellframe_models import find_prohibited_lithology_terms


@pytest.fixture(scope="module")
def project_root():
    return Path(__file__).resolve().parents[1]


@pytest.fixture(scope="module")
def run(project_root):
    return run_pore_pressure_workflow(project_root, render_png=True)


def test_real_output_row_counts(run):
    p = run["payloads"]
    assert len(p["pressure_data_inventory.csv"]) == 4
    assert len(p["nct_readiness_summary.csv"]) == 4
    assert len(p["hydrostatic_reference_profile.csv"]) == 1053
    assert len(p["effective_stress_scenarios.csv"]) == 36
    assert len(p["effective_stress_profile.csv"]) == 606
    assert len(p["pore_pressure_issues.csv"]) == 11


def test_real_nct_readiness_values(run):
    rows = {r["well_key"]: r for r in run["payloads"]["nct_readiness_summary.csv"]}
    assert rows["Boreas_1"]["candidate_configurations"] == 0
    expected = {
        "Poseidon_2": (140.00288306447783, 1110.2406141181677, 7.930126793223611),
        "Poseidon_North_1": (341.53500002135024, 1198.155163071352, 3.508147519277533),
        "Proteus_1ST2": (350.1931358785905, 861.8933646094738, 2.461194341936748),
    }
    for wk, values in expected.items():
        row = rows[wk]
        assert row["candidate_configurations"] == 9
        assert row["qualifying_thickness_min_tvd_m"] == pytest.approx(values[0])
        assert row["qualifying_thickness_max_tvd_m"] == pytest.approx(values[1])
        assert row["thickness_sensitivity_factor"] == pytest.approx(values[2])
        assert row["nct_fit_performed"] == "false"
        assert row["overpressure_inferred"] == "false"


def test_terminal_effective_stress_reference_values(run):
    rows = run["payloads"]["effective_stress_scenarios.csv"]
    selected = {(r["well_key"], r["sv_scenario"]): r for r in rows
                if r["fluid_scenario"] == "base" and r["biot_alpha"] == 1.0}
    expected = {
        ("Boreas_1", "low"): 12.006014395007833,
        ("Boreas_1", "base"): 35.386034009776054,
        ("Boreas_1", "high"): 58.76605362454429,
        ("Poseidon_2", "low"): 18.463263937047348,
        ("Poseidon_2", "base"): 43.13105927689639,
        ("Poseidon_2", "high"): 67.79885461674543,
    }
    assert set(selected) == set(expected)
    for key, value in expected.items():
        assert selected[key]["effective_vertical_stress_mpa"] == pytest.approx(value)


def test_effective_stress_only_where_increment7_has_sv_scenarios(run):
    wells = {r["well_key"] for r in run["payloads"]["effective_stress_scenarios.csv"]}
    assert wells == {"Boreas_1", "Poseidon_2"}


def test_hydrostatic_reference_covers_all_locked_profile_nodes(run):
    rows = run["payloads"]["hydrostatic_reference_profile.csv"]
    assert {r["well_key"] for r in rows} == set(WELL_KEYS)
    for row in rows:
        assert row["pressure_pa"] == pytest.approx(
            row["fluid_density_kg_m3"] * row["gravity_m_s2"] * row["tvdss_m"])


def test_all_effective_records_satisfy_identity(run):
    for name in ("effective_stress_scenarios.csv", "effective_stress_profile.csv"):
        for row in run["payloads"][name]:
            assert row["effective_vertical_stress_pa"] == pytest.approx(
                row["total_vertical_stress_pa"]
                - row["biot_alpha"] * row["pore_pressure_reference_pa"])
            assert row["effective_stress_nonnegative"] == "true"


def test_manifest_prediction_gate_is_all_false(run):
    gate = run["payloads"]["pore_pressure_manifest.json"]["prediction_gate"]
    assert gate == {"nct_fit_performed": False,
                    "overpressure_transform_performed": False,
                    "overpressure_inferred": False,
                    "extrapolated_samples": 0}


def test_manifest_source_hashes_are_real(run, project_root):
    recorded = run["payloads"]["pore_pressure_manifest.json"]["source_artifact_sha256"]
    assert set(recorded) == set(LOCKED_PATHS)
    for name, rel in LOCKED_PATHS.items():
        assert recorded[name] == hashlib.sha256((project_root / rel).read_bytes()).hexdigest()
    assert recorded == inv.EXPECTED_LOCKED_SOURCE_SHA256


def test_output_strings_contain_no_prohibited_named_lithology(run):
    def strings(value):
        if isinstance(value, str): yield value
        elif isinstance(value, dict):
            for k, v in value.items(): yield from strings(k); yield from strings(v)
        elif isinstance(value, (list, tuple)):
            for v in value: yield from strings(v)
    violations = []
    for artifact, payload in run["payloads"].items():
        for text in strings(payload):
            terms = find_prohibited_lithology_terms(text)
            if terms: violations.append((artifact, text, terms))
    assert violations == []


def test_every_csv_has_exact_declared_column_order(run):
    for name, columns in inv.CSV_COLUMNS.items():
        for row in run["payloads"][name]:
            assert tuple(row) == columns


def test_export_is_byte_deterministic(run, tmp_path):
    a, b = tmp_path / "a", tmp_path / "b"
    inv.export_payloads(run["payloads"], a)
    inv.export_payloads(run["payloads"], b)
    for name in inv.OUTPUT_NAMES:
        assert (a / name).read_bytes() == (b / name).read_bytes()


def test_export_uses_lf_only(run, tmp_path):
    out = tmp_path / "out"
    inv.export_payloads(run["payloads"], out)
    for name in inv.OUTPUT_NAMES:
        data = (out / name).read_bytes()
        assert b"\r" not in data
        assert data.endswith(b"\n")


def test_export_rollback_is_byte_exact(run, tmp_path, monkeypatch):
    out = tmp_path / "official"
    out.mkdir()
    sentinel = out / "sentinel.bin"
    sentinel.write_bytes(b"LOCKED")
    real_replace = inv.os.replace
    calls = {"n": 0}
    def fail_second(src, dst):
        calls["n"] += 1
        if calls["n"] == 2:
            raise OSError("injected publication failure")
        return real_replace(src, dst)
    monkeypatch.setattr(inv.os, "replace", fail_second)
    with pytest.raises(OSError, match="injected"):
        inv.export_payloads(run["payloads"], out)
    assert sentinel.read_bytes() == b"LOCKED"
    assert list(out.iterdir()) == [sentinel]
    assert not list(tmp_path.glob(".official.*"))


def test_missing_field_fails_before_export(run, tmp_path):
    payloads = dict(run["payloads"])
    rows = [dict(r) for r in payloads["pressure_data_inventory.csv"]]
    rows[0].pop("rft_count")
    payloads["pressure_data_inventory.csv"] = rows
    with pytest.raises(PorePressureInputError):
        inv.export_payloads(payloads, tmp_path / "x")


def test_unknown_field_fails_before_export(run, tmp_path):
    payloads = dict(run["payloads"])
    rows = [dict(r) for r in payloads["pressure_data_inventory.csv"]]
    rows[0]["typo"] = "silent"
    payloads["pressure_data_inventory.csv"] = rows
    with pytest.raises(PorePressureInputError):
        inv.export_payloads(payloads, tmp_path / "x")


def test_unauthorized_string_fails_before_export(run, tmp_path):
    payloads = dict(run["payloads"])
    rows = [dict(r) for r in payloads["pressure_data_inventory.csv"]]
    rows[0]["calibration_status"] = "calibrated"
    payloads["pressure_data_inventory.csv"] = rows
    with pytest.raises(PorePressureInputError, match="unauthorized"):
        inv.export_payloads(payloads, tmp_path / "x")


def test_nonfinite_fails_before_export(run, tmp_path):
    payloads = dict(run["payloads"])
    rows = [dict(r) for r in payloads["hydrostatic_reference_profile.csv"]]
    rows[0]["pressure_pa"] = float("nan")
    payloads["hydrostatic_reference_profile.csv"] = rows
    with pytest.raises(PorePressureInputError):
        inv.export_payloads(payloads, tmp_path / "x")


def test_hydrostatic_identity_mutation_fails_before_export(run, tmp_path):
    payloads = dict(run["payloads"])
    rows = [dict(r) for r in payloads["hydrostatic_reference_profile.csv"]]
    rows[0]["pressure_pa"] += 1.0
    payloads["hydrostatic_reference_profile.csv"] = rows
    with pytest.raises(PorePressureInputError, match="identity"):
        inv.export_payloads(payloads, tmp_path / "x")


def test_effective_identity_mutation_fails_before_export(run, tmp_path):
    payloads = dict(run["payloads"])
    rows = [dict(r) for r in payloads["effective_stress_scenarios.csv"]]
    rows[0]["effective_vertical_stress_pa"] += 1.0
    payloads["effective_stress_scenarios.csv"] = rows
    with pytest.raises(PorePressureInputError, match="identity"):
        inv.export_payloads(payloads, tmp_path / "x")


def test_duplicate_primary_key_fails_before_export(run, tmp_path):
    payloads = dict(run["payloads"])
    rows = [dict(r) for r in payloads["pressure_data_inventory.csv"]]
    rows[1] = dict(rows[0])
    payloads["pressure_data_inventory.csv"] = rows
    with pytest.raises(PorePressureInputError, match="duplicate primary"):
        inv.export_payloads(payloads, tmp_path / "x")


def test_manifest_count_mismatch_fails(run, tmp_path):
    payloads = dict(run["payloads"])
    manifest = json.loads(json.dumps(payloads["pore_pressure_manifest.json"]))
    manifest["row_counts"]["pressure_data_inventory.csv"] = 999
    payloads["pore_pressure_manifest.json"] = manifest
    with pytest.raises(PorePressureInputError):
        inv.export_payloads(payloads, tmp_path / "x")


def test_manifest_prose_mutation_fails(run, tmp_path):
    payloads = dict(run["payloads"])
    manifest = json.loads(json.dumps(payloads["pore_pressure_manifest.json"]))
    manifest["limitations"][0] = "Everything is calibrated."
    payloads["pore_pressure_manifest.json"] = manifest
    with pytest.raises(PorePressureInputError, match="identity or limitation"):
        inv.export_payloads(payloads, tmp_path / "x")


def test_manifest_hash_type_fails(run, tmp_path):
    payloads = dict(run["payloads"])
    manifest = json.loads(json.dumps(payloads["pore_pressure_manifest.json"]))
    first = next(iter(manifest["source_artifact_sha256"]))
    manifest["source_artifact_sha256"][first] = None
    payloads["pore_pressure_manifest.json"] = manifest
    with pytest.raises(PorePressureInputError, match="hashes"):
        inv.export_payloads(payloads, tmp_path / "x")


def test_coordinated_row_removal_and_count_edit_still_fail_closed(run, tmp_path):
    payloads = dict(run["payloads"])
    payloads["pressure_data_inventory.csv"] = [
        dict(r) for r in payloads["pressure_data_inventory.csv"][:-1]
    ]
    manifest = json.loads(json.dumps(payloads["pore_pressure_manifest.json"]))
    manifest["row_counts"]["pressure_data_inventory.csv"] -= 1
    payloads["pore_pressure_manifest.json"] = manifest
    with pytest.raises(PorePressureInputError, match="locked Increment 8 inventory"):
        inv.export_payloads(payloads, tmp_path / "x")


def test_manifest_nct_record_must_equal_csv_record(run, tmp_path):
    payloads = dict(run["payloads"])
    manifest = json.loads(json.dumps(payloads["pore_pressure_manifest.json"]))
    manifest["nct_readiness"][1]["qualifying_thickness_min_tvd_m"] *= 2.0
    manifest["nct_readiness"][1]["qualifying_thickness_max_tvd_m"] *= 2.0
    payloads["pore_pressure_manifest.json"] = manifest
    with pytest.raises(PorePressureInputError, match="Manifest and CSV"):
        inv.export_payloads(payloads, tmp_path / "x")


def test_hydrostatic_density_must_match_named_scenario(run, tmp_path):
    payloads = dict(run["payloads"])
    rows = [dict(r) for r in payloads["hydrostatic_reference_profile.csv"]]
    rows[0]["fluid_density_kg_m3"] = 1019.0
    rows[0]["pressure_pa"] = (
        rows[0]["fluid_density_kg_m3"] * rows[0]["gravity_m_s2"] * rows[0]["tvdss_m"])
    rows[0]["pressure_mpa"] = rows[0]["pressure_pa"] / 1e6
    payloads["hydrostatic_reference_profile.csv"] = rows
    with pytest.raises(PorePressureInputError, match="approved density"):
        inv.export_payloads(payloads, tmp_path / "x")


def test_effective_profile_must_match_hydrostatic_profile(run, tmp_path):
    payloads = dict(run["payloads"])
    rows = [dict(r) for r in payloads["effective_stress_profile.csv"]]
    rows[0]["pore_pressure_reference_pa"] += 1.0
    rows[0]["pore_pressure_reference_mpa"] = rows[0]["pore_pressure_reference_pa"] / 1e6
    rows[0]["effective_vertical_stress_pa"] = (
        rows[0]["total_vertical_stress_pa"]
        - rows[0]["biot_alpha"] * rows[0]["pore_pressure_reference_pa"])
    rows[0]["effective_vertical_stress_mpa"] = rows[0]["effective_vertical_stress_pa"] / 1e6
    payloads["effective_stress_profile.csv"] = rows
    with pytest.raises(PorePressureInputError, match="hydrostatic profiles disagree"):
        inv.export_payloads(payloads, tmp_path / "x")


def test_direct_figure_render_failure_preserves_previous_figures(
        run, tmp_path, monkeypatch):
    out = tmp_path / "official"
    inv.export_payloads(run["payloads"], out)
    inv.render_figures(run["payloads"], out)
    before = {
        p.name: p.read_bytes() for p in (out / "figures").iterdir() if p.is_file()
    }

    def fail_candidate(payloads, candidate_output):
        root = Path(candidate_output) / "figures"
        root.mkdir(parents=True)
        (root / "partial.png").write_bytes(b"partial")
        raise RuntimeError("injected direct render failure")

    monkeypatch.setattr(inv, "_render_figures_unpublished", fail_candidate)
    with pytest.raises(RuntimeError, match="injected direct render failure"):
        inv.render_figures(run["payloads"], out)
    after = {
        p.name: p.read_bytes() for p in (out / "figures").iterdir() if p.is_file()
    }
    assert after == before
    assert not list(out.glob(".figures.*"))


@pytest.mark.parametrize("name", inv.OUTPUT_NAMES)
def test_exported_artifact_exists_and_nonempty(project_root, name):
    path = project_root / "outputs/08_pore_pressure_effective_stress" / name
    assert path.is_file() and path.stat().st_size > 0


@pytest.mark.parametrize("figure", inv.FIGURE_NAMES)
def test_figure_exists(project_root, figure):
    path = project_root / "outputs/08_pore_pressure_effective_stress/figures" / figure
    assert path.is_file() and path.stat().st_size > 1000


def test_read_locked_csv_missing(tmp_path):
    with pytest.raises(PorePressureInputError):
        inv.read_locked_csv(tmp_path / "missing.csv")


def test_read_locked_csv_duplicate_header(tmp_path):
    p = tmp_path / "vertical_stress_profile.csv"
    p.write_text("well_key,well_key,node_index,tvdss_m,cumulative_measured_increment_pa\n",
                 encoding="utf-8")
    with pytest.raises(PorePressureInputError):
        inv.read_locked_csv(p)


def test_read_locked_csv_missing_required_column(tmp_path):
    p = tmp_path / "vertical_stress_profile.csv"
    p.write_text("well_key,node_index,tvdss_m\n", encoding="utf-8")
    with pytest.raises(PorePressureInputError):
        inv.read_locked_csv(p)


def test_read_locked_csv_wrong_row_width(tmp_path):
    p = tmp_path / "vertical_stress_profile.csv"
    p.write_text("well_key,node_index,tvdss_m,cumulative_measured_increment_pa\nA,0,1\n",
                 encoding="utf-8")
    with pytest.raises(PorePressureInputError):
        inv.read_locked_csv(p)


def test_read_locked_csv_rejects_header_only_file(tmp_path):
    path = tmp_path / "vertical_stress_profile.csv"
    path.write_text(
        "well_key,node_index,tvdss_m,cumulative_measured_increment_pa\n",
        encoding="utf-8")
    with pytest.raises(PorePressureInputError, match="empty"):
        inv.read_locked_csv(path)


In [ ]:
%%writefile tests/test_pore_pressure_workflow.py
import csv
import hashlib
import json
from pathlib import Path
import shutil

import pytest

from p2mem.io import pore_pressure_workflow as workflow_module
from p2mem.io.pore_pressure_inventory import EXPECTED_LOCKED_SOURCE_SHA256
from p2mem.io.pore_pressure_workflow import LOCKED_PATHS, run_pore_pressure_workflow
from p2mem.pore_pressure_models import PorePressureInputError


@pytest.fixture(scope="module")
def project_root():
    return Path(__file__).resolve().parents[1]


def copy_minimal_project(source: Path, target: Path) -> Path:
    (target / "config").mkdir(parents=True)
    shutil.copy2(source / "config/pore_pressure.yml", target / "config/pore_pressure.yml")
    for rel in LOCKED_PATHS.values():
        dst = target / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source / rel, dst)
    return target


def test_workflow_runs_without_private_raw_inputs(project_root, tmp_path):
    root = copy_minimal_project(project_root, tmp_path / "project")
    result = run_pore_pressure_workflow(root, render_png=False)
    assert result["payloads"]["pore_pressure_manifest.json"]["increment"] == 8
    assert not (root / "private_inputs").exists()


def test_missing_locked_source_fails_closed(project_root, tmp_path):
    root = copy_minimal_project(project_root, tmp_path / "project")
    (root / LOCKED_PATHS["vertical_stress_profile.csv"]).unlink()
    with pytest.raises(PorePressureInputError, match="missing"):
        run_pore_pressure_workflow(root, render_png=False)


def test_malformed_upstream_manifest_fails_closed(project_root, tmp_path):
    root = copy_minimal_project(project_root, tmp_path / "project")
    (root / LOCKED_PATHS["petrophysics_eligibility_manifest.json"]).write_text("{", encoding="utf-8")
    with pytest.raises(PorePressureInputError, match="malformed"):
        run_pore_pressure_workflow(root, render_png=False)


def test_upstream_calibration_premise_must_remain_false(project_root, tmp_path):
    root = copy_minimal_project(project_root, tmp_path / "project")
    p = root / LOCKED_PATHS["petrophysics_eligibility_manifest.json"]
    data = json.loads(p.read_text())
    data["calibration_data_available"]["pressure_rft_mdt_dst"] = True
    p.write_text(json.dumps(data), encoding="utf-8")
    with pytest.raises(PorePressureInputError, match="zero-calibration"):
        run_pore_pressure_workflow(root, render_png=False)


def test_method_and_thickness_candidate_wells_must_agree(project_root, tmp_path):
    root = copy_minimal_project(project_root, tmp_path / "project")
    p = root / LOCKED_PATHS["method_eligibility_summary.csv"]
    rows = list(csv.DictReader(p.open(newline="", encoding="utf-8")))
    fields = list(rows[0])
    for row in rows:
        if row["well_key"] == "Poseidon_2" and row["mask_name"] == "eligible_sonic_nct_candidate":
            row["n_eligible"] = "0"
    with p.open("w", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=fields, lineterminator="\n")
        writer.writeheader(); writer.writerows(rows)
    with pytest.raises(PorePressureInputError, match="disagree"):
        run_pore_pressure_workflow(root, render_png=False)


def test_unknown_overburden_well_fails(project_root, tmp_path):
    root = copy_minimal_project(project_root, tmp_path / "project")
    p = root / LOCKED_PATHS["overburden_eligibility_summary.csv"]
    rows = list(csv.DictReader(p.open(newline="", encoding="utf-8")))
    fields = list(rows[0]); rows[0]["well_key"] = "Unknown"
    with p.open("w", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=fields, lineterminator="\n")
        writer.writeheader(); writer.writerows(rows)
    with pytest.raises(PorePressureInputError, match="unknown"):
        run_pore_pressure_workflow(root, render_png=False)


def test_absolute_stress_claim_change_fails(project_root, tmp_path):
    root = copy_minimal_project(project_root, tmp_path / "project")
    p = root / LOCKED_PATHS["overburden_eligibility_summary.csv"]
    rows = list(csv.DictReader(p.open(newline="", encoding="utf-8")))
    fields = list(rows[0]); rows[0]["absolute_stress_supported"] = "True"
    with p.open("w", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=fields, lineterminator="\n")
        writer.writeheader(); writer.writerows(rows)
    with pytest.raises(PorePressureInputError, match="expects no well"):
        run_pore_pressure_workflow(root, render_png=False)


def test_duplicate_drift_record_fails(project_root, tmp_path):
    root = copy_minimal_project(project_root, tmp_path / "project")
    p = root / LOCKED_PATHS["sonic_checkshot_drift_summary.csv"]
    text = p.read_text(encoding="utf-8")
    header, row = text.strip().splitlines()
    p.write_text(header + "\n" + row + "\n" + row + "\n", encoding="utf-8")
    with pytest.raises(PorePressureInputError, match="one Poseidon 2"):
        run_pore_pressure_workflow(root, render_png=False)


def test_workflow_is_repeatable_in_place(project_root, tmp_path):
    root = copy_minimal_project(project_root, tmp_path / "project")
    run_pore_pressure_workflow(root, render_png=False)
    first = {p.name: p.read_bytes() for p in (root / "outputs/08_pore_pressure_effective_stress").iterdir()}
    run_pore_pressure_workflow(root, render_png=False)
    second = {p.name: p.read_bytes() for p in (root / "outputs/08_pore_pressure_effective_stress").iterdir()}
    assert first == second


@pytest.mark.parametrize("source_name", tuple(LOCKED_PATHS))
def test_every_locked_source_is_bound_to_its_exact_approved_hash(
        project_root, tmp_path, source_name):
    root = copy_minimal_project(project_root, tmp_path / "project")
    path = root / LOCKED_PATHS[source_name]
    path.write_bytes(path.read_bytes() + b"\n")
    with pytest.raises(PorePressureInputError, match="exact approved locked inputs"):
        run_pore_pressure_workflow(root, render_png=False)
    assert not (root / "outputs/08_pore_pressure_effective_stress").exists()


def test_expected_hash_registry_matches_packaged_sources(project_root):
    observed = {
        name: hashlib.sha256((project_root / relative).read_bytes()).hexdigest()
        for name, relative in LOCKED_PATHS.items()
    }
    assert observed == EXPECTED_LOCKED_SOURCE_SHA256


def test_default_figure_runtime_dependency_is_declared(project_root):
    text = (project_root / "pyproject.toml").read_text(encoding="utf-8")
    assert '"matplotlib>=3.7"' in text


def test_drift_identity_mutation_is_rejected_before_publication(project_root, tmp_path):
    root = copy_minimal_project(project_root, tmp_path / "project")
    path = root / LOCKED_PATHS["sonic_checkshot_drift_summary.csv"]
    rows = list(csv.DictReader(path.open(newline="", encoding="utf-8")))
    fields = list(rows[0])
    rows[0]["sonic_minus_checkshot_ms"] = "16.0"
    with path.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=fields, lineterminator="\n")
        writer.writeheader(); writer.writerows(rows)
    with pytest.raises(PorePressureInputError, match="drift identities"):
        run_pore_pressure_workflow(root, render_png=False)
    assert not (root / "outputs/08_pore_pressure_effective_stress").exists()


def test_shallow_component_mutation_is_rejected_before_publication(project_root, tmp_path):
    root = copy_minimal_project(project_root, tmp_path / "project")
    path = root / LOCKED_PATHS["shallow_column_scenarios.csv"]
    rows = list(csv.DictReader(path.open(newline="", encoding="utf-8")))
    fields = list(rows[0])
    rows[0]["water_column_stress_pa"] = str(
        float(rows[0]["water_column_stress_pa"]) + 1.0)
    with path.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=fields, lineterminator="\n")
        writer.writeheader(); writer.writerows(rows)
    with pytest.raises(PorePressureInputError, match="components do not sum"):
        run_pore_pressure_workflow(root, render_png=False)


def test_profile_cumulative_decrease_is_rejected_before_publication(project_root, tmp_path):
    root = copy_minimal_project(project_root, tmp_path / "project")
    path = root / LOCKED_PATHS["vertical_stress_profile.csv"]
    rows = list(csv.DictReader(path.open(newline="", encoding="utf-8")))
    fields = list(rows[0])
    rows[1]["cumulative_measured_increment_pa"] = "-1"
    with path.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=fields, lineterminator="\n")
        writer.writeheader(); writer.writerows(rows)
    with pytest.raises(PorePressureInputError, match="finite numeric domain"):
        run_pore_pressure_workflow(root, render_png=False)


def _snapshot_files(root: Path):
    return {
        path.relative_to(root).as_posix(): path.read_bytes()
        for path in root.rglob("*") if path.is_file()
    }


def test_figure_failure_leaves_previous_complete_bundle_byte_identical(
        project_root, tmp_path, monkeypatch):
    root = copy_minimal_project(project_root, tmp_path / "project")
    result = run_pore_pressure_workflow(root, render_png=True)
    before = _snapshot_files(result["output_dir"])

    def fail_render(*_args, **_kwargs):
        raise RuntimeError("injected figure failure")

    monkeypatch.setattr(workflow_module, "render_figures", fail_render)
    with pytest.raises(RuntimeError, match="injected figure failure"):
        run_pore_pressure_workflow(root, render_png=True)
    assert _snapshot_files(result["output_dir"]) == before
    assert not list(result["output_dir"].parent.glob(
        ".08_pore_pressure_effective_stress.bundle-*"))


def test_bundle_publication_failure_rolls_back_without_residue(
        project_root, tmp_path, monkeypatch):
    root = copy_minimal_project(project_root, tmp_path / "project")
    result = run_pore_pressure_workflow(root, render_png=True)
    before = _snapshot_files(result["output_dir"])
    real_replace = workflow_module.os.replace

    def fail_candidate_publish(source, destination):
        if Path(source).name == "candidate" and Path(destination) == result["output_dir"]:
            raise OSError("injected bundle publication failure")
        return real_replace(source, destination)

    monkeypatch.setattr(workflow_module.os, "replace", fail_candidate_publish)
    with pytest.raises(OSError, match="injected bundle publication failure"):
        run_pore_pressure_workflow(root, render_png=True)
    assert _snapshot_files(result["output_dir"]) == before
    parent = result["output_dir"].parent
    assert not list(parent.glob(".08_pore_pressure_effective_stress.bundle-*"))
    assert not list(parent.glob(".08_pore_pressure_effective_stress.backup-*"))


### Step 4 — Install and run both the complete and locked test suites

In [ ]:
import subprocess

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"], check=True)
else:
    print("Local verification environment already supplies declared dependencies.")

full = subprocess.run([sys.executable, "-m", "pytest", "-q"], text=True,
                      capture_output=True, check=False)
print(full.stdout)
if full.stderr:
    print(full.stderr)
FULL_TEST_SUITE_PASSED = full.returncode == 0 and "1632 passed" in full.stdout
assert FULL_TEST_SUITE_PASSED

increment8_tests = [
    "tests/test_pore_pressure.py", "tests/test_pore_pressure_inventory.py",
    "tests/test_pore_pressure_workflow.py",
]
inc8 = subprocess.run([sys.executable, "-m", "pytest", "-q", *increment8_tests],
                      text=True, capture_output=True, check=False)
print(inc8.stdout)
INCREMENT8_TESTS_PASSED = inc8.returncode == 0 and "264 passed" in inc8.stdout
assert INCREMENT8_TESTS_PASSED

### Step 5 — Run the real locked-output integration

In [ ]:
from p2mem import __version__
from p2mem.io.pore_pressure_workflow import run_pore_pressure_workflow

assert __version__ == "0.8.1"
workflow = run_pore_pressure_workflow(project_root, render_png=True)
payloads = workflow["payloads"]
print("Version:", __version__)
print("Figures:", workflow["figures"])
print("Rows:", payloads["pore_pressure_manifest.json"]["row_counts"])

### Step 6 — Pressure/stress calibration data inventory

In [ ]:
for row in payloads["pressure_data_inventory.csv"]:
    print(row["well_key"], row["availability_status"],
          [row[k] for k in ("rft_count", "mdt_count", "dst_count", "fit_count",
                            "lot_count", "xlot_count", "dfit_count")])

### Step 7 — NCT readiness and sensitivity (no NCT fitted)

In [ ]:
for row in payloads["nct_readiness_summary.csv"]:
    print(row["well_key"], row["status"],
          "configurations=", row["candidate_configurations"],
          "min/max/factor=", row["qualifying_thickness_min_tvd_m"],
          row["qualifying_thickness_max_tvd_m"], row["thickness_sensitivity_factor"])

### Step 8 — Terminal effective-stress sensitivity

In [ ]:
terminal = [r for r in payloads["effective_stress_scenarios.csv"]
            if r["fluid_scenario"] == "base" and r["biot_alpha"] == 1.0]
for row in terminal:
    print(row["well_key"], row["sv_scenario"],
          f"Sv={row['total_vertical_stress_mpa']:.3f} MPa",
          f"Pp_ref={row['pore_pressure_reference_mpa']:.3f} MPa",
          f"Sv_eff={row['effective_vertical_stress_mpa']:.3f} MPa")

### Step 9 — Independent-root determinism check

In [ ]:
from p2mem.io.pore_pressure_workflow import LOCKED_PATHS

det_root = Path(tempfile.mkdtemp(prefix="p2mem_inc8_determinism_"))
(det_root / "config").mkdir(parents=True)
shutil.copy2(project_root / "config/pore_pressure.yml", det_root / "config/pore_pressure.yml")
for rel in LOCKED_PATHS.values():
    target = det_root / rel
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(project_root / rel, target)
run_pore_pressure_workflow(det_root, render_png=False)

deterministic_names = [name for name in payloads if name.endswith((".csv", ".json"))]
DETERMINISM_PASSED = all(
    (project_root / "outputs/08_pore_pressure_effective_stress" / name).read_bytes()
    == (det_root / "outputs/08_pore_pressure_effective_stress" / name).read_bytes()
    for name in deterministic_names
)
assert DETERMINISM_PASSED
shutil.rmtree(det_root)
print("Deterministic CSV/JSON artifacts:", len(deterministic_names))

### Step 10 — Completion gate

In [ ]:
import importlib.util

manifest = payloads["pore_pressure_manifest.json"]
checks = []
def check(label, condition):
    checks.append((label, bool(condition)))
    print(("[PASS] " if condition else "[FAIL] ") + label)

check("p2mem version is 0.8.1", __version__ == "0.8.1")
check("locked Increment 7 ledger has 235 entries", len(ledger_entries) == 235)
check("only three permitted baseline paths may differ", set(ledger_mismatches).issubset(permitted_preexisting_changes))
check("complete suite passed", FULL_TEST_SUITE_PASSED)
check("all 264 Increment 8 tests passed", INCREMENT8_TESTS_PASSED)
check("four pressure-data inventory rows", len(payloads["pressure_data_inventory.csv"]) == 4)
check("all pressure/stress calibration counts are zero", all(
    sum(int(r[k]) for k in ("rft_count", "mdt_count", "dst_count", "fit_count", "lot_count", "xlot_count", "dfit_count")) == 0
    for r in payloads["pressure_data_inventory.csv"]))
check("four NCT readiness rows", len(payloads["nct_readiness_summary.csv"]) == 4)
check("no NCT was fitted", all(r["nct_fit_performed"] == "false" for r in payloads["nct_readiness_summary.csv"]))
check("no overpressure was inferred", all(r["overpressure_inferred"] == "false" for r in payloads["nct_readiness_summary.csv"]))
check("hydrostatic references cover all four wells", {r["well_key"] for r in payloads["hydrostatic_reference_profile.csv"]} == {"Boreas_1", "Poseidon_2", "Poseidon_North_1", "Proteus_1ST2"})
check("hydrostatic identity holds at every node", all(abs(r["pressure_pa"] - r["fluid_density_kg_m3"]*r["gravity_m_s2"]*r["tvdss_m"]) <= 1e-7 for r in payloads["hydrostatic_reference_profile.csv"]))
check("effective stress is limited to two scenario-supported wells", {r["well_key"] for r in payloads["effective_stress_scenarios.csv"]} == {"Boreas_1", "Poseidon_2"})
check("effective-stress identity holds", all(abs(r["effective_vertical_stress_pa"] - (r["total_vertical_stress_pa"] - r["biot_alpha"]*r["pore_pressure_reference_pa"])) <= 1e-7 for r in payloads["effective_stress_scenarios.csv"]))
check("negative scenarios were never clipped", manifest["effective_stress"]["negative_values_clipped"] == 0)
check("zero extrapolated samples", manifest["prediction_gate"]["extrapolated_samples"] == 0)
check("no overpressure transform ran", manifest["prediction_gate"]["overpressure_transform_performed"] is False)
check("seven deterministic artifacts exist", all((workflow["output_dir"] / name).is_file() for name in deterministic_names) and len(deterministic_names) == 7)
check("four figures exist", len(workflow["figures"]) == 4 and all((workflow["output_dir"] / "figures" / name).is_file() for name in workflow["figures"]))
check("independent-root bytes match", DETERMINISM_PASSED)
from p2mem.io.pore_pressure_inventory import EXPECTED_LOCKED_SOURCE_SHA256, FIGURE_NAMES, OUTPUT_NAMES
check("all seven upstream artifacts match their exact approved hashes", all(
    hashlib.sha256((project_root / LOCKED_PATHS[name]).read_bytes()).hexdigest() == expected
    for name, expected in EXPECTED_LOCKED_SOURCE_SHA256.items()))
check("published bundle inventory is exact", {
    p.relative_to(workflow["output_dir"]).as_posix()
    for p in workflow["output_dir"].rglob("*") if p.is_file()
} == set(OUTPUT_NAMES) | {f"figures/{name}" for name in FIGURE_NAMES})
check("Increment 8 tests use local top-level fixtures, not an external tests package",
      "from tests." not in (project_root / "tests/test_pore_pressure.py").read_text(encoding="utf-8"))
future_modules = ["p2mem.elastic_properties", "p2mem.rock_strength", "p2mem.horizontal_stress", "p2mem.wellbore_stability"]
check("Increment 9 modules are absent", all(importlib.util.find_spec(name) is None for name in future_modules))
check("Increment 9 is not started", manifest["increment_9_started"] is False)

assert all(ok for _, ok in checks), [label for label, ok in checks if not ok]
print(f"\nINCREMENT 8 COMPLETION GATE: {sum(ok for _, ok in checks)}/{len(checks)} PASS")
print("Stopping here. Increment 9 has not been started.")

## Scientific reading rule

- **Measured facts:** locked upstream sample counts, depth nodes, scenario inputs, and the absence of calibration files from the approved packaged input set. This does not prove that unprovided field data do not exist.
- **Assumptions:** fluid-density bracket, standard gravity, Biot-alpha scenarios, and every Increment 7 shallow-column density scenario.
- **Derived scenarios:** hydrostatic reference pressure and effective vertical stress calculated from those assumptions.
- **Withheld:** NCT selection/fitting, overpressure transforms, formation-pressure prediction, operational mud-weight advice, and any calibration claim.

The hydrostatic reference uses one constant-density gradient from mean sea level; it is not a separately modelled seawater-plus-formation-fluid column. Run and read the full scenario spread. Do not treat a hydrostatic reference as measured formation pressure or any effective-stress scenario as calibrated truth.
